# Chạy thử UniHIR trên một vùng scan hỏng

Gắn Dataset **packages**, **models** và PDF, bật **4 × L4**, đặt **Internet Off**. Notebook lấy scan từ sách **4, trang PDF 26**, cắt vùng sát gáy có vết đen và chữ nhỏ mờ. Ô xem trước đánh dấu vùng cắt và hiển thị crop trước khi nạp model. Đổi `INPUT_FILE`, `PDF_PAGE` và `CROP_BOX` để thử vùng khác.

Lượt nhẹ giữ cả bước định vị và bước sinh ảnh phục hồi cuối, bỏ vòng phản tư bổ sung. Giới hạn cạnh sinh ảnh 384/512 px, số token định vị và số bước khử nhiễu. Checkpoint BAGEL vẫn chia qua ba L4, L4 còn lại chạy VAE. Có thể tăng `STEPS` rồi thử lại nếu lượt 10 bước cho ảnh thiếu nét.

Gắn bản **FlashAttention cache** có `flash_wheel.json`, nếu chưa nằm trong Dataset packages. Khi gắn cache riêng, điền `FLASH_CACHE_DIR` đến thư mục chứa manifest hoặc thư mục cha theo cấu trúc cache. Lượt mặc định dùng wheel khớp ABI đã qua CUDA probe. Notebook **0a** chuẩn bị wheel nếu cần.

Kết quả trong `/kaggle/working/unihir_smoke/`. So crop nguồn, bản nội suy bicubic và PNG sinh trực tiếp ở cùng kích thước. UniHIR có thể sinh chữ mới hoặc đổi nét còn đọc được. Cell cuối tải ảnh và chẩn đoán, kể cả log của lượt dừng giữa chừng.

In [ ]:
# Nếu chỉ gắn một Dataset khớp cho mỗi loại, để trống để tự tìm.
PACKAGES_DIR = ''
MODELS_DIR = ''
INPUT_FILE = ''  # Mặc định tìm 4.pdf. Có thể điền đường dẫn PDF hoặc PNG trong /kaggle/input.
PDF_PAGE = 26  # Số trang từ 1, chỉ dùng với PDF.
CROP_BOX = (246, 104, 358, 257)  # (trái, trên, phải, dưới) trên scan 600×532.

GPU_IDS = None  # Mặc định dùng đúng bốn GPU nhìn thấy.
STEPS = 10
REFLECTION_COUNT = 0  # Vẫn chạy định vị và sinh ảnh cuối.
MAX_TEXT_TOKENS = 192
DRAFT_EDGE = 384
FINAL_EDGE = 512
SEED = 42
MAX_GPU_MEMORY = '18GiB'

FLASH_CACHE_DIR = ''
ALLOW_FLASH_BUILD = False
VERIFY_WEIGHT_SHA256 = False
SETUP_TIMEOUT_MINUTES = 8  # Nếu chủ động bật build, phải tăng ngân sách setup.
LOAD_TIMEOUT_MINUTES = 18
INFERENCE_TIMEOUT_MINUTES = 20

In [ ]:
import hashlib
import importlib.util
import json
import os
import queue
import signal
import subprocess
import sys
import tempfile
import threading
import time
from pathlib import Path

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/unihir_smoke')
RUNTIME = Path('/tmp/vtm_image_runtime')
PLAN_PATH = Path('image_restoration/manifests/unihir_runtime.json')
ASSET_PATH = Path('image_restoration/unihir/assets.json')
REQUIRED_ASSETS = (
    'model.safetensors', 'ae.safetensors', 'llm_config.json', 'vit_config.json',
    'tokenizer_config.json', 'tokenizer.json', 'vocab.json', 'merges.txt',
    'font/KaiXinSongA.ttf', 'font/KaiXinSongB.ttf', 'font/TH-Tshyn-P0.ttf',
    'font/TH-Tshyn-P1.ttf', 'font/TH-Tshyn-P16.ttf', 'font/TH-Tshyn-P2.ttf',
)


def input_path(path):
    path = Path(path).resolve()
    if not path.is_relative_to(INPUT_ROOT.resolve()):
        raise ValueError(f'Chọn đường dẫn trong {INPUT_ROOT}: {path}')
    return path


def bundle_file(root, relative):
    root = root.resolve()
    path = (root / relative).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f'Đường dẫn ngoài bundle: {relative}')
    return path


def find_bundle(configured, marker, wrapper, setting):
    if configured:
        root = input_path(configured)
        candidates = [root / marker, root / wrapper / marker]
    else:
        candidates = [p for p in INPUT_ROOT.rglob(marker.name)
                      if p.parts[-len(marker.parts):] == marker.parts]
    matches = sorted({p for p in candidates if p.is_file()})
    if len(matches) != 1:
        raise ValueError(f'Gắn Dataset và điền {setting}; tìm thấy {len(matches)}: {matches}')
    return input_path(matches[0]).parents[len(marker.parts) - 1]


def find_input(configured):
    if configured:
        path = input_path(configured)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(p for p in INPUT_ROOT.rglob('4.pdf') if p.is_file())
    if len(matches) != 1:
        raise ValueError(f'Điền INPUT_FILE; tìm thấy {len(matches)} bản 4.pdf: {matches}')
    return input_path(matches[0])


def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding='utf8'))


def prep_worker_environment():
    env = dict(os.environ)
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE='1', PYTHONUTF8='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DIFFUSERS_OFFLINE='1', HF_DATASETS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1', PIP_NO_INDEX='1', PIP_DISABLE_PIP_VERSION_CHECK='1', TOKENIZERS_PARALLELISM='false', KERAS_BACKEND='torch')
    return env


def log_message(log, line):
    print(line, end='', flush=True)
    log.write(line)
    log.flush()


def process_activity(pid):
    proc = Path('/proc') / str(pid)
    try:
        fields = (proc / 'stat').read_text().rsplit(')', 1)[-1].split()
        cpu = (int(fields[11]) + int(fields[12])) / os.sysconf('SC_CLK_TCK')
        rss = int(fields[21]) * os.sysconf('SC_PAGE_SIZE') / 1024 ** 3
        return f'PID {pid}, state={fields[0]}, CPU tích lũy {cpu:.1f}s, RAM {rss:.2f} GiB'
    except (OSError, ValueError, IndexError):
        return f'PID {pid}'


def run_logged(command, env, log, *, stage, timeout_minutes=None, ready_prefix=None, inference_minutes=None):
    lines = queue.Queue()
    start = time.monotonic()
    deadline = start + timeout_minutes * 60 if timeout_minutes is not None else None
    heartbeat = start + 30
    process = subprocess.Popen(list(map(str, command)), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf8',
                               errors='replace', bufsize=1, start_new_session=True)

    def read_output():
        try:
            with process.stdout as stdout:
                for line in stdout:
                    lines.put(line)
        finally:
            lines.put(None)

    threading.Thread(target=read_output, daemon=True).start()
    output_done = False
    try:
        while not output_done or process.poll() is None:
            now = time.monotonic()
            try:
                line = lines.get(timeout=1)
            except queue.Empty:
                if now >= heartbeat:
                    log_message(log, f'{stage}: {now - start:.0f}s, {process_activity(process.pid)}.\n')
                    heartbeat = now + 30
            else:
                if line is None:
                    output_done = True
                else:
                    log_message(log, line)
                    heartbeat = now + 30
                    if ready_prefix is not None and line.startswith(ready_prefix):
                        start = time.monotonic()
                        timeout_minutes = inference_minutes
                        deadline = start + inference_minutes * 60
                        stage = 'Inference'
            if deadline is not None and now >= deadline and process.poll() is None:
                message = f'{stage}: quá {timeout_minutes:g} phút. {process_activity(process.pid)}.'
                log_message(log, message + '\n')
                raise TimeoutError(message)
        if process.returncode:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                pass
            except ProcessLookupError:
                pass
            finally:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass


import math

PDF_IMAGE_VERSION = "pdfium-visible-v1"


def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t-b)/w, (r-l)/h, 0, l, b]
    if rotation == 180:
        return [-(r-l)/w, 0, 0, (t-b)/h, r, b]
    if rotation == 270:
        return [0, -(t-b)/w, -(r-l)/h, 0, r, t]
    return [(r-l)/w, 0, 0, -(t-b)/h, l, t]


def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w*scale), math.ceil(h*scale)]
    if min(size) < 1 or size[0]*size[1] > max_pixels:
        raise ValueError(f"Page render {size} exceeds max_source_pixels={max_pixels}; "
                         "review acquisition settings (source is never silently reduced)")
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert("RGB")
    finally:
        bitmap.close()


def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100_000_000, strategy="auto"):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops

    if strategy not in {"auto", "render"} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError("Invalid PDF acquisition options")
    width, height = page.get_size()
    bbox, rotation = list(page.get_bbox()), page.get_rotation()
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = [], 0.0
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x,y) for x,y in ((0,0),(1,0),(1,1),(0,1))]
        bounds = [min(c[0] for c in corners),min(c[1] for c in corners),
                  max(c[0] for c in corners),max(c[1] for c in corners)]
        full = (
            max(0, min(bbox[2], bounds[2])-max(bbox[0], bounds[0])) *
            max(0, min(bbox[3], bounds[3])-max(bbox[1], bounds[1])) >=
            .8*(bbox[2]-bbox[0])*(bbox[3]-bbox[1]))
        native = max(pixels[0]/axes[0], pixels[1]/axes[1]) if min(axes) > 0 else 0
        # Include partial-page/multiple scans as well: a DPI chosen from only a
        # full-page image could silently reduce detail in another visible image.
        density = max(density, native)
        images.append({"native_size": list(pixels), "bounds_pdf": bounds,
                       "matrix_pdf": [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f],
                       "level": obj.level, "full_page_scan": full,
                       "pixels_per_72_canvas_units": 72*native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append("not_one_top_level_image; preserve_composition")
    if rotation:
        reasons.append("page_rotation")
    if annotations != 0:
        reasons.append("page_annotations_or_unknown")
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if (obj.level or abs(m.b) > 1e-6 or abs(m.c) > 1e-6 or m.a <= 0 or m.d <= 0
                or any(abs(a-v) > .001 for a, v in zip(obj.get_bounds(), bbox))
                or any(abs(a-v) > .001 for a, v in zip(
                    [m.e, m.f, m.a, m.d], [l, b, r-l, t-b]))):
            reasons.append("placement_crop_rotation_or_reflection")
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        # PDFium returns -1 for the non-null empty clip handle on ordinary
        # images. An explicit path is a reason to render; final pixel equality
        # remains mandatory when no clip is reported.
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append("image_clip_path")

    image, method, validation = None, "pdf_render", None
    if strategy == "auto" and not reasons:
        nw, nh = obj.get_px_size()
        if nw*nh > max_source_pixels:
            raise ValueError("Native image exceeds max_source_pixels")
        # Small fractional PDF dimensions can otherwise add a raster row.
        scale = math.nextafter(min(nw/width, nh/height), 0)
        if abs(nw/width-nh/height) > 1e-4*max(nw/width, nh/height):
            reasons.append("anisotropic_native_geometry")
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert("RGB")
            finally:
                # Conversion above owns its data; no PIL view survives closing.
                if hasattr(bitmap, "warn_on_close"):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {"method": "full_pixel_equality_to_visible_render", "equal": equal,
                          "render_size": list(visible.size), "scale": scale}
            if equal:
                image, method = candidate, "native_validated"
            else:
                reasons.append("native_pixels_differ_from_visible_page")
    if image is None:
        # Do not inflate tiny scans just because their nominal page size is small.
        raster_only = bool(images) and all(o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE,pdfium.raw.FPDF_PAGEOBJ_FORM}
                                          for o in objects) and annotations == 0
        scale = density if raster_only and density else max(dpi/72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation["scale"]
    meta = {"version": PDF_IMAGE_VERSION, "method": method, "size": list(image.size),
            "page_size_pdf": [width, height], "page_bbox_pdf": bbox,
            "media_box_pdf": list(page.get_mediabox()), "crop_box_pdf": list(page.get_cropbox()),
            "rotation": rotation, "object_types": [o.type for o in objects],
            "annotations": annotations, "embedded_images": images,
            "requested_dpi": dpi, "render_scale": scale, "native_validation": validation,
            "fallback_reasons": reasons, "source_downsampled": False,
            "source_to_pdf": pdf_pixel_to_page(image.size, bbox, rotation),
            "resolution_unit": "pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable",
            "warnings": (["low_native_resolution; interpolation_cannot_recover_detail"]
                         if images and max(max(i["native_size"]) for i in images) <= 750 else [])}
    return image, meta



def prepare_input(job):
    from PIL import Image, ImageDraw, ImageOps

    path = Path(job['source'])
    if job['pdf_page'] is not None:
        import pypdfium2 as pdfium

        with pdfium.PdfDocument(path) as document:
            number = job['pdf_page']
            if not 1 <= number <= len(document):
                raise ValueError(f'PDF_PAGE={number}, PDF có {len(document)} trang')
            page = document[number - 1]
            try:
                image, acquisition = acquire_pdf_image(page)
            finally:
                page.close()
    else:
        with Image.open(path) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
        acquisition = {'method': 'input_image', 'size': list(image.size)}

    box = job['crop_box']
    x0, y0, x1, y1 = box
    if not (0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height):
        raise ValueError(f'CROP_BOX {box} nằm ngoài ảnh {image.size}.')
    crop = image.crop(box)
    if min(crop.size) < 16:
        raise ValueError('Cần vùng ảnh ít nhất 16×16 px.')
    output = Path(job['output'])
    crop.save(output / 'original.png')
    overview = image.copy()
    ImageDraw.Draw(overview).rectangle((x0, y0, x1 - 1, y1 - 1),
                                      outline='red', width=max(2, min(image.size) // 200))
    overview.thumbnail((1000, 1000))
    overview.save(output / 'source_crop_location.png')
    return crop, acquisition


def verify_assets(folder, verify_weights):
    manifest = folder / 'assets.json'
    indexed = {record['path']: record for record in read_json(manifest)['files']}
    for relative in REQUIRED_ASSETS:
        path = bundle_file(folder, relative)
        record = indexed.get(relative)
        if not path.is_file() or record is None or path.stat().st_size != record['bytes']:
            raise ValueError(f'Asset UniHIR thiếu hoặc sai kích thước: {path}')
        if (verify_weights or path.suffix != '.safetensors') and sha256(path) != record['sha256']:
            raise ValueError(f'Asset UniHIR sai SHA-256: {path}')
    return {'manifest_sha256': sha256(manifest), 'weights_hash_verified': verify_weights,
            'checkpoints': [indexed[name] for name in ('model.safetensors', 'ae.safetensors')]}


def ensure_pdfium(packages, env, log):
    if importlib.util.find_spec('pypdfium2') is not None:
        return
    tags = set(sys_tags())
    matches = sorted(p for p in packages.rglob('pypdfium2-*.whl')
                     if tags.intersection(parse_wheel_filename(p.name)[3]))
    if not matches:
        raise FileNotFoundError(f'Thiếu wheel PDFium offline trong {packages}')
    target = RUNTIME / 'unihir_smoke_pdfium'
    target.mkdir(parents=True, exist_ok=True)
    run_logged([sys.executable, '-m', 'pip', '--isolated', 'install', '--no-index',
                '--no-deps', '--target', target, matches[-1]], env, log, stage='Cài PDFium', timeout_minutes=3)
    sys.path.insert(0, str(target))

### Mã UniHIR và runtime offline

Mã upstream đã ghim và runtime FlashAttention dùng chung với pipeline ảnh được nhúng trong notebook.

In [ ]:
import base64
import zlib

UPSTREAM_REVISION = 'f67fa3f36063bf79b957e617f915b584da543953'
PAYLOAD_SHA256 = '955d0a0c07fe47237f4fd198ed254fcb854109e6293e36870fd647d014b6b6b0'
PAYLOAD_B85 = (
    'c-riJ3wzr(vglvorr$lH97&es_z}12J&xlvIh)r>(w^P5{V|k8S!^Uy5hXv8&He3n9sme{ASF3Y+uhSWr;bPhgTY`hm>CQPpYP3=$s(Rc_YS>#'
    'XY_xMmhoih#aTKD7tzq0N7*8shl@B(y!0}fPr{qb8_m<Hmo35zC@_jAQP%Jx05gv!H(oj#P2wbKoh4_<n<Z3AyuXGQ7n8_)9w*Cd?<!u5y~Q~4'
    'uG0C3IJxj%-z>&ZXs6ZL@wUBp>3lHuj(>i9<iYcYXr4q9fc0b&X5*v90-J%tapn!u=?t0%1kyRd8%B$0fGGd;`e!e^2;(GMz=v?*W#KdufR7$M'
    '186T(?<$Hf#*54g=aDx_!(lXRc`wsN)KAk7UdPL#SqQzx4grNSFNUIsa5UdW@UwZ^k235dQ!OEi1i@&zSk9v$@Z#w#oiDsFNzw%oFFQ-*yZHsc'
    '&LZ_<3|*VVef7-$z?;@IT7<)J5qe?f$=|BjuUVR?hcpv7%)$i%lL)+qM^$JBLkiTIsu%MdiEO-F#FPAS*@r$3qAbTdyUBkq!a0t6{tPUn9zMl0'
    '{`lni(c7m1j%D!r&8tTzUeBw^3OE|yGb4>~Y=dyINF>(zo8U5>Poe|}n+zKIdwlV|_pwKSTJta&CVpqH(QWKKXbhvvco6l@?hTg1@a$e=xVV``'
    'qGW#rqrK?t)xY-!OYsi4Dof}7X}i(cJFl!ACySjfR85zQo*PO3V*~$hu*Cldgs7!ad>svY7WV$f)0b%yoj0t1p*%N+?YFX<WH6qmN&G4D;Y*lh'
    '5f1lq(aOdl;MM!cKY-PSqF$Vh;v`-~K9s7r!pTIdNSLUq++aBXDlI_Na6%8Gkr;$QI2cDkJj4mPxbdlC!^_g;d=S+S&k_&*H;-U8B;GI{EPN0h'
    '8BBv-M-~d8)$Z<IkN@#S&4xE4anPH@*}?~wYl(_MoQ%@b!|pj25q#Xk@A9GHO#vm4JU#iQC0_VY1*XJ+VXCkADgfwy9ljI(VvvJCwiz@$90c)R'
    'yrpr057*b7P7wCv9*B&IFRC85f}fs0dvyG_zXd<PJqh4nC<H<WnizyLC_MpcF(TR^Uqp+5*)GUS0bX?%&0$n$%Y|<d87I1*X00f>jDcMcq+k2x'
    '-HSj>!lUD-C&8<q-@X3%U9I8Owm*<e-WEc5Tk$3fSZ=E3Mqr+%i_D*e1Aq}_!7!fpkdG8X_!*}9={Y~gBX1R6&t4*05`Y5I3TLw@8TwY=PD|>a'
    'Bf6qR9XnhtKn#0`WMvb2r8=xdAgtgq(3Ze5I%z$Z5M4u7H-?OgD+0&KFwpAwa%wfWoFon}IB%klOIU8behQ1JSi(J7gJ$y8IEp47$gAjL4)4DA'
    'k_h<MyI6uq_xd-qY$vcTw&V(iz{LuJQ~;)E!2+Lox%3b4*;x>m4povCw}9FYWGh_jLv#atA(p$AnE1GiwT8>-EYrFS;&>52@vQf5zKrTH@`H2;'
    '6S>|RN7us`#t5h>P=VziDFl6GN@+t(-=aY}AL3UYWGsV&i^#wbGsy|t7NP=^!zflPptGsqHsmgD2V^I8+P6#L{{|^^FIh(B7h2$Zw4Aq46Ulr5'
    'Y958dU=dv}eCj?Bx>s8+Mh7*?M;ZnQgy)tdVy*7|rKfmeZEQqua(F190@8O-EXJca8rIhD9J67MzJf*;4Q1zmO8~P5HFR2|VlaUVxG>$b<p`D{'
    '>`$$AHJ;Qw7+1uSzbcaBYf%A|F24@{1Q{%xSdrW5-z=i6c8&#o!GBbto)_WQfQzG#^DxdL@0W10j85irP#H$G<8(e>&K91fpwb*Z^geTMztrkZ'
    'DrXU`aWt`zxpBI}<QX6c-!+dI+ZO(;pYE)S<Ff>nWe{L}(84W5d4LcBz&3|oz*xvEMu3OSlx8Shr7+9F%cw+{iTX6X43TyWS_T*kpm2vHpaQA%'
    '!26(@A(|jGK(K%gGX&}8g@eU1oUoNZtZ7KGZz5)f@H<Y}4(fTfBq@|>O+UaQ;xm^sc(9SKVcieX58N8p0ZkgF8GzIq*!J>~5TFSeC}wc2TJE?w'
    'EL7>_5+?$1GXu`wX{@q{P*R6;B$FtHR$0K)%%?flT1;m;53qgE8l<xunrvbm6)3GJ;3Kfizta4oK3FVUVAq;!CVAMd2R{ZMvgOp9#@RGn48}Tw'
    'sx9a;=y-$3=dQ`|V<}6zsAPbi#g@IC>J8C_9wXiGNOJT9wM_BCN}?qr#gI>dpC+igM!*OI?emjxl0|9;(Gq3zKym32ETavdkf!GFX+xJ&<qFh*'
    'h?f&>E+APNus9+idfmEr->aRS;UZ`+IW{;+WJW-Ip3OmmfxvMNAqs^d2pO%DU%p4_0qX+Hc2Lqr>0}5qUmElY;0hu#Ljh1j<8PUb^{FGx3o?pX'
    '@I=?Bf*7JlSLzy4SK<d&%L)ptRI#$b609O2AxYx_(L-|>-4+5dB;H`ML{fQ-zEsEh=tu!^ZpbFx{LPg{DN$A{pn?T}_KB%8kTk+7(=drgP$Ao1'
    'Ch<6)2Ld0VysXwLF2hL-G>rmmgus%;m1z1F&k!IsJ6vG=u)t}J-8F#kGK*^GEnv85W=I|o4bqPx%_*N2Gn5Y!o>M}+;a<ok?XT1lNQ5xCAg(eN'
    'EjA`iwL|sew1&(F3lx_Eb>Ospep*AviPAunD&1Sh+|3CiI2Z_mOA9nBbk3aXXjp|IGsU$5hU8+B_Whb1u6(eAEf^HtVu=P(JVR~~?I4FFAdDu4'
    'd@z+#tWm0!7L_&uFB%q9QUbJ=PRcBTG)77&h^$ZY2bj;N=Ppf}zH-TI$~tVV*7`M#6Mq`c{A>a9Ru02yEw`4@`uhs4G$*KF4w}8&R4@Z%`E7-w'
    'TB42?luQXos+nu!oO|S2z)t=tP(s7HA1B-FIjPmbyl)NwXWRXDj#B-yCmgha#)Rl0p~Y3KTF9*E;_7tUR!>)JcDYc+{#RJRre-OGA<gwCK)#ph'
    ';z_zphC(TLnUceI5}~}YmKQO*wuPX^THkwDVZ2Z)$zVB0J&Z*%2w$|e<BZe?p|J8Wp$w2*q%<x|krddI8(Ro%sMJcq==W>$KCmpPsxUZFIJGR%'
    '{v;iIpcw)Btv?B;{b6`0iqgdL54<g})9ya-wzj-3j7h&%Ftr%1)^dg-+9#0Ka>xY?1(rK$3&VLdA!gJaH->JTE01E;fiF-%E;XvYT~15~ERM55'
    'mJ1fCuj~CK)(W<dT$BN8!H6Zp2^&%jM9Fy3-5EK!xr6%DMLEjy-QNP{o-H$4;~QRh8OD<kETMo^I({oAgF_E^g#09T8|LWUKw&lk4b)p^xauF{'
    'cEgA~rtEr&Mk6%gK>De+yf;TL3_U=!2h_~{T1YqbGb4k_1L+gVmx{R98(Bi5CB5!>U@4-rEQ2XG3_y@d-28UEj<tcG!vzSBGy_3VC<XjWZ2|;<'
    '7Yv~H3l?P^)xJeS7b-vpQX+ImASsY@+ZAz%W1mUY+^4;xYz89TuN{IA>(puP;A@;Z#GJKll!V*UXo^l@L#L^8T`Zol1J#hM3N7iSLw%JoIIX>i'
    'rbn`8puI2>bx;oKVp?j@qU%8fE3yCTEwOFG`)hO~e&x(;&s{WNt4J}4r?HxWwe9RC+a6`xgNu2(oMCTM(7UJMwcX*xJf8VhPnQ|3;Mlz?XhJZW'
    '5Q~c&AqG<!Bucz+yS56H1w{RJgD{}#b+1WS)vKtuF70d%?14#`_&E~9=o%9eOS6Z9WyePt@r5E$iF5kJ(cgmqdiCfnI?QtoXE8=vWz|(*{&IXA'
    'ynFiQ<mfR~@7C(3aO`+L^fWx+p|dC5TJ%je13w>B2}}pR-gF5oS3mMPtyX96d(VxQUh09As>~O`uW6r%3S-#eF|;YZ<+WS;yP(Z(dw!>@ty?=R'
    'WgId>`rvsqy4;}I5{{xGN<+RS!3GTplEA{Kp;@mNH=9s$NrBeG+Pc`Ehf^W?EgY#mg4x4Ap!8i127zd_C;n~ZgBGabI?9Y$e>ucWomn);62V2>'
    '2W85}Vl)rQ)z>3Z%2mv%pB<`YLrThg-O%h0bbX<xyY~9;@1DMT`TFSHQ{+B=div_cN&fWn+mknsj^3Ua8U{dY&C;0<a2hmLb=9s^Ik@)AS6Kh;'
    'vv()GS_ggR4D;QS1M&Rqb@1|4@a*N|lfUI}9zT0~^yv9XfWLpi>aSh~$4^g=|H?(4J_(-w{3v+!<jM1AF9qOtZ;oERee&wf3n*@WfzpqU-a!xE'
    '6+a*t+1R_2=O-^t-o5!de|_=#`J<!bzn;8&+^Zd3Tqr&H=-s=Mm+ufGXzlnUc=YqL=TPL`o1af|l|!f$Uaw5oka`6CTO^&FpFPywRh`qC<f9sH'
    'Zt7FiE~qnPzTG7_9~zP@RBFN>{wDi81g~d%x3%pWwI##$xUjO$z5Db+w-c3-Zrh7O7cJ_v2->pd9Xe@lq0yV-et`s}?HK4rf?lM=lyj)IFXD_;'
    'MgTSt`vzjR0~$AIX@jPV@D2AnZD%5qybID(yCA4+`q(R^@EV&31V!x~)YtK|7+VJfPGD&65+I6d<c1L{08uiyA)n|Znr1a7wJL)2j)ZkTn`5jE'
    'h|^`FHkb+vo{1JlDWP59<xR0m3|5&qk%l5-T8-3~vwKSH@=Ru~^VyV^4xWbaw~(38@Je^SS;2ordl;VVi}(8G^)nAtC-frczQW_5-+|b9EB9qX'
    '4>YUoBwFBxJma#$ZzvdAiy{EFR_lvjuO^}GqGfjnxv&AF2=orzW>Mw~?X>*#`e&skgukM?2Fo=JYtLI9%5ZzP8WllBtzw6~Iyynqycn298o3=M'
    'J@8yW6Eni2{U%~#&$WB@y|511Z=ZdLK>;4h-Lx+*&owFg=^&i&49yIWp34E{uD13jlT{IoM7{Omc9Ycm^HS#KlJ&}xBqe)A8Z_r_k^cQQ^0ju%'
    'tipX1J4WOm)}^U~0@^N@!^y_7fV^r`RG((?4F731WsD>~2`49iNYW<ms9dWj5I&ixa{)6ZYAT;7yu~)S)51YVLMC^|)pZAOuH%;OSlMKQ57XYL'
    '_L=7Fm(Mh3kt><i1=sLOfr7o4<I@mA5f+s+0TsAiUn8RD&B=fN{OrvMN9KTt$K|S16zJtXKC##+`RUxSCgfWlyoMj1Gb)FcN6<k$XS9$xX<^*}'
    '0ga+=*s9eFx;Bl7QWIaYPxa`{@zb|1!oCCj@o8Qrc>e6|yS!XW7L<DxgE?+xmWlu|_m>HZG!lfe1{7pTX%$&pEO00;hjOw5Q%GKPVTK@z0SRs3'
    '#G6)hy+FeSqXK1<4U8uO^`lqAtsNsC3LQW*Qu}78@hK$H35sZvwK&7Yi=OhJG4x|`=o1Qqjx>(>-D&$A`#;F$gYCg`0pp)-^Wqw*XYp@qT&p`b'
    'K#Bt7nbI9>u4BS%i&a~=B{l8Gz^VmnIkZV6&sH>fgbP2<_H+q9(&c1`9&JzR0w9v6>2L`%f~|-Q_e@8$8p-z6BBF7Iv2Ym6g^LsZpq34Yol$%t'
    'XQ3*-Dui_NV9pD4b>{Jzr@DS~E&v9!%a4;F!1zrh#A(en*HPqZvA(KW##Nkh1nY3DfCx+vI4IqS6Y#b1<>6rh$*o=Y7C)ESV3Qw~raAk@Xy166'
    '!n8}1cwlUO6mV%xQea524|3%IjzR^>=w9M<Iz=H35@PLzmN($&H4I@t&7zfSE<6y10uDzLz<UDKgejpf!suNh-W2n>ey*!f7ffR#f`S}Qg2tjp'
    'J@Ic{(YkIzA&pS?*$O>V9KUdAnkYcCX<#lk-+OA}eRj^eFV?L4Tsq|d77mX0**UIYD3z$OLR~`Plj}~ei3<b~G^Hk63~^f#Akyz>KJRI@-oAVM'
    '>gRVlr9u!?z<e?7<qNhvOY#C}w?XX-=vC&EO=+v5Yw&96o}E$(d512Gq7UfN?$)b|2*<a0wTLiy-fMb0-Tl3Tx})v_j4bM2z7JS$4I_#~<EaA_'
    '#Kj`()kM^d9h^tIG#P~hwa`q$r5V`n?CjLXzr*|WZTSQLIR}W?YGET*h9MzKBRQ8@kEjYZZ#?YDaf08_J)slMQGxSqK>__0!_}aTOwCj3;8M%e'
    'TFRqPE>3<81Kzy_;fQmF`+nN&oYPJX8hK0&<@1--7Afc~cAbv9jzt<3rB{%^lKt!ejJ4wItQfm>0=9UNHjw6P#>ZJE1~*f+S*7M(4YZX+(?)Vb'
    '3luI-F-=PIptx12*?|r&(nY|JJ#VM&sA=%-O0<kQgIf35z0cBQGT94+;<ojPSP0&q-9vnx-<XGLYQq>eA>wcnf70W(?Efk8?{dq4P$pzSm@9O|'
    '^aG=~)gFDx=!;AK)Bei$Fk3}D8cM!WS#In_8nV}|2pH^zC9+8r&HQ$2x46gmD9oZ0`U(9i#{I0#CbHm%7z>tv$M_HXd(Io=D@FgS=^Icgy0N~A'
    'I`Y6_O~;p~3(9IVblYO^M<Er|oTG)}D?OaAgk#u_rPQoeZl>kEh0!ftw*?aT4%wZK9tnGZ<IZwbwYf!pWzAoPHW<}p8*ByAwG$Z;h`W3u0!%uM'
    '2~2MF{!We-!h+PgDqo8bf}e%9dFaV%!qXysD~Hl`rNH91DZF>M1#0YYfsdrqOI?<nF114wyM~uf;zO@w(KJ^_qnN#J)(l4umR)1n{vuuD!b2}3'
    'NvlF?$%c_i7)E>IGuiF5Ejf%wqeX@aw<FxKR%t0SWCd71`bXOc6Fgm96Wa{d`9CUfeXtkoJ)jb<bA?#O-)VTAomH;$cNnY89ssedXB1@yRHC3y'
    '^Sm=R3)50R=gfh!axzS!5FT;$qE5DG3^G&|d>|Rjl4LOtkY>Z^B1{4`LE+O9w}(qB0&%%sIzY<PZzlLOnj;(}hG_kkLzZtsNo>|0#nQ-U&DUHJ'
    '11-0;@KmwrLm+X;%&2D;WlUt2(6V5$oFXb0bOll8R6^&bgDSESCY@P%{Q&M7JjqV}&T-q~z)i0tdfV{ki?VkT(B1pY(7-yC+l1<9|7QvQOUru`'
    'je!1RZ5nQi35rF5a$bt+8tT1hTXTQaMxJu{I(OL$ekAOgi*BbiDr#(0xa7yPsgM(#k^s2`K=WQZ<{6MQ*7S#JZo?^9x^ZrV-><j-wt3Wiaije!'
    'cIzy5DViRvR<3{2!_x4CMq1c`aA}(=>TK1avd=Z760EHF4=OJljD4uDHSjvRjt0vGMHup@`~`ev{yFp<u1h+HsXm$B%bGfip4Tlqh)#Nl3YtG$'
    'W9v4b1CCv0-O8@@aWEJ2aF#1Z0lD3Sp#EjUL%p;WB7>2&p)}V9;~)jR;PnMf;-Ch9p|R7$&fa-Gkr=QYl`KLe#hLgxnJpLA$%}&DUQB5fM2FTo'
    'kD?4+w!rBfTYHCwxyaZX(jxt(7LY`%j)d@DmNT+U)AebAG^4>zYkxg?bM!V4AyZ)9_9UAcVF-wXbCac@=-WKcy8b1!CA_kry#NWDO8bWzupTut'
    'Se2S`iVJO?`@DMJRBe!uov|Wjs9qVPbZ$sGetWq3q{g?lZMGPbm6Q-h8BoF*@3Mk4Re><(*|j1G%cHu^^&J5;qzflk@{>VAv(gvpMgju(rZKqe'
    'HX^XKW?ADVoK8e;nd}3pk1gJwRIzJl8fD`YO`>pl2_v8qq_CvJx&guU>2E8=cyhnn=+;F(7=BI3WOlkB!bD0HDq`x9N<kJAtX|!R%XksMaM0WF'
    'F1!RKBg_<j^wjF(9%?d_|N8Y_@9Z9{E%zMSu1w4`hVi{@pbQA-+IBgJ>?9kJQP2pl;}C4p6s8+|f}hc}9}Q96{9XnAF?PWOH>%LZ)<~ydnBX3W'
    'LX(?n;U9SP&+di&!P&jW*}YHW<{&w{M+noq+S<_<c%}&%WF?c+UJ36IN_VN?yPPFI#WTN}EoG%bbc=liY7<q;Kr0tHBI$&I<*Du;pt)ClmP(v('
    '{tQ|vF6<P-D^J@UioU=bp|iq5DMIaXcZ9L?sC_6q#4L+^aUU(GHKEeseaEMh-EQt-?#>r}9gVNd=s%{gkYwwf90t+b@IVTuSG;4+yl4ajzFma4'
    'C4<W}2d}Ux!+8QgcM{tDW9V!a0gt?X{G>stEVLA5cyhuF2>e}LAeh*Vj%;gPDP($cdOm1F4s74&5A=A#I}dL>NhoeHRFR9IqX9vL7reN?DiPs<'
    'm5OIR%*N-n)wh4Ov)$jq+I1MYXr(K2p6&GSL0Sk!@+!HQg8-sZrk_kWq^retiKklE-3>SOEKZVQl$i*u`aRE+f1_+kWrmg0oT}N8bq=eFgY9~9'
    '>ZrUGc9JZaA_6STlnyGAi|5b|CFFDf(p$wo$nrd9rE!O1O{&ToCZL@2=DSZ=)~l+SO^_+nCT&ZT|3c&H)YgloVP}=xb4QUKE_$`g#k5I8Y$_s3'
    ')2DffhgcjLOvHY@+V-r9Ct8)4q~Tdwf|ag>Trc7TM)|PI2r?{f<S#KCI)6Zak%*1G3%Aj_M*Lbvx_*v@DwpPU*wD@JhcecciF(Tu83uD~WL%Ua'
    'Ot4TbOOH^I8bMR;3tGL+YJQ0$r!2hhbFSl`@-K|~+DK;>Q&hxqN<ku4HPBr-gi&ez(lF{to1#4Z(fUzBJM{uWM5EF$7%Z`2Kl8&LGL1^P&^e_S'
    '`Ux@-{#+^Y1n;%Lx~#d4A{j(&+C@;_wvAw+q|;t&XQA8{Yco<}TWg&(ZjqC>^^bUl0xZwWg?1m=)zjIWGaB_&*=MhVx9?uPIeAR+aiVX9bJ^Tu'
    'j*Ao&7P#Nn&Xv?xRud@!OOi4wkdB0mRiHIi(t#A$ruw=Rs~|?6c(k>E;X)rny(If+rFP2OLjsfs<B%uv^vu{2x5+D*tmNc~oI9cvg<{JZ>a7)9'
    'oqrcXDS!!$@&4@O4wf_2FSa$2&+9cvT@b^dZKv~_-lq9f#fq!KhLG^^?r@dl5+o8~lhi+`R{x`kBU8uCgLWFO=<bF);Lu+9p?>*E5_n-A@lwaU'
    'E-Leu+UL0_epd-WYj$h#Qy^Y}Q*I86U2bsN5AqEQgTS%tEYK50OGCmm0|L*kGkix7Cu-)E)lac1s`b>ObPY|x-~;}@xHQgs(gE8!33N{9-a49J'
    'ki+*iees7;Hkil6x4i%Z6M~?wS8avEVGxQsKB!J|?udwV@-n|Bu{Gyxze8hZ$PGkju|*irlW;j%^u$&dEH>L<WdveIaxLX1C;`qow6IKgrZzm0'
    '!k~t88YTu{2rtLcWY!as%*dLTCvlQ#^2Cxsio*@LYZ_?!QysN!R)P@W&}2^3yIWn0Y#&bQHNOCHaUZ4CIE_I!?47QNci~_f3djK+KAg+2P8FQv'
    'emji%ljVHQcA0p01tJT~z4-@dV47ZH>}WnrtMHJD3HAV^Ub&}y>Koq?!{%-ieAOzNK>A)^(LSO|rjbOi3`>dL=sxM<qSIDqjFuFF_?ZUaun<<Z'
    'R(l*kM1q9frXWJ76p!i`JrQN?&Cf62J$rEyaCVBfs7!%ihGyyP5nyAbPykGUSV4etAP6YlA_#E)2Z21n$m{i6GILIVLwynt2<KkIy9buhFrD8!'
    '^giFi0~aq&TGJt3t?`%BpOg6Mvo~HpOcD$Vo=ldwQ%-YYI1fh)K94FG2l+9?Gj<Ut3gAQi4cbbA^;gOj{c?_TAC?N?7Ho1W&C70UmRyjJ`-fqC'
    '=>hzU1TQv-F#PrG-jDd@JC9ML-`{=bJw|LA-Y+yC8{QjgB0BWmMw3zVO@zVGfb-KBU&ny5_e?|RjdoiIdwLN%9pilQ{P_!P<8*85B)OQx*?4Q~'
    '+*eepd;j#{)7f&rf6zW#J{axpz~87lJX`MV9_;63gX5Q2oIw5M^yv7x*KK$AycaMG=l*!Hm}Q6C+wt_Gm5t+QGK6&{-R_6Oi)b6l23WR<Wt#^_'
    '-A5<gdR`|SOu_{0L0H{xTIu{^8_TxfZF7<)7p+cr_rZ3Z8hstkqWOx(s7w=^Jd!}0k9InbI*)GE<O}5P6-`o^CjEU2YjxDz>pnPo@<eOz>gr0&'
    '6a3y9B-{M7{pr*7=+n*3HnW20B^k%7x=DqaJFQN;xwF5&cl7B0LTAz*Th{-9zmK7N^WN1M)^`gP;l#j>bCb_O1}9mxqD@h%d3+o^`g^na@)-Vm'
    '96V^Xo1hp)iPp$q9+CcoCy~-5qn0LIJKOHxG?PIy8E!*O(f&{I;_0$aT3}X+#0Bv6vcFA*w*T>uzdh-61l|5`wz!#qKAjHYaH65XH9&uwn&pX!'
    'x3>PiyotSkVBp&i*>d*de|-0#`~45w_|rREw%h%E@4uD_?W!!x3cQX`f4NuzB@UAb{C9zn<ja4+QafIT-rw=Pc=sF%UBL?cUvWaP&X&NJZ3YHY'
    '6p(%kvVQCxk72RH2~L{@3%vWsdlwDH2{E*U_Be`W-gDfKM!|t1?3b8wdLM%>up|liJnWN@vJpS|h>?Wobf3rlB}tY4Nc!3Ad;aScu=6k-dz8&U'
    '%csTw%XChlw?P1sy!+AH+7iMz=D6KaJkJ*1Qpl+(EDJGA3ut(P!M*S|xmcnd#_J8PJRs$pi?j*<c;j3UmPqE-)|U5d!BRf+h9PQPB<<;1AeQNq'
    'NpwM{-m$Ng7hgD1_&b`dNyN7hrB|dgqa}y+8=?S7Y0;nMWCGeGw1TPFU>!*Yq-o=BhwSBO4oe)KFK)5ihrt>i?0~Qx4!gVF%jk+Orf34KNMg44'
    'o9$ib2hzIpz<Y-qI4IzdgtKLTLg!_?Z~^5FiM+Md@=k4ext7#B$9j*SY*ec?8;&rVqzvt6(%sE*CKNUaEB~B!bi^`f?7GgQNfgq~t0Yo)zkA@l'
    'g2B0rqbp*pnQK|~ydOYnc-eda@AHK@n8mY*G91FXe0C3NF5tUpSPso<j`AcJ0C5!$7h?d@X}A9mHT}c(jQCtY6)pkbj4=Dql-F7fo!~OLX5-=V'
    'iDY=XLJI~EHzzWRoFO(%cnghg0mBHWv!v4kx|90Q{An2v;p1~nr^$x_lld+tH&Ww?<fK5PF4UY5Jf7*3Ln=jh^HvvIdm2Y`a!nw$at$RyRX=YK'
    'Xb@qip-rJpWtc)9dx^%?aG4B4oOAGw#u6O>xJ4dK;WdJu+)!GfB!i(r7nL^ukm>0bmT5Scr-b=5#$S`@63^J2B|C`dQ9ctj9Z5$K-6`@M!H5`A'
    '$N~?8e9!@u@-=5Nu1KD*i!$U!kajrbo75~?Lg~dkoQ*jnDXmK_sp0)Kx-n#`nCeH015s5&IR~OJRELi>nYV`W=i~C$La?O_L6}tyAT^6=B6vdp'
    '*77V>Ye<G=bKSVbv;Jpj_DM+IrY(%ifSyOQ=2c8vUX+=y8I4BqAV%=$$(k>zdULTP$fT1;uy$e7V-N&Q3OD7YhXOg7IZ@;Q3Llp!O)}9qwEYO4'
    '$Jp`!@5&?PK9~2oc0bm8Jy#uOS*{6feHFnr-Y#gB*XM3H7%X8@@Hk(_87`Q(YT|M%c4xc{STn|6L*0CFQ2ac&DJ%kmXb*>zH?;BNKh-1X>CfZj'
    '1N`%r+-=@J;cK(mRR3XNlK806<7%K3;0>1msX*Lr9e~zVHdl|4D#&_3Fjtp1SCflPSSt*`e@}a?pz6Lq_gmZY!BJIm4}8go;lV>7;cN!-=PAHt'
    '3<1GTx6^t6|NpSn?Q{<xw%ee<8W5K=WDTeovzmcTm?In=w7vgc#vobVqR&&x29V*$6saUnQh;6uvj#uEfB(K8X5+Jj5|&V*RSGU9LODwYLuFzD'
    '4A~OM?_*TYxhZY8(`XC*wV^lQf=x^xgX^fgBo56EuX!Vjp)2YV3v#Km1Qs^wulAaA=|`jRhSuVGVc-K|`Z4HM=*E@-Dw11h^s$^#Kq?9p((-kD'
    'Bo1&4m=?V3Dz`bVFz;KU?0Z_7X7ltCL(EYhl$s>2%<1xC?7bHrF7j8uS5p5$3j=dJwdHS+nf5$?dG_`hCd_<%^2@X1lefLL+_{!q>z$pUYSV0j'
    'Zo}LE(RGOaylnf)+o018I=c^pZtK_Cg(?ZcjdMtQd+Hr+lv4H9HoIXm^)1KTlU-J$^p5U<;glf=16Iqcl<yx!_4d7j_WqDpH0EK_j6ugnIouFA'
    'yYUOGU>E41X1T6TT78c-Lk~Ut^iIb7$ZB+Scna%3G8!%S(fq<zsB-<F#Wu(3UtYd?cXEzD|0klz=;;)f(lGH}{_^bcvm@^aG`Qvf6Bbdp8fju%'
    'TQEz1dgT4`=I90J3bY=|@oRZ6XhAc$8nlob?-F+!sXq1D)0Qy0;}Ne`Sx);Ppy^PQT5iQsE0IQ~Ic9P<=_IoBp^61T`ny5!JZ7&Y{mS(LZDBZ6'
    'E5rMod)`C&_hlS}AD_{{Nae6ZFXlc<*oxl>_#Xa!srp*7>}nlXa1@UcoSd_;i)vzWkh{)btQ`>V=sKc=s?AFVD<(Crsi+`uR#Tzq6-^Ts42CF='
    '&|-@os?}oylLRDWAFCL)h+2ivT)_K?08W*ap~{QoThc(tmi;VRWQPi|=Hs896T!$1ZeFlHXsSi3?9;bzga)P!7-@XAvZSgUag9>tNjNt-#@0I;'
    'P|_NN@P^z485tNqXOR>%N;*(jHB8q?B*R5(ci^c^R3TjsFuR%KVgPNTJ2=ZPIaXkV$OhAetY=i(7Ic}}P+k%o<-MUs5l=9$^CX<nHIO(1Rd~Xz'
    'mw?_%a%7JFChTv7#A1&EEDY@EqWdnE(nphf6hvWM9^=I_pZ1_-F8i2=9?d|sg$zAHE;86Rz>2JwhZ)}6MoDaC*s*uSbx0(M6H=sk(w0vIGd{hE'
    '02P>&VUFC5#}mrL4AcB27#0;%H#LNbDezU6OH}UfC>-GFH5Adfy%=2y)fyAVg0_SFqg4hSI+{_4Q|orR7gFsxh$qB9<dlS{9^vgHutJfiME8>*'
    'chH?z=tY5AS$yr?K&LZtUG+G<@@{~^5CHwVh{CBPH6?|Ek$83drlHG3?Yv1ixyj;8a}0_3rHZi36yZ>01ta2op$85)s9R)j9MMgDgo4fl2z!PM'
    'jfV0SXhLA3E1iUFRKdPOsV@O{d|-%Ba+x7Hh%H7bMot=amYFb4hA}gxcSK%1X*E(~eHzyk^C4o)@)&qVt3tk;%a{*VwRrmjuNj+UKa!IX*0lM}'
    'k3VE&P5mfk0vm(NxkwyI%aX?tnLlicvgl>x+EPw9XKI+``5JjIK0^v)AcqK~fNmX+(-bfF0A}Dz&!8b{`U5&lQDtS-h5{{!M0Ah&dnqKg-&1^f'
    '%X>CbgeL1)w1B!!BpjRwryn7yu>mzK?*;aB0)PSv-fK6Xw|Jd@OQIaj3w4>fiiLtTnpjD;=`;pW^8Yi(8&?7hc%mhcS555|nL=;h7!}~zt^lu^'
    'BKHD+qUjAZ!?}c0oMek$nyq(=v}(6I?`b;iyocX=qSYg_(IeF+@+_}v1;AX$0=V;@*XXUS7oZxFTLG^UQAA=Fy0Rq@YXvw$*&(T`d_FYr${=WY'
    '|AWiRe)Avktbwx=Z@Ta>nHd#+Pw24jO4reFx=vA_Tv-5CV<X}q1cQm6bMGH=D*~a=SiVMywL!^BDwaPtRV-bZy2W+}-R!9WQFk-3ra<jWz7&&|'
    'Pm$n8Z#a1M*7F~MsY25zqf44}D=`!@tuWM|ffkK+y9@vAz<&?mzg_rm?~C`NyM_Tw46hDEtH@uKdGbGT0v*UaUntGqYoO0;p->%6hiIsR8MWQ_'
    'N*VKh*_UqVmz~lt4@$r6mVVhQrHhGr&ieQ5H{HUUof4Xz2;LNy8g9MdE*5a!m11#g8rbp=g)M1XBt~=bD(9Xsk3c*RhvG~l=KB2rlM)SX$%aPj'
    'EuOvtJD3zXhfhM_)CEsr=me_zxfZ+)Z<pQO&H(?m+x*un2w7(-T|8piv2fDJZjvwCDo*J;T>$^r+gC4Hu2%ztLELs^_qr9Vqc=HT4b21#;2D?T'
    'IQt?A@W6n;W4|rV#1&hxdi<HWt|8?6m-33(yY)-!_~)(gzMQlA1G~Dj)7zhmAOw3qn$y+`2nJN0BDJ<Eeu|M(Pf+In{O(EfARh|UCs2iCHKgUe'
    '=aaM<o2l=mpv7b%Sw@B@O_=v6liz#(`|%P)(0j$*nyZ`Eg!dP-#d}Qz&u|MNV=o3;LlOI=-cukPz$3R5w?x)$y(O2x_mojAlPV?HubH>CC6g;i'
    'TUPZ;YO!iUZEewHLmh$aIavDp={58(oe%Mr*od{c1l1$lEF$&3&5JEqqiuG<gmYB2KoY$F;h#`Fpg7Kd{&?MK+;ke(-NsF~p*D}W1egl{{PBHD'
    '?g+!;PsTP10GJpNZd{^zr(TjjmyY8}OJ9g3vUVpau5*uYABeq4^hPOQ9Cs>cBs6~_i9tjra~}CaY*iO{fVDm(s0frL9=~ItVWxSzgsH|EC#gDb'
    'aU1PZi*$mM^!B0fW=Q`vuGcNGbFOq}Ym;CQqV<e^5ourl_Y!x^7CiQp%zYlSz+2ZV5dq7&c1J(zkg?E`dRgV05C~1a3Llsjof3Q$V4dn(BbBIW'
    '8}6g<Y&nYBBkv8c*EF256@Ges4%DUn=iKQ>Y-oQiaTM9bbQh5^j1M1rM}rSZdNqlL7m@PtLG4FJKb<^x#<AeFI74)^G&4{dx2YhR!KYxYrB5Hf'
    'an^c!{PVl!<LAE|KM(ucq9E6L9FIn2wQiziaxoZ3$yK<mDm;y|$LXN+XmvZXB-eWpU4&1=<l+8bSJ%wT)(b=%4m=TuL=xy&x`1*=wTLi~1|mu^'
    'HP_gtJ3T)39{n8ym|i~i9<<uG3S6pk1+}~<bk&ghu*8kf8S?`zA)tGKnmTIfF+a575Ji)#U*qIIv%zxFiiXS9AYo~#dP>`K;X@AgI9)HwM|n|j'
    'Yh*aN95lgTZY9y8-tg9mDbfQmI$besj$>A2@Ece%d2kQ(cC@4jQF38Os^k=i#qN`eSs_=AvMu61taWF}e}YIyS4o8Cb@ura##l8$CQh0ZWI|%P'
    'fuaNC6Rh0$zn=H`TgFj-q2@TcNi;{1Uk#69$&mWb<7Gp{jp3vCEaSxT5-KJPU;Uj5$&fI71MJX{yuq#Z)AR#(6k2_q(**|#X}R|R&;{)hH=<ue'
    '5skd~bd*tOoC+ym)(C}C4(8s)62rs?U_<?qPL`nTcMiShh%fCYXLZH^W&||B{r+`1p?By$^CanUSOkfNLp&?Nuit`|{5Xy-(uQ{o!W%G1;;?~)'
    'MHL@+v!jfwNmm`l_kUg+Yp%PTPdtUmCi~r7eVE31^-jChIoR9X*}m*JY6x6O8Ie8T?{@Z@P55W0^W`l0A~aD9O-4Di9R!$;I0#y^8w^WU@9|GS'
    'x4Vfi-{lewXU4AgND6%K`C`~Y-q@z-jEwHSoy2tYCq@&$ef{`vO&Oy3jLy>m-_8%cqZvi(cU$c<nyMn_!;~)Y6%TWOn@-OXaxJ`m_FN{0ATN_v'
    'NhFG34+Lc_NRpO1MbsKiqU-!nkpz;?gK3z3V5dF*WK8ICWZGg5G7h1~kG{$gIf*dNj+{8+H(l(2t89iL`<OCKT^o(C!etgt7)Zb2fx!6~#KY@`'
    '_u&%%HLrB1z;pVJ39h+>%yEyv&}~kKFq1Tvn5_rYx2k3l);OhV=NR1X{?72u>)wB?fB@*W>pH4*jv=}QqW{nLT_|u1h=;3ryiDDNciq5=4X919'
    'QNwP~N`EU_lFe8|T1;o!rN{QWJQ-))@O<E=3p|}8PBp}Miz76HHlmNG2i-hp9C_!{Q9Uot^F#ue(;8xhherqjnU=iS(avHO(8im6Fq_mV!94f2'
    'JjSnXwZ=8fHu0<kH&tw@qyYwFG~*;FkFUCB?j2AAQ-&7MueTPdFJ_}oe0TklYG!6#y&hQb{;5b025tNPq=CmF>vAwB=J46l11-{Cr+)5gPH>(Q'
    'OTH9Uz9vc<E>Tc#6lYySXZm_B;%1m19F!4XghJ3WU3jSuwb4-``BtR&ph<IuqfMFADAAcXe8joF#@_$Inwb6^Rfa3|D0BOu)hVlE-ZcQ=BUrrC'
    'ch<oFzmh-eW*sJX6erno>Vu&+o?U%x{y4sxeZ*{HT&!NzOtS+(x3=(t;(>YACzeYJ69<!DjwZBbL_?MT48}hT$bmki-6S7l7slA|MR284kgvFa'
    '!4K$Nl+o3{m<LkQR{sVN^hYKx046GEFsT{yrMws^odafLMIzkuzKHPN5u=^DQT%E}@hf?5F3qsxP}-LbogVU$N}R1_J@xLZhUzUrOW!2AfV>vj'
    'DbVJRMfN_P8xGNHXH&A{ZI`0b@H#D^bX5_Vokv^<4Ku!Cggai->qN~x2I(5_BAly@;9RLo0|p;pfC{+qdmxMNz__8238WLMkov(@6Xw-gFt1j?'
    ')EL%6c}>2xkXoC4<q^jG06WTDhD<WpuhtsT`7~Re-7^!$3C`t9#&ZCmLbKNn%&C{c?Fv6*jAmpfa!gUp>}+O-Ag`<(XTiIWBKSbdYax~kx&(3_'
    '@fM`m{?6UZxH*Dv2#D}Db5bvgIe?y#B7io4hJ4=RLxfISG#HMOaX1Lq!)24&M9{4wA;fA1U7=g1rW#G<DOiL2ilsh9^EC7QynaK21uG|K@)Ycs'
    'rV}+P*77`nMm@EjQFcjfFl9E$5?xKtqR#83QS-FMOFXJ-mda*kRacc@Wd*LGN@w@1^Xw)eP7(1#!u~#@Y&@PTGI4HG)KTg-1zL-FjQ6pi$K^rP'
    '?$h6gXUl_!!|o<6(@$(s(+VBO@xK;*Ms&BccK|4khSAw_ZztSESYL}Qe+BltYn~_$Esg%Fkv(K`c!yq*ATGAIFCKGHK!a5QT&H#gd8%e)Fo{1z'
    'zFESYjFeWl-og(cL`E^Fy?)-<|Ff%7aZnGNaWYaD-V~H;qf#!yd59>#3P|nfdKL_$L3rc81!dtvNO6AEgJoR1+Qj;bg9in)VS9-evXMihh5z7*'
    'bkIiM%H$>{`}ae#sC-<MEy$RfkKUuJq%}*ge5(Kg)iVcCdS`^URZ&G6F375^61>$jF>A<++z38h9VFBpXcG9MgPChVZT|p<&1qZI2D69f2Hu8x'
    'pOO<|zC?y4R>d3&-V2I)2wjzU`u5z)6>!SXPUHKXb0{XC&TZT=FC>4g6SQ+<>=|4`RM(d8coKlJ*PUPV*(lP6m+gprTE6+|(b3sGj6dNow409!'
    '3(xLR{$$EtG`K;tgz|)x<oz+tlsrE&gEx{@p?A8+uCG=ST~TiP9+kvKP%G^TlZ>?10iCn<ySuv$PyVUf09zPv7)c8dXM%)*)4;Ev?&vA6B(hZ1'
    '$+i{Uq25%PT$q<5n?<-W5iHUVAS8YI2YNG?1A&=_%rcs91$h9b|0Qi{%9<^+IL!8$Bee_;a7IT%l0oFt)s%BP^-Qtt7KRrmL;q60-)vHVg%f2p'
    'FqD^<l0q=PX8#l))_(YBJe3>THF+;j5uDau4oo6!0;w0k({A|J0cQ{v=u6rIx&!PjK<;L~wigB%)dZRh#_;5<h8!$7H!mUW99Zb+h4dlGN@9{o'
    '{#>MJQWQssc<VMW6#6RvXpoMA+@rD*Mv>N>opN|PI5s&Yp@hXoDao{TQ0sC}e`shXi%yqbR8q+7de3|`plLnp=?}7yEyq;f8)k1B7O&)@4wk1b'
    'pBMA5K|Z7z-aj&2%G608m#}!$w8iI|{AWeq^mU#H*?>XrS8B9YtkkKh#0#>k7HA!n%?YgCHT`REYs<@Tg5<P0FVn>{Y!3aR5ob^JO~`d^vr1h{'
    '@SAN)ZMX|Hxa_Lmbb$kg;Uu&2T`HatD%-j0U$IOp|0s)(mWgnSU^7qFs`E_w(BS(EiXFlK<6~wo!fV`IJsDm^FXH6w7%JfJHxcR+ux?)CzrNaD'
    'dqKC*o1O|TD?-dL?uo@imhwpkKA)Ii0&0&uEf^)A;o;GYLY8JQ_H^)Ok&<_Y6d5UQvgMrPh~)`;guu~Y!x(#v75g{j*Z?U{o8z#GL!p+OuSfF>'
    'UFgduJU_;3>b!S+t1rEhv2YSy426;V)~wuUaN@*(1JsMBpz<1kGNk7!NQ|$9Ls=lPOQ3HJ_>9e%ga-T7F__2zq>k1odbG^W&f;sl_eP=_UK5()'
    'twfX@F*ahc&N!Z5;3EL{7{;Csz8OVv0|YnTC>*JOV=6(1#CV7A-AvmmwDP{CjuryvynkOP(0cUj`LmZNM{nN0Z<$R!8%d7LJl^qXa16Q&3PLz}'
    '`|kbwM%Cxw<ZnmESjcRt8ZAC}<SujkGyM1dJ+XAQoN?Mus~;Roq$jm`9n+Z^VX@LN{skViYIsrWqJ=#Gk=0_eA;*In2`4Z!`LvVK5Qf%;ATEJb'
    '*bkP@Skk~S|4;6=3+XhJDvZeZ!+d-=`Ml}B%Qs?W=z+G@<B2|fA(r#=5Ibjl5EHKOf#*-LyyAWSRKnN2(m~OIss573S!e{t4Og>h?$=wYgLSiq'
    'EhTn(s)u$7DNuT<t5yjiD0-sDjuynM?rEL3N|_V2XVStdn{z}jCx9`W)9?eP<P(C&=dq-H-G)F?Cld3+2?}*yE>u!aLza`^6f#`%0Bgzx^v>rJ'
    'm~D08b1GMcwk;d>w&X7(79b!74zfY05GJA85MmY($4_<3PGG^?6L;qF)0QY|mK4%j78Q@~vJxth+oq-}(9nm!thG`Qs#{JNf}&bmc^xAHC7oY|'
    '^P!+vJiXu@G7$8=PH~L$XKVFkcL9hD)To}Zw?!Nu1KC0+`jGK$0VtZ1i&%b|n?tARlp6(w^VOlm{=%}$0llHF+Sn*W(5u6X#36>>;b_UgYO&EF'
    'd2|?q?wX0ivSl4N9Rfr!J@mkK4aiZF9KreK!dZ0M>>i$1I?N6Bl~qZ4<mJ#M{_x?$J#>seI6#*;dhqu<d;9QycYlPg^8Uk~5>3m!{k>f%FzP<U'
    '+>1M!utDW0Abp0!jBhlri#^nRRKYkYEofqS&<{J90P^8J{eGz9M}Gmi-R@31M|FSi;DLeaa<4ypfaxO-`WiYopmLrlX}q0xrwpOp=plmN?H-to'
    '>_+>0*vN2*hz>f6KO9rM;hL+uAaaVJGK}^P25lUn2uEl?8cO2q?R9}ryAQg6*IpC>84rg29krM&4;~)CyS>4KuEC}_Hcm|I-Vf%|WM6`2kW?;T'
    '+Z6!K`s<U&S#lFH>7+Rr9_%8?2K(A7xqP^{+a}K1Gw<0@QhG~A^2frOyDf!O36cEL3TP?2o2l1@0sY(3Z6A+2?LKHDlXnh)z4j0GBFXCdY<v*4'
    'hc-Q#)kdQ>Oqbog2-CPmJ7GRKmN0n`KzjsLcl)CO{O*i!hJ^@hcN9hv<Zg#fMT|gH8T<-Dr-WaBG^e*U6f~P*qT+lVBK!xPj@;7R-5KJn+1-D@'
    ';zg1H_YP?vvBQE!??`)RP{>oB59E&;W{!;GE0=$xCuPgG-ZdT{kn9~KY#+xe93BX1Sj^`x_%g?1n^zJys?~aiQ`M=$xg-ty>Xv>(st823J*f?C'
    'JcKS~(Z;!;ialwu4#YZxp~9&KB@{z@Z)jGEslN--tmX5$TVD@lQIJj$elw3`_=)bcp&h*6(Cs7rAb<LO{`C9&>G%25@AIeM=TE=SpMIY|{XYL@'
    'ejlmNqDgLhK8ns83R15-YU8}a-wmg&Zvv^`q5nfSOf~tDXRZ{2Z<t0LXoa_hVX{_-7`#hk{x-J!H?rX!Br?=k7ir8j=4(vqGOt=Wz;8~5Z)!&6'
    '8|NfD-bwxSaARGEFWq7*)>L-fU&LFn`Z)}np2yerx-_9)XKxGs|KV1*{otTs1D!@;(mQRpb{n4jbM9(-XcuzmLx#Sh<81Srlbf(M4_ly21Awz$'
    'kV4Twm(BFb32tkdz48$-_0FC#AnvvW@$;Wnxq791zZEt9wW&Z|UZlZ-)=Hz-nyL6M6~{O2nkqq1S<RQSfKNQ1vp=e>L)Dofs+W|DTOKgdr#IdO'
    '0n>YxMuh54&pn8~QA_+=cUE-}18chrJNj=GRzbKId0g5sqNrh!ThokzzL<wMfjhpp_Pk~ujAE;kGGY69;B><~&4J(vtev_QSxq-zzWM19o_Yb8'
    'd~`Ukqam7JCA=7j$UJ%oMs(MMI_syY*SU?r%^B*utXI$L<*Jlb(NyLgHD!yq?sfDP#sOR_W2}jKfV5n^N)}N(@XJAaJ!$Kc=~~xw8Kg5wC;Tmt'
    'R_9JQGZdcG6Tk2jo?cL%sHM}Ed_+<m|JAfiOuZ^VQC37Oht4Gll;2$)T9<?Bx80uIJBD`IgT`8lOeL0U+*%;{b@50=Tb|c!s)hwoZ%PY~^TK0i'
    'ka~+qh_RYC4G%92mZ0H9{xrtp52vSC3Kq@N@^a_r`QS`bI)sAvK?az!b8sn#>O(Y7q6rJ(&MxXA7cj``$O(G1_oM#kY`NF&?&BWU&H;bnis-tY'
    '$-_SyECaAQ^#(=h*vwA1t^+yM%Hj#wZo@k~XUKH&$AIgAl%u-0JLu!?$=*Oml~Zluo|w^)*M$)w-NN}=p!fpW^s<YmF5Zabg)c%_ey+QZeFiLu'
    '2F8q>Se#iqP$a_oVAal!M(C;(STwvB4X>PLM!$APyItBj!i}<>&Ys>otF`T(^`?cV7fA*>-~R=V(31%4s3}Gg9!MO{K`Q#jf%g%aeZbgLjK}Tj'
    'hl%jA(P${TJBlYj8lVv^J$NGCY#k5R&WF+b9gZeec=YPc<C8bRo0BKcPmbTY&npor`Xky2Jir~X(H`#6clW!v_cx+FvcZV9@CLM__~2~$u-ysq'
    'FJA4q-;D;i@i(9ixP9Q~{fE)W)dk7ad~D2I)6kba$WH;~7IgRAwzdVp(^jhmQt{kD8~N#WG;*wm7@*7JCK_*r8BQ4=y0ipwbWqoXb~XlDO(H&>'
    'EJXswlk<t&hIbLpST=O6)56Q$L3_`n`fiur&}iTuEd1Wz-B)|Fod*~o(C_2#0e$U{4&<ILZ-hSF+j}53V8s|d+&e(ovybKXx_mw7en;*W?{#rI'
    'm$yvCm{W@ncSjH8PTc<P?nC+J;oifCy8mRX`gtXO;n1t@uel=OsW+yeC?HPvj}0)QfVjU!9F~*SP*3q?c{)qxs3Z@K$i)QbuTdbE85(SvaDZ|P'
    'Tscsp)!xSFiZHq=!-(tX2wfQn@%J1bar#|5Mm0d*Ocue7d7f1}T_?$*&na2!I<KXXNhJ~y^f-;r%~|_v4*kjjUF)9$#XULw_1sY`S5ax<R>(E-'
    '-ZeCo?@N%{clB~?AC?s)lQgDBtB3|#p<ATYP5yPPB!)N`$<yB#ZK>f}fEb$OPGfsm0cn#pfLElT&;jJ$l>(<Ch;bzd)IOylkRM0X^?YALvtcw@'
    'FF%;*;6Am)?x6H~fZ-Yk4+b*M>tKlM#O}jE-}}2q0tTZr4%++re85CnPDr~CaShskfRd5nv4q+mc5%%bb#`yZ`;L~noqY$+e{<kmJ!Q(sArPkq'
    'IHX}W3imdiU<90kUpU;RDAhGOIp>_<)>B3L^EIb%8PW8fama?*Go=U*9^x7Q;m!kVX&Xj+w2*}&Nr6srS_(a<aH!kCMh4%DQ?t~EMhSp=5y`zh'
    'T<P|9`)zyd1-2@s0)<L-L$m)m>)-;4G}<5S;T*IGM^$oS!F)CrSx3iQ(h91Xv)&v-oL2XZln7Q2|Gu(^?x|eaH?ESi_=>){<gC#zLrGJ^n=4^)'
    'G7CX5i<1kN2G*a1$%lZ_a`xJ-9r?m7-y9iT%5^lJZMufFAMWGdy?u;YYDWiLp86yeERIV>^*cu!pGT-tcf*d1{1Hv>47#M;wXr@Z;--)H&{NR^'
    '7>PT2xI>9Fb_?440UqsM?uA_x5~KFW9s?;aX$Rg@5=-l>#DV7P{y`VJDcI<tjLb8LLkzBJX>DCuZ3!7t($?O9q(!v9bDOrb!9V1T6P`XwVq}=R'
    'wIF0_yZ!$<Zg9jphu&g|sddaj-`!!geK=B*{@~$2npkQegs%ON$%qxP05PNZ&k${r8xBT@;qKlrvbjNL1AWOXs`)}n=j>5n_&=5UKb89bl}bHp'
    '!1<^s*4wgJ1SbQD0^xZ!D{kphr(Udq?&NlZgYM<)hS|96HUpyMV1$t?AB$PBkvEm|8DANOsrm{WN9=`}Re9`;WSA1Bm4O3v3MgivR`A;3H^~jT'
    '@uAY7&?h4UiY8g@A-A!YgRiwxy)Rb^LDtp$gxwCV!x1Xh`_T~BoPCtv_HtTq!BCw1!;vg0bSH|F&*eC0v~gHviZH$|S0xuXq=I2J{ec99ZJZR>'
    'B3yrX*>&3{_E>FhrL3?!=&S&bRj$LDb@Xw%g%wdPfvU9gN|%G46-lbVn07&4^oz+BWYX*`Pm2)cnGND;{=ionnK=)vObQCZ$7Pf)a**Ntf)Z}K'
    '(<qeWU*Nn4ABR(wNCAF}*LVTVB<{;EuPM1WSGbvB;HvoWI36q-bZ4XZOv%sXSH40(UFJP+(9JKDqiurw!<6qtZbafuvGe>I6ub`uI0+A0Nf`@o'
    'eEb80o6K5{w;Uxl4KZncpnt#}Hcmdg9EPnp3)CHhDw~n^-2tGyL=a9UX8V1-urL{BSTspm!w4@rjuT354?W^2SZ7hXM;0&Pk&!6cb{7~XJ`8il'
    '_n=W=0FzP6qH-BVlDPO&?O<Dv!3E~ByNnll2|J%mK2p?dE5CC$Pa32K7RuXWtwm8#)t*31&w(wl?P+-DpH;tlS`W?I3jLndU9Fhj%bwZi5@1N4'
    'UXfsqq(?}qp7CTQx=|Dg#y2w}la;22qAkZUteOlW>EUHeZLP6kDiY51?AWBi=pw*Nq&T_@v+7wNamgzEhTD9X;RNH6+7z^Su9ZN!1Tq%P{c${6'
    '^mdHusLsd&S#Q^d4E;rt0mt{+t&)$l;dyrNVv_bT`)0!;o4oDUT9<K~mzhyseJ!`}@T5W>W1-0v-vm{+;ESj)vyfpl&FJz$C{x3op3%PCt&>Vc'
    'm(KR#GPbv~Yc61D9QKvr-hUX%ufo=(!>`gqK<bSKM=#H9?X=t7n!ta#zz!oXQGeB(<J?OcIb2~!`u~uBdfdlGb;%_rbixB5#UMG?|6s;!QhXkb'
    'CK09O5!c-JJQ`?q5BZAvG>60t@wT?Cd8ZglUu1lZiVbc}-B#W(SwwucxK3QZf$>au;e8NLI;S8&Q5^)e!`!HZm3huRqTelf<@;;;;t%u7-g`B1'
    '?T!+p-dp-stLwlhW1752UK1y^{8spyKAmy8WsA8kR`}dWuU2+(yk37CO=i8?YjLT1Ty~wjc`tHJP3pDqRwG?N+&qCI5#KA|mxs2y@1&P!Z4SM?'
    '`i96U##ooC>Zbc58S{JZftCnZyOvB=+6=pVX<i>aX0!4^rfX;27oz2t4w`u493kDG(B%tY@STh?%vBiFkqH!NlJ;FX-#j@Psi`fgqV|DPLGAc>'
    '6(wCv26{2U%Yeh>B833~6z2x6+It7O{-BO1Kkyz+K(E6)!!U&WdDv(C3I*@FZ{K0bcQ7IO>dSG6Hw;IVP@K<q_HU@g$E#Yzd(jcUW_!2UZMOvE'
    'S3)E*^s5kWi>C{~<UMRTE)72Sp^it2JJ&qE=Jh-MitmNzx)u)!*c=f7@oKYddo)>I!|Sxajcm02{Mqrz%eN;jz0<0j^5RbmJ$3c1z^0|R0n_AB'
    'NP4797xk*-wxlRd(wQnfk1#$Z$*n3ZDL=~NlZk9D3r7)0q-HInB<b9I;dI6i*=WJb%ztVKar8Xi`**^;2<u2Vl`bEkSvpyw$Rl^Y)Q|!^-;>BA'
    'ee!yitAv$H?d4#sJ-`|n7%#y7osta2<gBOg`aE~1LCXOQ(cpvL#wS<9dJL%81rQ%l-?Trd(4MO}8~fLXf}(Y=`J+5`WEkYv-j*=dU_MXNc<5gn'
    '=MIh*i{ugF=@*V@8>bIMc(O&hjm|D=l}S@mPiKiE@EX1PN$aP1x}0H{H8~?M@G0x<blHz6>-RKR4G$G@zJem8csw)aTD9?!8#qpr%kI!OAQ%ru'
    '1f|`j4|f6N@-BdY-ui0*9C}Mw`G~8W=3>alK{=1{evBS2?vDcvvybxANAvv$`Tm3X{!+fb<oDXvRztfq=%apNkZebZ9seUJ`uF<Yz#BtpK)LUA'
    'JRg47y}{YNItH9C;)`Xv%zO)_9K?s)fVeb3T;2u*5fjEQ&&%236NJGqT?8=3n9?-A83enE$*NrkmU2wS0i&YYXLJZN^y-d#NY@^2arA)5{a(Lb'
    'eq{5ynj4hVxo@&1pBBs!zpfin=uMO*(c<=Us3C;&HxNy-oZf(02DcrIudv)=hIS6S)xTJ+S|H7m4w2ZobD3x~fhRFK<?=&E&lDg#@|1QDZ^i=M'
    '9Mr?2gu)(lH|c@d;M!g28eK4fd_^B%kPEExmtJuoU8k-RJO_mbs|Hm9x*WNV3%PdHEY-OmH!iNt$DE~neRqmJkz!GJs~E%|8`8#r*4-3Z*MfFS'
    'B3~;cG(rnS$03=<y6AbV^uBMZ0e$BsDQL1c(<#`I@j#9uUo(JeFOs9dSy+ZC7$Lw8U^GrHqs+nMw^-$K2Jf!PLrT&g>YH)65kWV7HN19yCPI#a'
    'xpqNmpEhrYd>tiTO(jx{;KD<h7J6+9gz6*>U@=*FK6F3NeuII#8=6}}Wh0sw>8f92cWs_q>ygDNh@F5A09K=M2gqh57Dm+y3R_2$b&!VrWh76r'
    'R&dQXJ}$no4K2$BZ&_Q~*H6x7HXD2tS^ADJiyCfLbdPq4@V*$vwC*k}a1;Sc`3tLb4wf`Z`LMap@?CD#VjN$Qr=iQ`Y%My!5lhQ$V(eTA^3(=V'
    'iS8z9V6Zp^tCbPYsE|KcrLd>m7iLu4ht-FYD9!y99W%oR;$RY8@<q3-HoHc-cmN21R<$bbtx8sv(Xbofk<z@IrvkBaR|Piss77^h>|x;Uo8ZB^'
    'BYz%!k>M!1j;`@ZPoi!rFg=A}y<P&CyQ|8XrR?Ff35MTWlKk|q<GY|d{!_@TD92l>7xsK^dL0Kv2yE^u0^pkdAbaKLur?>uPdp0gP!<_;raAt$'
    'l5$Mr;c!w+`ZO#fmi*4d3Q)nn)kqrMQb6gRIfR+}k*bRGx9(Trpc4aV<ghYwKgfE9M&9PyMn=KalEc7zfj`~7Lf|yF%LF{fhU#J-R{HZh5O`dX'
    '=iR2#L}~trvyxNH6Q=^`xj)WMo1JsV6pHD}8{Yu|nGiF}v=7}Iqa_VvHLYDolX<wT{q@!YBnRohC>Jb}3@au-_KZt93@mNZyDhMM0+)XmGu9|z'
    '%{#*cBxlctR$!ry&_qo;E&3`=w5U{S?)<w^H7iuway|HewSG$FMr^o1nJ%WM_RA)dH5CTwbhcbXnx@pq)e6j$sn*#LH<PY#`*E%t)3(Yb6{xcI'
    '&sS);w4Sf(fzk6%0r|064K~&c*+Blw>is9mb0>PRhGMS+&zGtV75hJx4}%!XS&3z}oK>m&_Sy?S<{95+TDAy(W3%H9tq8in^^avqn4@7+U^%O@'
    'BH-(qHiS#}fZBpbaE>c))rMxmmeWcA*l~haf1@vMIjbW>;?40uUla{^2N}BBV9YV{hw@DP@l|IY_ODc#Kdn-kOJt|1l7PniX$|G;+bCMAq|6<N'
    'kaZRLvI(tlU(U)^`4w8cUUh99-mX;M-0BJH1S+R&VI+PJr8f>Q(Bu14xXiLROxE&r%P1C1zb#_yL5~<OlMlf#p7OcYo;j*3bxbA2_(wql6~Jj3'
    '5x;LeLpg!oB)zx{=jxVs0_%TjVD=~g-ZR^E?9&M<SfX<=R1l-<ncr^hf_&ws>h>uc&JhjB;AS>dlz}vO*dpP~LHM=$be9Ik2)oGxmev~68Aml0'
    '1D!S+MG>wCB#><vP8JD)(c=a+Ro@Ca*4J%OR(Bp-M~=p_3`lqT6Db_|cUI-CqzQ&dl4Z@eR+Uc$^gg+;vne!;B!#5C`MXny7e)D5ERla`BF-aB'
    'FjkOV#TYDo<BFqYu?_vLT|o<p(<*3hb*aPbzMd!TQ=VM&!q+DYhpH5c8K{HJ_X*EZ+b9;L`zIYgX14W&A~yU9v(gtDzh7*((5%kenqG6;@E#4E'
    'POChs2LaFKu+Rh)xd{y<nEzrLXL!?taISz_cMiI10)jXiETW<Inp(#gn*91|)EY#6(i6zW$Co((TYn%7{L$IHpVEc*SqJdN6Td*!yvYvF0OzyC'
    '**#1Ja!)?Aeof=NM%@O>1-VKS{@wESAFBHI>Qz{17=O|FzY-U%o=TwSpv<?WfNF9Ry5CKY)y?O~Q(@$T$l|l*<3qF&h-tJ1QeYI~X$6!M=qRw4'
    '&vNnJ-QBAY=sS(#c9S@oc2E@Uo$i5KsPB{r^3x6}IuGFg&#T3Fw_Id*><$=l0`1n$UZ-Q1(1Zw-=yY};+UL=}^PU6w`8uUYKO7*bZIann<gaTf'
    'Uh|Q|U^z@DuT6GBWGIhaW}}5R&~-hxIq{_~LC{W*T3ih?Ez)@=KO3?MIatU;QJ{gv7m015mJ2Ms4KurLh|pvY`M|R}AVJ3|e)A(8@=yo0E`#YG'
    ';RBf4qZs-^53(2?_6U;jGG55S|BbI?VS0GM=TYhRlXQLzD|$G2{zAgwlOiUNxAg88x|d19(%%tC#y1Ek12*nhF>Pxay7E*Gzf(^dj^;apTa!oq'
    'YBtKMa1lS}Cjss(Id={&{WTA_4`6tmrzDKi4eI<u$(i%w33c%_4HvUXx|qa$Xcm9bo%a|Gp*Ptw;)Srt6X%h2^a(b+cNl%q@Lq|Fni_b*DOCu{'
    'gTK*S9DIiQfpN6`^&D?5z_U?eREre{MB*hMKP%K0C#!fIbaG)oRM2p1G>NYB;AdHXFpqGmv;SloW*@YJEG_(FHYX&$sFm^2EUYGZ2D2-VCe#e('
    'c}TOEaE^yr5M2YN9Cx)wX)Q{tq2<MhM>klxUNHk-hVf-|jI&x6aufc=^VdMwgbtv=Fp6`?C)0j39O4P_*8&}WLxQaEDxed0x^1e>KHnak%{JXG'
    ';|x;+;2CoALKMC%lcD`(GMNU<Ej?Tl8s_o!k}n{F-CvOSAk;Zo1!uaoZnO|3*w@&#n!^IP;BtMit^>R|u(xj<<(4B5z_N)4xueN4nB5Tn28%Em'
    't25o^rC||rOW`29b=urKtiHAN;R+88uCYldixv)>s38G)n?JZq!SLjzmIa4#K2*3^tA_%tMP4d@uuG{S#8T?PZqXi2Y}kJ1xX?@+WO1vR_Qrv)'
    'd)x*7?vF)Jcg_{)7So?gFH<9)Fs@UhH};Jd29Kaa%Xm?k%f2O`CF3xO11{Oq=e?GZOdP1iW5gIbTB9Ye=DdxMO~^@@TrBA#43T{UNyb09!9%^J'
    'N+#y=b@NVP!3cUI#0B~btMoVqd8n?%W2hSU>fi0HJfZ+4k-YaJ1sf$!9Q7yR4RFDfoOq6%^>GwZ8j)f}{Hhi-E@yZ$MX`LIx=t6kCWS(*bZ(S$'
    'lc;MmRav`6G@L|OO<+A(Zfg^?iqlq9E}gwLV!{w|V)e-pjgX(EBfMny$g5pK=&daZ_j0saLg*k8i!(art@YZsdA05U0)6NL1Mvph^B5z%eJ9cl'
    'Zyj&|s(VQ`n*~m**Kkv%UcriK#vKK?1%<<R4R0m3g^2)5k!vC-Nh1ZdDGH2ouGx|=POVDTtXv^9SgRHa1E5nzX&5<WRB^e%3!OBr(09Qv5?hL9'
    'bJ?nw(QftdmUn;6k+xPq*Kn7s-hkJ7dim-Lvu@jnW-1gl1J-h+hNG>dwC-JTf;%FlmEBMSP`)U@Ohz~@dDMm%W;xYcSK`{bc73C2{W#3HQqgx@'
    'Y5-)IoDEaYOrqoh-E?GGp3;d;{TfT%$u+|~U_;L41e81!1@VqDkv*WKY%+|lqw4a!5Fs{D?ztnvPXN@1vJzl|5`3tw;b=)p*hUXwz%AQ^aunLQ'
    '6pK;d0E#kM3?1afV&Bo0J~IU;dB9N|8gT<S5NVd>q>?7TK=o0YPD;@gq?v?$&<BYwR_&ZX<+Um47~QB^sJ?d9QuNnFz#QF8faEv;XHQ(cc$O?m'
    'Ya2rZ^HyZzF;3SFiNwfFn-GZH3NdyMR2;diVr-#~ATba5dUygNyPa1Jv|P(?Ni0cm%pUjE<EE0VZB~1l-InEPH$f--X15a;Bp?5a8JXm#mYsFe'
    'l6#%zV~W*Sh@2^0fbzz0LO>6SRRNJ*UIP#pMeC*nJ413F(M}!q<s|}!bs_Iv6b{CsG=dWdl|U^MpLzpbz@BrXOZ>g7aWs#-cB@Ts6<MCMf;I(<'
    'aX!sB8Hu9tMUu`-NN=*bpb`>Ke#r1HFmab0W)z~NZg|q4llQ4|9?ZRYG;~BO-gqIV0HUeosjH=(*yy5C;F3>tfdqdSVr9W8samDb%>LP2L0PV3'
    '2joC9d#nlHn&l;Ihk(}QI)}V<p>1*BrnJ$Pv=W@DMCqg}EUO5GqOT+vW>~yfTtux8@LJ`}HTK2|z>kt&$QAs{(TOM1ZaLESmM#PNl=3m%N1e}H'
    'W+K})@swzWYnaq5Bb;N2*pROnMhR0DmyY595rwoNUxTfRYf7geWZ>yaIo7h4*Y+rzIBv(`HSykH5=mKLK`*X3tN8)yNQNj|WOUIUNoy;2P3Z0~'
    '-Y#fSDIr6AYkAR>|3VXzY7}APXcUuz2Ye|cxy^lRv2&Ps_0mR89ZS}ES<6<g2>0?K^RHMdvbCP&8$WbWsl6NIYYSdOQ`H-qKHexH&oH`-2Ss-a'
    '3NUe7mImAzk#C}vzjIv2Z0uk?!XW9VJ=62p@cuh^e)6)%7&N@U{$e~j4!IKky{gVZQ_o)=|225~>>no--rxnu%JGEbzDsXp76y$rTq<y%x9%$J'
    '@j^ZK@Vgj$l@ZMjB|A^LTy#{(XfeBm%!56}Q7x=i9ID!`H|T&*yO@bxYaxGVEm8p&1f9A|)wq6J@{&yQeO+w3-ijB|)UU5tt)(Bwbf7u-OzKE@'
    '<#s@sJF;FuQ*m*VDCG8&^h(r|hU;vE>{>7XS#n1>f9>Tz8<iYcO_ki4<}Oz+(^-U{OsB4Lo%tmlUl+5I5BH1BI|16#x1heAm$u=WRo_+I?<B7D'
    '$*)>SHg{>wN?q_3dWm+Ah<@sUKQ^Qq1X{wUc;*|5SEEvwavg_IE%J}k!-jW==c#7!?-l-gj)B5h-FIvjP-Z(QdIf*$6~IN{dYquNIlDI)&#pc;'
    'e;i-UJ`OOS4wtK|mhj5i88+_TqNtb7RiTrK<5~i2l?KSlkc!n07ryab%0bKcOY!3FUNNfP0lQwUz$qv)$)#Ojb$!=f*jBONHi&Y(-O^QDTy(i!'
    '+iFk&Ub9dK83=Wla}A(D>-*(Zpazk)vdcV|jgG6msa4byh&Y9uy#wTzn+hxFqHlMZQ0m3r%kJEFBDbChw2ci-1Ts)va*=;Op!DBJ#Lr)fLQU))'
    'A39sd6!J!EmdY#M7EUyk$?K9h%2loQ@)_$=#2cmUoxA6>PL|ekHpC278(>rRp-~D0bL%@{6o#{?0=QPPNtoq(Wf(*1wy^w4mg-s0jZ$A)Pd78F'
    ')ZL!>*jVYn>N(z7wYIkO;W7_&7%o)d8NJ22%{ZV%JezCcMr<ni1!|w##<QIXWD(BsY->-f9!;lk{LXv%>fOnqcNr{t!|PzNJzVszgHH20Xah&5'
    'e66LYq6E<pcSy`}m<8PV!Yc5ienC}EB)`z#ra<^q07H3B2YRhuNytHTJ%fc%SMhQIFJUN2Z_T}7yeKZo4*DoSh+eVft-`PsV%OE38frv|lLq1~'
    ')r!eZ%6tb}4N2P-ik6X8%*ln8>#afxiarf&PBa}S6SkQhj5(<0OG57n-YN)XpbPVKcH>)ZHVO{Mh6nGLL?}$xhrPS!%34Wr>_D#`(F_C(THwYw'
    '^o%D3LpuyrIzaM-p*iO3j*&)YiWH@gQpp;+{*6%0JK-$L-28Agrp)!;4|Lc>;9{%ZzyadRX?*Tj_F(Adlr^sS$qk=25Gp}3s^-OLY6eogq@yth'
    'X3+&+ULibYPz&oe8R2#6)A{Y`ML@SB=BA8wJ>muyj3g&sCyAH0m<S%0UpYgUtcWRPGtsFW^g}iJ_btfh<&Z6YlJ>~w>g#W6f-JQ|MW7WGh$v8;'
    '$+)VtHg!Z3)Kz75jmeH{cIxGDzEph631yE#d7b_|3a!i)4*!{pa|@EjqPZUJf$1f}IgTt|ktF2?<*E2^$?h&rT9M9BXmBaxPB!CuFf*;J;haW0'
    '+R9d<m$0I6?4neLn-RXXDqy+#T~g{pbi@8GM^Q5+kSV~RvrH81HPS@YYZRKo*(`3J(g-SkYu#1e3Q1fy_3Q4nuh0`*wQ)vq_6MxaKwy&QfL-@T'
    'A{XT)3ffFQjB4=Z5mCmxURvSQ;E~f>A1pZeSUW$*T%oe?{X*f$EbM3`XA9-s0O_wbMe3DX@{(MRJ(MaIEmB$k>=@+xo{3a_<sN^<KvsehJSu6C'
    '4{4!dp&l`sQL(BEZWc1i=v<Ts$FOmN7PhsUbI%&jvwJd7i#OSP>pR%1bPJU7tlNVA?4C8;1&G#n6XGOIE~-K5JW&bTFxfV2MEVUXahJyZ+Mo-P'
    'qYQqb;5v<{N|}2k4Ix)CieDU>v&~73qFk1>{!?u7q`}2J92TM+g8?4M3fQSZge;GH#9MiDT4aaXiLTX=jw*z#v3mKd3{JFWz@0`?mW_91C2OF|'
    'I$Wk1+{v!@9k}r9if42z6xsId9-flAXD}igW+f;~-cr}zxgAR%m(l#D;9fF`VofWqLOv#25>_<f@*ovuQUw6Z3unk1yP7dade(DeJr$-`DtOHL'
    '7VGt_=Z2m4jV8H`qIFb}I3)wQfM<(cAFCW}C1l`xV4!%*`rwdlB_(uS9sRbf$*)k?xlkiNwVSP0*NaZ*q7v=c6L;(ymu!X>H0DONDnGPBGHkR;'
    'tJ}DlE3kN32CeaRmO~&+hp)FF#4We$7!Y!}|B&{Bt#%6~(^B@yRT88l=I`418d|adL=<-yw@Nw+O7t!FPoTZGBm_C!Sd~GGNKG(bzlK?#FV+-T'
    'aSp|?zoIuuH#qy9dc*OyTW>gBQ-w29u42aV9uTAIbo7b?K$VdgxlkZ)5+9fMb<7(s{8mIwsPW9-)QeeF(3PXw*6Na5=d^vkR$Q&#QpILxK`bgJ'
    'HxQ|>7i(+I!+<BE;7F@$^=iH^pU~A{B=?)8N~VZi=r*%I!YjD{@N?g>oJbO%+M*fv?XBSlvUf`ctdec0-ujmQI+a(M`<;#L_it=%-?1TFn6L%F'
    'uIRrtJr23B$)RdNG`glF0tzbtx!nvFILpPb5~v|=%4k|BxWSmM;~crAU!-inY~63&(ON-;KfNno?_B{-`bN%`yL5i&Cv^0aVr!jUTo3%(z8=R('
    'nU&t8wSk*6Wev#x)*htWpO)}3Zqp=Y{kJ(R@oj_IHx6XKeGvQY0@!aAynYvf>+1xquL@XSyZ0Les()o@`sQKjg-A?SEF%?e(Hev)+wHTX^x^QU'
    '0_2e;onTkbxs5{P|BbxNB|(guf7w!`HgqrF(zE=(!>Meihx>zgmBZ+cyvhv~fCwwg;d&_z8v60jN_X+<Tm-9vL;lS?%4*<*GtJfiKfTWXnqKFx'
    '@HyLDAraAEI_lsE$BW;_PhH}sF65xw+*4hdPp)2iF5T($wTNQXe%kQ<2jfGH{*iOJb7*;&F(ULS-YDnprCIn23jfc#g{vg1><0%z*6kTmaFXN6'
    'w()u=_y2TJ|LLOsL%FDV7UGPjutT&nc{1{%29LOE6-48?X~gDG5D40^ShH@xphXjxzb&oJx_%w6IDD$Y=Gh|FGb~`FTmcu5m0Knfb$e|#rKuSq'
    'UW9k(Oh&?JodPU8)#{R7=kE7i$}XjU*nE#b!vIYm<AuYYS}xnTc7;}1=KZ#0z1yX36<Xy|SEp_Y-q=(5>AYTr(R%*)!V$Pl_fY=u+cJOd9{jm`'
    'P~1HTql%pLq0CIZ`Hq4ndn!EsRauzm!Ud;Ux~Vf>OJ1~M@1dAPY2B@mZ?kRk`))g|xr4CI4#l5a4pmzY$an0@#Y{cd3t!E=7G0MUEuwkm*X!(m'
    'wdzrlp_LV>RJj^uJfr7VBsR)hkRnUrS}#e)DJ~@tgC&l#=-|JmqE>Z{JSTRZ1eZ_d&U2i6v-~E%O>UF*@|xI`T|1jeF4)WsnX+UUw!E*GO848Q'
    '>A0h08ikHwqMzNJ;lbH*uiI$@Ff;ED=la=?Mwsy@3in*wJz@}Uv&*BpV}kmMJ)Pes=SN{Hr+C0u+uXtHSnf4kWvUE-HPc>HYUC>ukbE(3M211$'
    '?;aov3?H5?A9e-<cr)1P(3?K+NZ9RQnFIX26CQ{JP|F9=!QP)+S6{Vtwb|Cw*KTzEdu?6u%wA{fihVWfA6&lWW>L|7P;S919*P$MB`w)t`^edb'
    'y;Jp#Hl=tbl<rKS*5adsyuQI!k{q7@mHSCKLGRdCtc{e*VSFb$KRD9=vU@t#R!xI8U@CTN8td=Qu%DY8rgD>oGIErRu#y21+ajT~?0UWpz2Uqs'
    'X{+kCqx!nEtl0g@8LVM2TcP(~xf4_l$lT`1%T-N;V%gtvQ|R}a?YG<>(l~g-4I-=NzxSzv;zV|CFKFLa2J3%|?TX)I$KkFW?cZWg;I6#k|C&1j'
    '>-g}ib?#s=x>#TDb|~6aVN{LP3TaW7{Zm2xRx8KvsTci$^@BhBLcQ5~3llo{h5{GC->uNq5H_4McOZT>(0>FGT()N_%8THSs&iJ~d@E2>Y}VPv'
    '+;r#gR$HIn?)=nxNUXCJeTSQv?rRM}HfS5nF*)744@&PTKNOeefLt;zEXNr!kCJpg4W@CDmui=lBHzE4E#`FNGGD7@IK#0lUEm3w(Jpi2YwZYp'
    'gIl54qtkZ7>zo@`@F4(XQxhaHZZ-%fQB_ws6OKc8Y<4M~4eIt+>}ilJwoy=W{;8^8V5kkQHMG0qV6{#9#6uKMFRVeIUaTG<%Eoyo<D)_Py5V6!'
    'w;mm={<VXn_2nzahKAahqPQbJnMC;{)s@=oK-;X-J=U~!L>onWd)*Ok2({bzw;ST!A$$1uVAL*3n>;vj5+@k|j_536!xUHzd4mzOU%ZUTfMwtJ'
    'P`v2DeaC}q&vi(fd!BY7L$hO~*fztwzXMZemd+R6Y}o%E|Fp7b5nz($$X|gjgVt365$w4@>tpe8IQ8=|&m2|E#kvS5JxaT%-qtJc$&%$1;Z>X>'
    'Dv?NFnI5H@>SB|0;r(ccaHvNqmlxg-J@Z4ym3-M0)q&Tu-neTh(IG3S7i;CrEj9{*XN{H$r7i=t%NfXhO}u8#;k5-CA{`wra&=xx{;0Tfsi6P#'
    'HrORD(YtzOKBQ6zQi1H*zPA-M*3Kk%`x_97%31XuZBb3E-a+1cr*`w5;Jeu0E$V7L^Rc=-87$VbqlQcQa!<0JHRl=?6<i}<^>GbY=B|C^*D7fC'
    '$d`Sn#>nJ_vJY$Ys>mHHOOy_xp`JM<V9%6?jq>7U16cBHHKgM3m3^#$<r+N4`$}wzL+5zEx<BrLt13{&4RW43CTL+_la)98+r*U%qpY-*9TtDk'
    'cTVGTV2I0Tk`Cg<P4(3(IrL>>$8(V$fp={Bmb)L??$)7ccGuPmz7Gr3D%VHZvTJ!h0G(2Igj{yNng8Kd!QkK2NAr6|c5meA*dQ;^X8w@7_ha0h'
    'BjQfH2Y2SuSY@kzdsoIE$_sH9zJt4RXMEMr+<_b8cX2`dc@zWw7iL*BU#O+4wM=4w=+~9)ju-xPe)o8GS*UJ%kc{Ycc(qZ?x)G#J8q6k<=SmB@'
    '>nL*s{AX}^X>j?ULE-<ZLE)~r@2`&v*L-L4@<<gH?ilPsP54_r9jz;=O+C!}5tn#t(y1a}#izIB)lK-dnrAn3l-!Ab<SSenU*S=?Qzys<tiPK5'
    'E2n@X$W|fjm=hLXuACeWc>3c;Q%32=r85Rg{2NS|^&L}p;gGt$8|n5gth;nP{qA!G@hzDuIYX{JQ*vn9(_JntqLk)HK<!-IcP3)=L7ytp1@K_l'
    ')+EL-L9^K{&%Y$Uo?gsxCnaIPos|x?`4p>iz%ZV{{SZYn{EbJ!Dj+DtOgQ{{ZD@>r&+`>x+xc(N#C@+5H4p0b)?gA&XTD=A1J921fbR8j60*V|'
    'AZ~djdy8wyJyQ+QIq1Gsn@5G)Z|Wr-h?Z<s8v2f7r=_IY-E+>@6DwGsTESP6E6|7Ac7wAs2=5aoX2u()2k8_f2AIX^Wi<D~$prZq1Swt#+wd0S'
    'WkxZaU@A{;yflG7*>Z+k{h8O#Pt$<K?Onx-v6qCG@uHMq1vwy{i6#9howN1o@{Y;rBxD&{Rg&p$WXTF_*M?KU4D%2#m4BHoo?!?w-n|<QPv-M<'
    '?vKvyeYX1jMcy_}qdS{JS-QP@GpY5}0!uJ&QA|1d35=Y2$4`DL7@4lqv;fr=&Ce$mg@3F%mVec;Ssl~{(fb8BMLliJJ9;r}W}RUE-yweAL86i)'
    'W#$#R^)=lUWr^}OI!3U;i7dS%rDuE$epvgwOR;lh4HQfMZl`|!B_hP{@RrkWa?eA_L|JF=vB(OQwdh1KUBgb{(At+_8t{pgL$9Bv6PN<F;j)OP'
    'Go<HoUb=0B$3o>Cmhslw#)mFxL^NKzFbmDyh|_aZ@afm86(^)BdNqz=v7$D9&^v8%aj`{1oK#(2cH5npa=^J^uap9y957D&AzF=(VSeXHfA|s%'
    '+fo+iB=Ht<Di*RxFRsh&23>A#)puUEC!!{;qAi)A%29!4f1$jYMavn|0eT8`TCR?z3l_&>*iFbV3`quypcI3OD<%^K`!5?T1exmCU=RtcF!6ee'
    'sgajSg2+=cRqu^_DVW+78!OJumC8#PzIx+grGKmP%3sUNmH*tx!&SaiuIb;prC)1vf7Vy{vA*7a<;Z<wy01zKY~sCIKNZeO2GZ*~K3fpx7*Um8'
    '#EYziktfq&9EC%_%xhMdXZb4W7^QjI$A-)siL!LGz=X;=GTZuEDh)n|S!6DF;)0>&WDzInG!7>|waBLi7^|$;skbsfKcXb`>&1mT8VTnY7S0-$'
    '|0=>*m#}U-`BW?3unq=jpHA@r#C<%0C09A^ql_dx7d<GpRKJrIp+bT{P5aG#``B%jiCO-w0ERRw=k>rNL-~_F1yug%WQ4|LB2dHB6~8k)OzdFl'
    'b&AC7&!g}|afdxmwOQ2ZpuIwlO<_?m!9j%_eDF_WjEykB)pPzUf<3oa>+h&^gu_5aipw~WRzI32gK=14l@Mv-Pmyt+(YSNQP6Ds4$FRTXML}dt'
    'XN%v)BG4zOnF^I;u{#T@vCj}^aRQ@*8isY>&M+P<Yyp7e|IqL*alDM;bp0B%a_DqV3L`fQ<PE6xGT)!!u=s!y<}%O6TGw4x4zQ(OqylK7l8(OK'
    'h927o@_7_bCRIu2a$DAym#dB93*0>l#7|s3MBvsWyn)duUZO)sP|r<16j$oddmX26I22sxxL*fHMkJngT4Gf>wB-`6(s9HKd3#gIqQ<_Ao`h6V'
    '3SswTBQ8UQ8|j_Wzvr$J`NcH7@~P+?>R#EDb}Sq^d)xH7BM!I0Rxvr5+35HeD#LWv3YUv?04#`!Z<oUm6W&Hi*q=m0RwE6MPoD%vXU~O8k&kKU'
    'OD|cqm2OZ^?XJ{|Y-(;S-GT*KnA-fen)g=ooq*gs<sau30JS*u@@2xSkSL{WW+hzSkiFJrtxeeKTd>q7>_oo8&G~5<VNUEym$Jb+#`Z&j(d<2k'
    '{}rQDJWnGMGP=FEw>Q9Zm8X~gbd*yHP*44Gon+0Gpq_5lIlM5M#@n+K>KVHqPJ_E3ph{N@<-AeSp~d{^TQojBY_!i^6@`(_He2n7f5y`Qu>a?e'
    'wK{6k`fFlj`d}XA78*7KI!k~f$mcYPlZ)*>tdWy#`4}u0@g!@_Ztfj=_rCLv)7i~Dz8Eh&e^B?j?QR$J&Orn!DDeHWWY8je0?p`f1lmsw9m!f}'
    'iKF7K_vi*Rc2pm|=Zhg#+@>3i3U%MUe*CxQbKvp+&)&DNw~ZT%{uLhYcRx#6TDFup58o<1**tdhHLp$FZqK!SDN3Sjt|d_sCEMyY_qX4903cqZ'
    'DBH<yTW8xuB!R&I7!2lx!8A{HUV|*Oz>M<qgW&n|;D`U&dG|bcfXAS}!?K5GpxlhmzI+zvK|e{R0d$2j&<Fh$wfSOQo1pCwhC6%vJDt6jfczfN'
    'Pp9#ip<e>@c`{Do8~`stNt_$lgklk%CBbMp9)tEx*o$2acDRc;9pf4dKrBC7EN1z^?k=eEXUjfFfs@^l7}R-^Co!zc-52TMZhxHhcPDW=-F@@='
    '`O9~oUbYrji<aJT)m6eVbx=fnV67w)pB55@ND(lJ7iZcdNS?D*0KJ^f)H`Ak$5T3aN55ryf$3Dfm9kZWH~bA2=>@HcTo!!x{IK)t9X9h_uyf5n'
    'Shw>c0PQ2$8D#iRAob!vJDTua@G%h<P!wpt3t$Kjt})iM5^x3F3}^OvEus!6sp%B(){Z!^69N?0FgTs3LyFx9gTYvE9TBA_48^=H6<3*34ZO>5'
    'I+MNv4G8mg_(XhNU64&BF0WCEt4xyo44=5tr9k_viJR>ijCpVwoMs6w#Cg83E3-zA&tVW>rMaf1i$eI1x`pAW69#Q#VAM~^9rZ&bC1V^Rc}9e@'
    'HQE9O#+=9jCnlzkcJMUN!-(|uphKXm4Ul7FCX6F^=A$@AI)E@xt)4W)0>Ftj1R7)34wm456|_^)FM+T)n4`W1|10RAiD$1cjVyCym*Y{8jkGW$'
    'JXg!NiHPat;sA)q4H0VBLkKhQbr>typ@>QvKi&H!3|{O?+Cr$8*N576p(?nZNuoZ4+^C#_cC!Fr&Ia8nuFpfe?mWO}ISt2K^q)kEOB~2eU5z@M'
    'sX*pcJWY}KK@gJ_7iUR+q)F;0VxEQv0fl`v-iD$F0R8|URhM)$Dky{MPLk8uU=jAvn|$KogQi|_x2r*h;w=+uy9F=Xd+_JpJ${G74^tPaxr0uk'
    '0W{0C8(Dao_5Lx2;!>CGNjThjIvH-$*oWMjtV4tmVjb8DL3bN22Mdi(a83n>N~dURY--htQUk(@C@ObLt)sdeW32=jjq)Un62iOLW_c*iue}t9'
    ';EcWT53n^3#z2V<mW}DO^%mx6ocNZ|OFoK+;b`$(OJI~vVL@CZ0}xMO6laL|bip!6%Y{Pc^*L))B;w4pW0GMonzKE5Rro{cOSiLcXh``IG>b;F'
    'rC>Ep({XDup0V@k69HVHgUB)oOj~b2l!@n!vPLc3`f1D_LOc8iTS?#iwc&X$S}@?O2Woew3_<8&*(VhZl;dO!CduL~(}ID_6P8%3!7-Z@mxzh?'
    '(`?u!ZlMpmK9%_A`8<JzhLYhj&(a~t>MFIA0xZb@{~VB&J6IrRZSTAYI3fr6>pLK%A;`$cLQuX0hCyDBAmNbBmfu9je-6)p>HzT-a^}%Imfl9E'
    'VcG#uCNJ1Ow2d!>is#fQM&lp=-m;iy<D4o?fbb^ENihKulE353V4&aApz#j*j@Ea%x4`guE8C+^l0!F;+k$2e0bB~tS#mYANPY)Zj@CFkZ4}+2'
    '#AkZk3#qW(J+xU2yV#UzuaFWdY$o_oBU<e;HZ50}c3^paO32|oW9SJxVOR~sfG|Izx0>TX3n(g_f#N(lZUzXkDP5JghtP`?j}JK|IScn&`=G7A'
    'jUpODzqgm8kJLB1FbxDDUxUSlVxHXA?za^D&v4u<A{+_*B(=LrD_MfsXHd!G#bVx&ERiz0ZC@s67udEfPB8B%Ysu^Vv^p)W;R{DXa~k1XoWR!M'
    'cnoW*i=rM-jb1~~k{9u|E-`6v4R@5OAaC5fwY`1w0T%0&b|rUP^K3%tl%+%T`5`aRo&>uFrb5{XHrj;>g*+OesKeU$Q33MA=JbO$Z(CyHV7wGc'
    '&c-n47{x^~y?hzj$z5oH78o-zPBYkfYP`_8Y*bUG<3YP&d}ywzV2sOVRiVqJ6$CjLpLMeMAQ<gSwVdkEhH)#6x|7D($2SvAyzroZ{ts4dtyRpI'
    'UOZXl43&A>`XJrQ4cm88+)7@87~T+L_SW{UATLTm@*}4{8&ArG86FnqD0#!6Q1cU()(7^x;5B>(L5?&ov={(4NB|wpJkVdEFoIU9g7ucz5z1^s'
    '_9%x{v(_vd^%f?|f=z0+iW*|$M17Tvd<|8qDYw9ig=*yBIB<yFZ1zYmEtpFw^>f7H-5U4<tbyWq-Q7C4{dM4Rw|u@#rk#jR>6QCh9-w9s{0v1w'
    'S_fqS#~R=Kv^+fpQ0RzZAIt*2em$?tvl+T%cRGIrpG$$uO+7AWJcwxV*wwul9<R%dJubpog0x7yFj4*y^-2_>Sk(-bJekNDv>{k)(%rI}p{^jl'
    '(Z%m7(N!2w!g|f6Q^S43;c(LyU9m?V6;b5Viw{4bb8<R_g*TkV(Rnh*qnKs&U<D;tkpsqjYu7F!b7grtZa9pF*&-4KJGZv(aw2b~to#<&*n8pQ'
    'RHHtE;XOkr8dCC2bP><Tz+ey&RY5M*7A?}2@{H1>gduo1F3!#~iJ@f5^cIu=<N3opIh+GoB}2l~`1bMbr*{~jWW38jFIl~`;_~C!Y`lu*^i6V0'
    '_|z=Jp5w}v!c}c_fm(S^-_h?EY0o@-JWtVZHH{~k3|8a#4CKh5j!sfE$L_Z~C57X`U^!WiDOA%_@xk7chq$sV+EJ=-HkrXHYnY$W{t6l|wv!G-'
    'L<4rZyA6B84hBCgmeXV#hGQeBx}dI&L_;4I$@k~FNrc!!%2pX;v;zjV$Utw)=8303HVWe4sK?}j^VU0Q1+NzfZHX)es&dHX0-0tIYt+H>9KZl)'
    'bFv>mS>ipjWWWpxl-eblt|)c=2)1-Agq9<&2t$!BKto;56KGSlf~8^;O>R|jTL2a>$H2`mQ((KRG#Nu%u-#8mw9J|vB&1^WU}&iCJo;g$pDdu~'
    '__%AlJ84K(t?_4=vR$`mvn4m{=jNdHe}!(LW%w+a1DVmPIO@UjTk!JZ;yq3hKZvJjwUfgwY;4*<v1JZ6tdB`N#P>o#$ObgJ+-7+j=-!#&?1Qii'
    'E1Zfe#GlPi_1MdcY!LTZtZei^`R|3ny-aZTdtou~fM<KZ{p7wAwDUIv0LyK@V!x0u32Fq>3SJ;BBuuU#2KAFUdjUt}A6Nv!0ZP**21nV|d4E@9'
    'Api9E+zHA$Jv#Ue<u1=aDgb7VtjHp9SHrH;W8@Wfe_z;UR0de$2q1eAUIj2ohPa;UwcF|JwbwTc%)X*kC#rXCtM@y0lke1AiB1YgcsT<ga9He$'
    '!Z|p%5Kyd+PTAZHg@UUxEak*3J`j5Ku`5nsTo*f?4;O^3;0jEMuJf~O30zBC_o>Xy5d1jLKzkepKjQ(6KyB^7BS#k1d(iHB2D%q&+=O^0y5Ha8'
    '$)VsU<SD9+pz-#nXQ0`oby{lo%E4N=1KgjVHG>Sw&o40<#g9KflY2}n_~k6Ij*kEnyh!F^E1)AzJJ@Cb`}g3S@^Ryk#RR5!7cvE$;`rgP6MHR&'
    'UK@0PpD*Sh+v$lk={(6Qm?ZIZ1_CfOK4wOoNv(jV5O*S?E0{@YtrfhITy_W>14@L17CMWMpsaJ24b;YBqlxEN=|wA>pYG0vqutK^z5Q1Ee&@k{'
    '3m%&|?V!4-tQ>N84SC1~MdEPu?ss}Ehqc865EXmS+aB~H=t+UMg0OqXTX{NO9&hzr0tdp*raNOYwg!a}Gmv0|hWZ7qL8~x|O}Pd+=ulsG<b{!n'
    'YT^2+vGyK4eC)#x0FqNm9T4PbPM)fTP@jNYC#0e#nCV8YL7Y^BPfHWSJR-jkAVoTQ1zLNpy^aqn6m5o}$kVYGC3xYG01=YAei&h1lz1Fz&GI=3'
    '`7FtijUpI3H%D)whVguo6O;(O((93JCp&u&eb_?vbPOocsPJx!umi?0{YPAISG{>WZcGw3w2cRZswfO)W{0|f4IqC&!a+(ec#BxK($@j+L9AWi'
    '3f~nG+=}7ln1g1aP{#q|kO5dn@2BIHZL%6C7a(<_6Q?H>hBS`^oswj)uc$?uM3*4!nv}Y(uVe`;?X2qMHg2RCn1f79#OPpSnnzS)V1UX{pZaW;'
    'L;wmz+8Ps&PVLs7Z|eJTPOj%P8%hfOn0<KZ_sUYEs4gREHhM48f!V4nvaqTR8v_ctaiI`MAg1V!<nP0P(pDYdoUO7YtD_hnPxu6`gAT_N3no{<'
    'I(4#wp@0ECM>`Ov8-dB_mGC3z)+Gv|pt!*f&ETLm9}!@-LWF2FEH$I?%5vVnBvgPzaU4)V@^}e*Jg3l|p5WhQr*w$JEc-hb@jOLhhKZvgLkx3J'
    't)8<CjZp>g>1bQf!gd&JGb6`eLYu{(t9U-8k1!~R#CGU!cY8dJC-MF56JZEbfT1-xvuficjFgNcl@O=XplpGr+VfIzS^xhkxi>9$MFly`_#l}r'
    '3cOl8RD#&JDUu_IgoOI)P}}RNzu8HaFBC9pFa(K2^#F$YinnM!`d|{?8Bir=BoW(Gj>W}Rbrsz6H>k!^ZgL>(?+9&ARQ3(2tQM6Kw`aFqKh!$G'
    'C_<sNCdq<FTSwXxG%d^w!{wYp1bl7Igi4DRB~SMXS@&kN!(=adAUjk>F|eGFXwb-!@s6a~2z~8va=lo}@xgMc334Fa>HL``R>f_KP1syMAq<^Z'
    'X8tx6DUQfOLZ+ONG!QROPcgDW$j82cMGFUvX};KOL~)lb1S&DCtw6b+$QqR}AMR&Ml#5oZy<zb3EE_ZPZ{YcFB{?#mD$8y+nJM{m?cm<;l=d|r'
    'XP4iouVe)H>V0kdx(YJHe4(usG~x3VvnSW;^CoKrhXb4EeeVjeZ2YN&cGq$(5a`}6fdeCW6{O5scV3tZ*%88wHO@5)jlWs4#|+bpR9kh^Qk#QT'
    'u;UqN9GTy5pt1OW$~a$ASk$#P0B4lt=Jm-OUt?r1wV+4j>>yf!5T{Gb9yHHDZ~LZrVuKqXiuzkVPAM+pGMx^gmp4{!NCFK6)IdNjJa(GT+-`Vt'
    'Q|zabR)WT-U!FDPYot74oa>A!*hJ4VRWOXY7SxW{LQ#;p07Iq;940^$a237yh0#?Ov?j0mezsU-6IoT(;`DZ8PsnEUmK@T)fiA?QAutoPPt`JO'
    '_<AA%kh-R{wPM^c9Z}p*$G}p^PQeg*z20oKIFo|X)2B}Z$Ej-g2ZgWF@(s$mS}X`#3)$6>FIMBk=S0FfrkdF`oNdiY;4Hs0zS6C+^XP}_=D1Q<'
    '>rKNbX*Kz50Qx{~`pRl{gdHpcF-T&xrKgZzP!W;fI5^%ScdYayo+m3jjuxe-(`=qZ@uJA+hN(R;$)}m1YwN@aUX!s-rAcfG_fnVLN1@jf)5WzH'
    'R=7;M<Pr%h+#B8dov_mR;1;mW+eU%HnA(-r(QvI9vrA^mu)?O;MT=qRH{o^L$&TJfWnZ&%uzAc<&MgClfv79YKcU}{)3q)pb9V&?w)NSXJcn;~'
    'JNl3~{HSZ<R+@Q^qbE$22<ApbRtTsaW1>Ev)9~7`2r<k7w;U$`gBI!wHUhrEPX-K*HH$!+cShpgz4J?q!1v`hwF3gJOSFrXQT(**F8T`5!_usH'
    'bZ4<RZ|qY#&f;<2cxu?|$@9)r%jd1K-uZitE3f!zl_^_)CGVXPvsb0_#;Rc~Q~kldOG+(S^~=uO;#s?%cXs>5YA$~I%E(2nKX`DoR<QfZs$-R}'
    'EbFd+=Z#gvWeLuk(u=s%FRO%?Tlii&mv|W~+84?ycna*0;+OSZXD*l(QG0Yxxe3m~=i0=B9v4*P?<x;f#x*5UcO6(Q;8J5BO*i=M{DAZEQ4X0R'
    'siIc>i&dRh>1F`LDZCky;C!(vur7)@0lvc1gXtYjw3`+1a?q^CGJC%cP3FT@P%z`5mXPqkb_x~Yw0aeFtz_k01fupHPuhQL(Um-bKg4<R4uftv'
    'O7Is@=00Gi94&HC;_l5tDi5l>LYZBhpyQ<4vQquqOp7Dtv$#lP*x;l<A-~k(+J6dzH(~Gs!`P?<4~T4bD3S>kpS3$ee=4Bi(LeLdv3{~mYnp$H'
    'qh)O)I_?9FjM9x7W(;IS6hDi`p;(0J@JbsW*e2z`RBqf3jHXL!!j~u*f<+?CH6+u}qc26sIqZUbhzYkB0ZQLxnjU!Pw*$|6yT+(-)x^6!tGY3r'
    'wm4#Yv1(M}XB8|9cE3klSMgupdX`>+iuEf~dQs7a4YSTdHoz1Ku3b=0M?|9)_Y4?!yOmze&Y7mfg#lzlL`wCF(33bn7iEmgk1OspFJz1Zj@~wE'
    'hJg=`=gt$sq>O`VpD}>?SNnR|X6qtNE*p5G4V5BUljOd@pw_1Mt_8D&^V%)gGUmFrxUAXQ#qG6LbNuKQAWw;UW24tDzWZ7qWW_}_oRXrPOe#?i'
    'z4-Vd#w+i(ErS)7ARM9_gwtF{v|AX`T3HHNbKQirZYnOnQQdLg(J^Q~(hWaOTJvO<z;4#2EJ7ZlppY@>K)ZR3#<a{N^;b|iWqaP<a(C7@xKWo%'
    'k_1`{$n&;E7Dgbyq_Ky{inMhado4r_Zxd_g<)+|kjMH_S5U@zyQ91qet5jqvNpt<_3s1)*w6+{R1V>oNtED`t*r(n#?it<CKPGmoR=Puv--J9R'
    '+R>p#pU)HJ-J{}gQS0TaSFfMHe);Y&dUkjS|Ga+x?xf5MV~$|eQK3e8x(<^lvxb!hv!!@hW6yiFik|lU=1@!mTLY&YOesZN6K2zw%-<nnc|VW-'
    'ssQchN9G*rIrg3J?sD=7ic+ybf>=)U)z}Pnyo|_IB+DfMLo@RMgOQ#r7YVXwU<#ORO6hHNb{7}AMpm1`c37AT-5|G3;hR%u9-mr!=L|QwZdtg4'
    'P_u^7v2}GgBrH1p9G6|F(jRAob6QJXOHQdw3XVBt=>myj2Qk(LQC{bS2y<wi7%v@Y_vO(Zi3kU(ITRE;g5^3m&W`3t0{V&bHQ=dyW(et8fu<)L'
    'ty{{c$N8=`^6PSbQ;nn16;~LQ7V+P~p(|^3Dq45ac?#Dz7FAtir>Ys+qFb^2(!9D5>RnspbsAkuYqhQb`f9rC`gU$#gS~|||5fzZ(slH&tj%tq'
    '`8Xn2^}u_SoAnUt=r0~9hIVde=C`j6vc7BfdCk?54zH^0`wE7JYE7sfKu)hsP@2cY1!kK`Ka0y%P>d}02)vikb8Dlu@lTTZsXL8EPj!R5XsrHP'
    '_cvG8`>)oZW_0oGbSFOItqI84l}_9qtsR7QSm|YJ11ra@99*3{!3r9yRB-&HRcDabvxId>>nQ3szea+dqR+*Z>Bc>KAl^2NOHZ|XBgI8sVL*z9'
    'PH7t8^SXdbZd28uuaR5*$v~_v<#X4!saV6;b*9QHanbe6jB^`JF}Pj*p<v(X6cb>U*E`qiov*2f;Q(9~+QVxGHP%O0m^3PCoKp>(HTtVnga|Y@'
    '*Id4lrUILyU*@g8)23o)?79oO8dSv|Rsb^>bq!4`44mseOC7v#v%@<d@2&ZGE&T9~;OI1&-Zq}M2G_GC-Hz{h{eaYiTwcKr6{00?78i+2OOnkg'
    '4KuhGSOZri&bEVmuGNa2{-S<+7vH$OY)yL=W9H&Rtxz22T`<D!2P52a;DH8bSzMlF8oAzN)6+7m7?4ZNO6!e9)`;o4j{Qm6``4}MW<9n1noX`*'
    'aLu|J{nu+=&+vJ?eDq-N;qmhEljQ#Ka{s~O0le97kB*lQJMBGq)4{L%qrJUSA{K#^*4v{p0y-%N#B8PnkV8}_c26{1mybt-N5{(t$=(oqwTHcW'
    '`1rx+^NUJ)D@VFI*aj;!Ye9B<VM3Pdw&58YpNiw^*X6jMZ-BcCw;c2HYhmuwz{_#J*Z_AQ<Z|pUu7!OKYv8Teuw4`3t(&ZU7Q`DgV#)1AeD%Rv'
    'UuGp4Y*fC5>7fQ^jdfEOmCRkib4wk|RTCF^XU+kteUl)ov8}_O%A5e@b+12p^^x1Gf;N41TC8HKy5i8-h%r~Q+o%q|^}AHQ#!S>jY?I>Uv*4*-'
    'wLSRepw0&KRbwcl(7$f4=Xdf#yQ$}igqr@7<aqhy;e&k=XJQb{9(F)H+kez~jAG943BK>Z`v;>3AnZIEJ%sZ6d;9n%dHfLnjvwIPPdc|6p&9w9'
    '>1)Mk`kT9p(!6GrX7rWAG^4vX&AT|wyEx6eIL+IP(~N2aYDPDR)Qr}KYDWKtv6_*!Hon81P3}2#7Ex|EHNrTfyEx9@HIB2~=vmcR<ZsPUSe>z{'
    'hPqxwDs|4*(vd2fELTS|uf509^csDn)Z-f61-Jg&2Di%8KHcLj%owm0dW>s{Ni|(R{+KV$SHyC9VPkl6_ck=QeNg8l!`Fze{e2>Azjc&t?Xj&9'
    'XM5dP+d{C?Cm&c*_2kPm>|OtRkI+fRg@=!l{p021e$pXRLLc8e+{X|5j~=jH0qB{1CwRBdbRo?T4-)jUV1du|j@S6Gay?sCf$Mnm)>zVS0fh}b'
    'd}}rTZ-b7VX+Ro|p0o#p<K=$)(L-ry;@N%t<Z)Z~rv+sWwczK!{LR~2VsG_`^T@*yPfCU`*$}3~jZgf8;p<yj*L>Dv19v|Y_KZlgHhi}q)ow#~'
    'q#P<JwZnS*a(=zOxX{aca(=^}_&h`2zFb_dFFyF?y}7tyZ@z|u@tZgqznP=4xXQFx%RdKyL7Vw#zazJ({rl~Q!QTS8yq`Rx=m6wz*Bo`VmC6h1'
    'Z^|UrIqZyTx5F~(I>lTV>)A;4xt7X7tIdn5{c1G^Y%{)8&wsUMu<p)I-JPAfJ3CeX>=fnHJlga;)5#4FP~B>)*EvN6o7m?-IeAC>7?sx{$1aGZ'
    '$IH=ZZxEbc6l&OmzER-r4A$KltUtyXtnY%^Je@8YqvNfg4}#xH9Y0^f;4kpq8F}^fZ^c)GQ}g%Zt>)dium9Y0U-e9+8y*YW)ClKs9(V=Uwcd^z'
    'lB=gyx#(Kk*Re)+{m(od=j11E0DF8(8HK;{xj6eMj(&Y|SG1tNbWT9wU(`=1Zf9}z9lEosPsTHKk;0n~qK1tWU8BgUg?;DcTrBGxLL<nav1p+q'
    'x7EbmsiQyksiS7F^VIloU@qZ29WLXsdqAYLpOOn&gz97#HKmqMP^GvJW-V~F(){|hEZ()A?M&~*$anSlwX%%u>jNFiXQW&HsdCHKdafqpdv`MQ'
    'v3#{0k5|F-Y&uQMJD_hcn`O+wGp%bN0himTQoUZAdyQS<E+*jLFX0vPt&)86Ygm{$uWP1#4LhrFE39Q|W?e@ps>$n2C|y5nOYKq=+trlT3g{L5'
    'th7KCJQ;!Edl^<rq+RPdtkmzjw(4AJmNp2uTlT@eUBpkm{?{Ab>d{<Jl`dbj>H3;`!nKyiy8WTl)mI5s+bcE(u*vwCyT_V!=_o62v%?Ma6>Hbg'
    'oM}=;u&-_^)n@dfq+apGD_UVQugHn3mjLc{?sM8)H7dXR#W`D5sYhx3x-}gN3fcKPeh-Dqf?SqT<PWFP)oS8)dUK~e-sP<LN_uKkne@S%^WoZA'
    '90D2K^=Kc@585$LZtO%k*6X9Y*C^hb#CSWcmIrv3FtbMOyb{+l2t+ru2t<E)lR(uH>y~}+*LMcJTlT>(ib~eH?pSm^Vy$1e{77^yJ3;jI%p=!h'
    'BG+HrC=h!xb=9@b)LQ$0ya~SvU(#f~O-Alo{Og8Ryy(uLb$7_+YaVjBeUq8ysTF9~JTS45P0i~WKHh>gSg*ZEMOqMFj>hSEvbM2uXqDdPT7B9)'
    '$kcAT-21#n5UU%p-@L)Wm*~z6dS?c`4Kt|XiTGDodOF1`(>fo*@*^`=RsG%9G|E<lY8fIUhg;pUIrnC)``Y&2Te11p-w6J|C!%U&TWi352Wl#V'
    'v)A=!Bk|aC!e&iO_ib#}b^Eu;{;iSUrf0|M@6ozaQgue@c6Q+FGppP3kG#B#=lX;DBJP5}?#%c9Qs(=+Fth*gFtc)_;550sRkJL+t;>SfuH{c*'
    'ZeJz~Y-DWo^nN2NEIHt70P~tyJ^ebKUeo#sZQU6_?+l>-S_aU&n87<^*<H-wH;x&+nd$0w4NQMj<HfggICun@Z{%e7okKu{QG*FL8m0q0=J=a%'
    'S#LS#*nooE!)>8a+_IFY8rFNe(xF{x7S9)HJZ|{Dhk8h5*9`~71Eya*Hk@H1;C+}Uhp7A}LwdQ^h*RS(I>#Cj(|&+vmBLbkxj+T^4DCz0vO(Op'
    '3_Ha;tMD>UkY1tW_S<Y5b4809JdZpujxmQeDQ-XKTe)7!ZL8t57zRE&iQToP9q7wV19X?NjyyP?I7rf0fgYRK(4=6=07p6uj~$D0aH~fgpLskz'
    'l_fdj5qFE5Yo3&(A>lL*w`Ql3mLV~JzDx7IQW&jB=>jC6fYKYA0=kNt^=<SuP&zC|B+knuJw03Gl(B@e#RQ`igz8{6&jv}J10<hW4wp*f&YJd3'
    't!e3StqSL{m|0p<VFfGKwLB)Yx=^ZZY8zf<vfiAr$!%lan}EIx2J-}@+(3?UP)PpJkD#9f`B^+qU~LRQc!GCl2~Y{KLS{(0epmw2!KBKJQmj$i'
    'iW2%J*GPWdG{Djpl4}{<HTvTA(y8}rx#<lmH6+5ukbpQgk0a6(vCwZ{>asi7n3MUUloa(WN0tsTDy2NE<VlNEMXUu)I4F?84WP;FK6|#br}w?Z'
    '`cHZ4^lOK{-ZcEx+0}4&9P0Zs?vUE<TfMwsz>Lb}J+5h7u5V!8)X02W!OQsP-aUP`D@M1P>OIl7W!hG?7j=V>+iS<sZo-7WZSeH<B57|N7F%xB'
    'jcSV*mCk@arct<tC~;>E{v%t1zp>on+8#=%>MP1Hr54)Tmt=fmfQtEZ*=bc`C|59YXrx>)ax3eEQ(do~D03T8i`S1{yhUMVy%VCQjB^b?L|tFR'
    '*YHQ&#3!-2Ut)9L#BKN|ZtA0`7u4&@gyFS?!nNLjwH|@mfvYv#0=It%;hMgHo1H$mi9_HPj~(2^E8x8)?YdOpI0{Mxhg<Wv*5?&X$5{U=n?I*j'
    '_vUS-$v#@7fQB$iB5-8gN5@-Z&S~G`FUMOax`2IKHhx@2V-@m9?gf9rjg0#ObvA*&fgZ<mujSWcvhSyH-hCAV)qH&eE{zN<5gJ5&@Nh$(#uM69'
    '=kY{s$M_DEZG4w3U}J7HT`qXMB^KTB7S_Wm9_EzI(^JGb!tXdv@DJXvI*w6S+KLp65C9r^UJKF$ELyM^#Gu#CR!j*k%Ku2dibnIuXyk;-8vQ>_'
    '7R>jx&DI$UUGF1mF`O_uxV#LUxH8lrPd_*q2`Xx1DQ*|5iKZYw8#h0YFhCPo@zTLkBW9zZ4PHoF0;qWM_qmYG?2hU{Dre!&5pm~;*u)XR#IHh='
    'h=#sFJgsLSyPkJ9kHY=EvoZ*x7sGsK3%j$0eMMVX{l)xEV+^&$?S8q7D*Sgkf?Lk`Kw{k@f&}?uw;!J-<J~+x9j7ypLV>DVv(?r?u=QQ=9E8?6'
    '%JD&C&<r|zod?0;SrYuTJO!Ec^cBd#Ao34d!6E@M_ju~5upj)eS|r05wZ}J$VGH$vU69;UR1HR>bez&%30U{jhZld}d6N!c&Xb+jL)=Y9X)-?u'
    'p3RU)?Q~juPz*~Ro~1dc6XPT}i}RqLBvTxCEIkbRD{Aq@x)y_MO8D*U?eBE<S_1NWA&oQiOMpI4#tE$UVF3IjnQOR&d|y3Uut&@BI2beL0nP`4'
    'iZ9}HjMqZLP8pmn7PI_dclQ*kEc>7gOm@$B5JxbGyL4?b>_HRIS9bg3tiL+};cEBI>*p`ueR|o_yRAA(7^V(_h)<6CiomCZbdXMF*?hq}#PM`A'
    '&nEmk5Fe(b7xQ}!l~z1hq!%%Ahg=qY_WZE(>YXSoX{g22Bb;~3IaP`HrL6dgp8l5R@aZ{~3t)I7_jL(u`RoO&!tz4#!qVdf%;KsIS>1p62HOdw'
    'rNBk8;Xzx<@N?7!;XityDNEVUVGK^dpch$?!%QTe0@)~tgQK3_5Ot+@(h6QL5a<$T9U!Dr&?uqz=y-{_U9J^eP_+Oy5c06ZgF!Y&y$EX&q0Ev2'
    'Qxk~!%Lxo2-kFk&Nb#T&tO~_IE9oK`02fK1b=4l05-pm@tEB<J;^i2Y-en4FWR)gkXe-Fm39OB|Rf&*-#R5IYsuuF%E%%JuJUY#?({Unr*G?an'
    '!OkpR49?mQcRHO1Ck?smw6z8_x?5eXX?9og(dXvu^nZo#Ft^jQWS#;YA$cA3fb14bc3ix7LI#jQJVoD^oG|G2VxEn)0T!IO6?{Z37TycdG#k*Y'
    'aGT|Ape<yAWCZdnEL|$D5Pt?5oTmNhXL}nxP_}xY=w(7QyB8KS28cd-_}I3dyhs6|oC>Flm7EnhNg}Kg`;4z=SvHn$Tbk%LT_nI@hDfpX5V^ni'
    's8bJ<<1HQf@mA1CTS+SnMoBW<5n_HbP#zqta#*lD*%1v{CyH||vTX;|cgsniNthwYajK0#p<VGEx;H^V!3r1_5>EoVn)YGC;TfF8(`hnZkJkNi'
    'tP~8<DjBJOOM`Hd>!gD(>d_1?Wve+9UbyZ&$+Phi>ou`;>OaA5>6!rWgnDhYA8ytZfo?gA682kYDqqZd0Zd9yUUgN`a1VVe<7G5k(S{xYV?I0H'
    '>iMvlX49Q9*?EIvUk_v(K|}q5rU3=umrXetY6HRh18GgeE-(_MLWu;lYl*G!VJ<%4@0@=V7>PKZWrSDo_<5NQ&ZARV2Dn+ioZ%*&3|n^mw~RlA'
    '(V`<ERa2Af?0IJgs;6V<E^%cL8?mEI_y`@X*m1EKbQ&xc<8YqMfOXfvVz0I5!$AU|U4>R6Vo@x_**xpVeUMofDJ=tr@bZdmIZEB^^?ES>XYvr-'
    ')2B}ZN0uuW!@|IkF^)AznpK~+d!Haj+kN$}3dFJ+TQW}3h$#qkeIi%0Hd54upg~B%2n=B)gg2ro#s$XsCfG{*u+wP6s9<6IY#<yT+P+_NIM@h3'
    '5aO|zaw%u>3n~&VRx|Wez*1v%j%BEbt<hUv=TxboOA{2jAP9sN`*#=j=dfbW?6&RC){f>DkQ+kxe&}QWU6`fN$E&;ThoO(LcD<Z3EaNO);2RIa'
    'ibdarC9Qpjmx*nOlJ@gDdE-sqppU)ojkD|#&OA78tQzjbIBz^P?78y1^VCuks6W^}DCV>H>DWHh)D}NO?|9)f7eB2s*8G}}rN(VJ+s+%ShR5yV'
    'oQ<KiiT~)@rFB52^P-`yx?G$4XP8bxx(@M-|G5kU<Ye9U9{j1B@n<woikmxQKEbpnV0YkO<SAWsf#bsmO}-?%(F}I6kj@q^A_1)3+JmnRL<UBN'
    'nlKDfr}K0eoq1ZN&NMv*`3D{amCdvWv+}Y>2O(LOPta$Z0K#oCNG>LqC74_`Jq7v4dQ6J4M|=gvfx<*gS=+#5kmdLib_nVO(M%*A6953{ewZ}4'
    'Z=o7&Gnx@800CUC0dQ#mun9FeO>^KQQGYoaVNlu@p@x2L`q5llp8A00=|{5`{7VV{vL5_P3IEb?Pr4oErGnkoI(=Z$e=g?gsMTr-stN9?(qjJP'
    'ENYHbI_D;?poMvD5Uh|vHu?J5oi;8QVjoCWZPy<KP+)J*8&Cm%LZ2EUQUyDNVXwf<&*Jf@aV7D-YD2qMjO{^i#fe#tcG@Sw?rzW#7+!UXLUs|='
    '4S|E!hn*`}^;hiz{Q?*7z3rdF;5<~(x)Q_hiW}I2;WCjHC-Tw{rRFY8M=}bx2-B6rRK(!A-#)XW9|LVa!)yTnyyv^257_4>eBc0Fay1k+&P!^X'
    'n>EDnh^9gbq0Bt^vWQ!#pS$fKD7PloO4r6<RLqa}JT8oRo%?4r{aTeS-Z_Vlm4d9DJ(lkrzQUhWa0W*nL3b}Y!-g%`6w-rsn6^ZSfX%olZg$ym'
    'HO-+A%Mw-(Y|lBUw!?LW#0h1PEH6M4G2M-x^S#gm0+r+uOeirYmH(s|c37QOtL1Q1K%NKmy?Y&BMd9u9RW)KzFoatk_yvjOz@HJ`3In!}d%Mks'
    '7EKzw8``@_<}jH=6VW1z*j`~}I-DenScIRMaWnYALX-hSUeXgmZ*JI@hvbkcSR@Z}O~>wzh>asX0J?OM4wCLRqP6V=?N$Wm{C-H6V)=_oMPec)'
    'A|G2ob(7_S1Sf!hANM(RMYe}@w6^Qx?TzX1kCj9V!|4F?U)Gk#5s;<5{Zs`CP^iHB%C2R5ejI$!V&zw$$pD@73K*IU?ciPf6^iFFnW0ld$RZ_?'
    'Q$^%boq#2+2YJcBH9Uwm9ZsVk{qwLJ<5Z2&==Wu;s8QM7>h)LJoKJMqA`MN6;1#+Sm1DBgusoF&VnIc8pd$;^2-RC};-L%4OCyK|OAJHNW0+n~'
    '7fvbEX}H({8YV(Zu`!{$K|iD^tSdz7K^;hI3)!ZbN-|%7d^nCiV>n6};OPWo+*-(>%=7yqc85O8b@7?*x^Ea&n`XR~S>e38f_|60ta9P6Qi_=V'
    '*t$rQ%ZAZawSBXRN8|>E-(u(X;$A%50HW17{qY)5OH(~~Sj}h9tv_E{V}qsy{GS1sRjtrh&@Fdx5?G=rv~zxGj8|nVPBC8_ZdcCHl;Bs}=xb=0'
    '9jp-c&zAy6KLGj~8fJ&4FHB8MW1(ao@i3ogI;I^G=Bvenc4BCz#<ia8M~6!OxQ<3SdY&X>VHr~E%Nhi`wsgT@qtD&qp}f?2d8vX81FYhqywrJl'
    'DG}^OqIQ{TV-7nJCF*0uhqP}>7&I0xOrdKI(Q3JyDrWqyuUX%s7G1bZqY;`UI1cK_RH&iQiG88#pk4QfjbprTI<PuESgz6Eqvuy-MXHbjeEXtT'
    'c#v}s-7Vd*ycLXb-K;7MM`;B?OU7rHnqE$?m2CgU2AhpcGJ0-(zhvn@4OTcfBn9FN%J60DU|M!B(=F2UCh9Mi;<}bQH);7!kS!Br%Y{}?;!$S}'
    'R4yuwYBpPqWr=i+!d>I3bc3t2y;W>mI89xaYb@IX&P|v8_SCuTvMA%$x$LsX!DoQ52SZDxSvvCNvr}_!I-AW6W;?B0p&tGlaMwSMCA?nJN6*$E'
    '*c=7uMAyvoSc{MG34dX2_>WP@tF>>NK#egn&jX7%wK@qpY^X+6bFqZlfQLaT+o}GeGbMu$8LUU=HqFgf9IqQKVDk^L_SQ)|KH*4O-%6Zoj@*%s'
    'D!dgmWhE{x)SNx8E4?~kE>9P`=xg;oVKE-lw+I7q!#-3qsLO{6>Ef5`Qj&3*U5n$E<O;?U_WFHAhGcg8LRp20y6i|}UE}mEF~g138|_@W{|v3x'
    '*RJPX7<jAsyWOq)TB48F$Szf3XRno=iI`ZCm1~A)D&WmgCAufRY>axS3jv}xn*$=|>0oquJ&Z0LjMf3FNXRqB1BE&%DM}Jc)pmSqdt0}37wY;q'
    'BwZv9?C>;yTFGV&R879oaLT!Aw^O8StKz6x?~yIJ_b|0aT-O&FEaG3GrpRm|(sdkfc@|(v5bTXMLQgJhw#IA0w-)C57X7VKVV7qTK`y8Sg_498'
    'dS^5?*xKBQ`LDh~tRJ6i?hUuLHGBoVx@jF6rbG$ioUk}qkmc&2={!+Cf6cSv0P5*D>&tVdpNA89CINrT6Pg*!O_q)G)?)D{?aLB}hp*s+DD&a<'
    '8~Oe<%5fq8^I{%fil1CqGMraSTF7Vd{MmGhp<ZgA_wWFZOrODEr;Am$-FfW&jtMWKK{n3j-9~4BKMeNZf6Wk=F@8E;K6(6PaJ+ondDIC${`f-~'
    '1hS8JxAV`-c%E!G53v6BAe|4!$#!tOeAsC}4CGfW$XGjV>N)MF;eQ~p+1ncdxQBbm@p3<j9|n!R9Y!0T`;P(B{r1DhX1UR5ZvbV?0uPeCp<b+M'
    'law5m17{o<A`S+?dT{@E`6wAY;Fs{`$>UC-jbwmL-H)+Q+!6Js{^64??T;Qp$w!_3Lk7C}II<^$M-G^`_+^_zRTuG)hI;UoBJ^Y_deB+rES~2{'
    'G*3oR%p{7jQi38yN+dj2u5Uv{K@22Wz~+p`o9#axAOEwTUE#JnU%>CDjpO6t_s!7%Ppc`*C(|L8-qtyyj4_)hE!c8~@p#;DTQw2%MZarZ!mMd0'
    'rp`{ypsp_3vxxVq9R@4-KL(|)I;8$%{4^BBJn5nK*ft+K=Hq=kVV;In3$|70f1_bG76#kfO>d8+r>X!U?wKh<*(eV#ki!I~HqDVQBV&?NDwBSt'
    'lhaTK(=;z8z$!?5T5(Dwrt99a5T9NYA#{yxSX<AwwPKg)aB&s}XCzFh;-<2zB`*OWAn`#t;-{FQUVseoV3h_61{s6C#Q*f+8UjC-0FXDg@UMn|'
    '>>XAlhZNx1(=Y(mO=l4kP_dIaY9-c~8528LEMcrDL>lKGe%#~k2fG2DjA(kxuVncZmKWnxRpClfBmAh<fGof2cp50{!%jEol;K<Y4KVt?w4VG{'
    '#;qefI2Xm^*%=1IHSSrQE;LwU(+S1J;5Nk*nTr>HRaEt-gV%XyS8a!$WPM0&tz--mcAhkj{FXU2s$nsRVRg|?QMBWhJP9!k$GGbqyikrt*BF(M'
    'Nu}HIWlZ(;S!`8heV!+SMLazn|LgSH@PjPJ3&mv!0P#!|W|KzB(p?qH5S*9+8QgEXSxA7-6u0=8V3w5Rb_GBbTE(Hnz3{6Dh?L|<hamrAC?M7B'
    'KYoPL+5V&JiJM_PY12fgpN)qiI>g1|;lmEdh5HXWAcF08Vh}DL4W2ymfI=uBuFR9^DG*3ofYPdfG>6h3%c||2`^_?Z07!P8L`-RWKAi3k0H?>J'
    '{u6xDA2QH2;KZ=29qynSt|sJ`3YXZuHn#p`(1*Y8w_%JPCig*PefVey((wL6kgscr^xsaLS457Da<?Ch_7T}99UO=O)cc*}|6+9<lr9B-i?^;u'
    '7;EF;8cV`h4F{zQf$@RGKtCS0k#hF-kOm+1`P+T@>PdSrq*qToXzEMm;2NvNSttiLTpG3XE=7wN=_h|*OE)V>2O|yDLtPEdg@yAcLE&$!j4{0g'
    '?268%rqjm#(472c(+6LlFxig&uA+|7P5qr;7KdRSr}=^sU7M2HR%;hO@BSrD|DH}iWz*AVt;J$woPR1Q_(OHU!=H8z^Rv}-=fj>T7_2SXu3z$D'
    '?V=sG<VoqtZe??d5Q?qjQV}*_m4Bz<Mm;AQV1hZzbEAc#&sPUFU!L=1fj&-pw3<1O#0Vm0MmF2CR&oVJ@`kF~Jn+keZi|MMG96Q=qzs3>Y%S)?'
    '1W!a4z(Z@@Hp?OUMpP>3vbBopi-p98hg|F6nl|+*k_`%I6;)?*ROx_JXiLC7xASBqxi)a)p}tuj1i#7Q$D{G+sL=5H;>{t>ez`$1TLdrZA4~}n'
    '=RtB%3n@P$xXIp!P2=&fvmgAXQ1}7}CN3I|YNn3tSo7r5Q8)>XOC;kAL)ZCh!_)67I?#cZRrO6L#+>-lw9|0mXg%&f#J%~!gP8WQN1Yg_^%06z'
    'o%?$zhm86(g8SxZ@{R{zA4(zS!RQf?!K3{SieAa^F~W`?7`3(>4KVlKD5z1`-hGr8+mG7j#6CzKKMZ~wC)0*+)S6#Ng2KL(;OwJ2-KD&q`gDrZ'
    'smu{`3%c{u+O_EAnzT}fPPCf{a%Rx!lnM)VQL8QtRpyyg?Yp`)k`;&&s#rdH`1k=%^#EzC-QV{nZZ{8Wx;%O?iD!xjQtL<C_-^~e-97SUe?f^#'
    '@q%Y4)1s%mWc!#88_FIhYqqdj2|w~8tbHK;@!G`p@bO>|7iyvw<fDEE1oPhz$1fzP@8hD}Z;ycdpX|4v_{q*hM)P5&(;*EX=9QrNm%d_ms_~Ty'
    'I<q$P%Jln2$s_!EQe_@B3Edz;%S!r>bmrv<p-cKr&ovo*G~FWmN`kMFX}DwhnkFD_WidEa#Q?b!jbo)OFW3p%RlM|FMsgBm=A0)gb1tG74;Jvm'
    '1mxw6qj)^-$AfdpqkIS)?VW5mDhAZjK`AMv5y_^=T?Z*YGR3NihrcfKMZ|`?*$lJx_Oq)*dbCQ6_b{nApZeF}PhDpo%bYLfIFJ)29uJmd^uwgt'
    'vH=*hQN_A)a${b#Bl0*z<a40k@HccHs~jCXIB|-1sCb8planIqaPf{?+^QUnlhHzGe%vs^v&d`}EwY&?w367_g|4ExaOR-;$cQ684y=B*SY#7X'
    'ZY4X|StsR5_<xc}1OHLL{=_&Qlv3&*rt7YNkB64>WFRu!HDqHuqA^Uv(on7KaBGzq$MYm^6DXX#9hy~!!p}I4O}9!%ROxV)j;zw*D)uNXr;ISS'
    '_PqeHWAvvAOEEe|_jd(U7<2ydG}t3)Fxd0zoRkwYGTo~DQeQlDX0$twC;ed@Tpd6skJ=|dA?P>NI>l?1lH#D!*l?kj`|65Ncv`pulCzay*<XvY'
    'DGVEsU}Y6Hg<VA{4z(zBl%Dtt{i+xG^`uIcRpgf~F`U<C5K$U3#AYoZBZ337RU-QUJ*BjCQ(BoRt-F-BP3jsoSz7m3s{1R|{gvweD(e0!>i%lG'
    '@k|Hq%wjevF`HkT)pabc!nm)iWbiBQ^tviGTjJp<t5pq4+QZB~>=@%v4MBOvE%bKuE^dMv3d;s|f3n)DP~f31%@?c`oB>R=*Ma)@?pJ>IE4@3Y'
    'K7uewDc6-j4Bwk$)TD6pYB1UYBxb2m^#rr3=|tVE3F=n@#nnq_yaH5-s{A$k{;TY}T9vZzbrALpqwKqvH<THIs{4&(vtqJY>14BFvRPe^Y;b9R'
    'Z!L=(k_~s5M&LSRb5)H0S0<Yq_uVBM#<=MFhGcV9th-9Gaif(495?O^-p<y>o`qVnR(-ej1{OtSfkpq3QENOqH;P>gxuK(K<>Fs6^r=pNzFrtq'
    'z0pL$W-lyWRScwAbk(jHLKJ+&`cbbdMi7hh;A!zW)dT1Wa`eUnc|(7MeWzKrC^&&bnXa$5%|VOTh@~u86<cZ8NI~I7ly<2w#)T#)3fT{aZWc4B'
    '(zTVst|<hxF6dpbxCMdY;~IPXeuCD={RfYg+4k`Rv^EaNPv0LsJ`izFztKNmBzE23_+P()Nt#z=2dedGIw&H&;YfTMRf!l)Saf?WnB?|eSx|>q'
    'pE|U4=7FMa$w2AA+nt~a+!_}m4nm9lh=mIgaX5=};OFzAR2Xbyx$P$F&)N{$aSM|fhSOF=xZdl?@w?Z-C|4_#(TH_bApJqFvbD+~?ZdJm@4Ns*'
    'zI%MxXoz?*CUfjl*W=9~x4zVaupVq9puNy9+Zst_)s{^~?yJJ>lQlE1wS<-my6r~;=+}pHP_`2~T*oz$B%kcVE>J)#tArgAu+3WhvZR_|n4;ZK'
    '!m6d&OIaz>X&BemW8RmG*Y_B1v*|Stc;J*QmeN)q*TJ{``f=OXbRn&^ZMg>QYrA(I5BA7Y`RGAu%$e;`41;SqU;R9{*nV(DMuJpf^GQY{y#Gt)'
    ';iscV@(5NKu)NGJ5)749k<(hBj(VbRx<q{iG9xCQ6ySWDW;!xpd6passl%(n=(gi+n`?g`w3}Z34IoxG0wHh(ME*<ctpM=9%2$AjYQ_<4G_C;d'
    'kcI>PT{ZQA?G~fkH3rg`?yrc*v%)Axqdai}a>}F8t>~?;*T<v*#n7(3=1Ezgw*#(I2OJ-`XuDv&x~;GJjsb`N-64QJ8VUsw2uNjd=o`&gu_%oB'
    'DggP`QYnBx_|Y4vD%TADyp1C`UXXu`qd0ar>y?s8uGq!m7EmLyVyDZRrC7j}0=oueozRZK*N=yz`xrVicw~z39Hshb|G|I-dDu^`+F!!oR`}Od'
    '2Y+?^5<A}Lm`<0KyW_w537bKJKC9@8jZkETF0+J*qDZ`GQ$WjOS<!hsOqaQk)zr*}+LV^ntaIW=;k+Xn3JcuSt2I@>cdN=*fA|wY=JGuWgEc9O'
    'ZVh*9k%K^G1SELEYKXH~)(c%$RarT_b^un5yz~<m%~vAn4|Ek;*;<9nz}3ScHqp*D*3ek0_F!Kgyj$QCT5YMWY6DxVi=|jpw^y)pPJE;>b9do;'
    ')L5Jz;ztGhfN?6?^YIV*itK5nDvBm|bnKe-Fp!+Qv$t1vFvZLAi_f73sOj89qJ-OWD0-<1V4nEL62->PTR<Y<F&AyD)FLp5BP^k3m#O#<HiTup'
    'F1?@P`ezF3o+UKWUwpnyrk!1Ra!};K_%3)30wQ~!JA0i6bi?P*P%=17;t82a@Xb$r8TG4pkOZ%%gBA=;Ji$BOgHIn`{C(#Q-`l?PdWea{MrkrX'
    '&~BYC?rC44sa562_@56iqvt=peEyf$?|$^(<-S~`V|5FtnBMo|veFj`nue3<VD&CrJd+pKi&>1`#Q#`vZW!?<&7%d(&J_5=Jda|lJblF*7WEI}'
    '5~+yW$H`vO_1bvQK?v|$=I!w`#K$nMN5@+@h#isbXG`X7I^GgUl$1Y|C~}oA*nYteMHG8UR~ou9JCD;mSu<9s76{u$bwL4l7Uv*6XgZrSGDp7t'
    'MWZ-hK!4~^j@8Xq{HeT~#+>{ydMFo^oV^qCw2|oIt8D(9Qjx!TYaAUCpZUsK&j%}`lqzz<#E8+^`nnN#($Z;ph*2&cof@?o?ra$q3UU-I$K0DY'
    'pvO&SOtZPwnzdr)JT`^0eiBvsJ4-~lw!`I=`8)U3q)&@`%0D)ZhMvzvfW8kFjoNAUjTI!C&sQoLFZ@Itfi>qx5hXs0B3wfn&u+-~A*El6$9ba('
    'D=&&b)`z#q9NncE1`IXIXUU)&(LYh--vvD@<r{B^e1?>U(|fP7x+vM*6;0%gJrZVS!80S*x?5e*2j&VitJ52iypFDi6L&nX*_p=~4F=CWK2Um3'
    '@wSCI-1w!M)~23YPh(9D8-Ka1=K7J_!?QHUb14-3ZrnP}02djEVj1kyZh>qR#KBRoAn)EuD|k((T<A)5sG3f5&_ow_F+m_?H+`1k!62K{omk>_'
    'NF+0y;b9QZPnVN~jw%cj&=$>WvtwWwAj}b|LYt};z6;!QsW#mOj`h`@?3bYF2CFm~Lt8<fPJlDatqu}W@d7%kp}zCzhn;@1fS%*yuJP`qakf~@'
    '@`K&oGbZT~kTj6ku3NOJE^~iw4r>2b6u)8Q&XRe$U^aBrv*)sRLaPT)FZ2^N+-QZGXJc)kILIY$1s^f7EW8(H(riGZ%WamoL6I&v`{@*km5R5R'
    '-aLs3F0w(~XSY?OmrfVGFu0cqMtm<Uh8{3&@3)`ae`quMzafAnOvEZ6C_rHg!7&8Mw1O8%4G9zMbUEq6@_=H)6mN!w9tE^J<vej>c;*=0l%wrt'
    '81z@gD9ArOk##d2<~`aFpxosdaH&B&#>H(Bx%Y<jwLN~Y_r&S%3waN!>?s2*5EjuB6*sT1ZMW0eYp-wk?VAr3tvU(L*S31!Ib-oo&6Vh+fRs)N'
    '0WBe>OvT-*yfFSoC@5F~!=l{bu%l=nR0{EW*x}_seCf}1vC}zmLD&kez?A4ZKg*WDx3rZHo5wi86&?II&z3WI`7_>a6sSu=fJ<8GIeH$neK)&9'
    'CklQ-ToW1%-a<#x;3woNs*RxW_NQkUxK^j7cCQ?)g*(9g`B^i_p#1z2MZq6`ekOOBR`3g~RQoOm(%gA7m>|R*%Bu|$!{B_DrNkxR6M`w;g-ijb'
    'IDR<n#9)h|*9INnv!Ij=Ei>mTm?ZIZmSw~?fO}9Mr4LC=A?`&)S1^;*S}S-Z`Row31{4MeEnwck5tPmC#GtceUTih-{3^X@W%JYB*>JSmxxcsH'
    'YTxfX*l)pO6Q?~$X-}pz+j4&mdB_Dt;&AltcX};{wZ#LFc6-p<9=ch2c!w6wNCjc{@OJL;@_4J~5;zccHr?UdFN5OZpFn~Q8tNAwOI^%i6q|Al'
    '4ikIbVWLv^;>S8`@6p4@KI{NsI$2JFJo&tYWuD*>l~WQNV5)E;1i4N~qD_Fv)EdM|HTbkl-#d@Uc@9X?<zE$O?X~tgKCDo<8OHM=-6yL=FB}pe'
    '!s#GJqZ=k>Mki&_EPL?evm{5(C5*A8bC9Za9zUPtD$RQhWZTKk-a{X@P(2+3iZm*`+am0MF%)W}VE$V3P+VlbX!QE9eyS)8j}kl71#AHM0}>8W'
    'dcj-7x|R2w!+Xq5jVpYY^X?W*A3Bb8CF(d}1S5xa^nN;C*%2e-<RZpn-J_r<?-j@+L8o+`uD+rcX%bz6xNB1Cy1vqZ<kHTnUT)`{I-Ho6xQz`P'
    ')2}iu7#N^3)TbUb6#xZ7ZH<ZFgFmf3-_)a<B>>)(W<yD#AF~fHH5}|lF?0$^v(bBz4$M|vk%bro%7!TB#)V2CftbQOlD`jw;NAPfmj^g!t86Lq'
    'X-|_&!Uv72i5);Yq=Ly6uuh$<U?^aK&jTdpY>HZ1uY?~#w=Pi#9b^;Op&1<1=Jfi^R)`SoxL=_gjaM!6%S%E9NEF8*@n_xW@ti_;dV+tKMduKQ'
    'S@w4>;(3b13=>B~h8X6cTs>zQI-8O>ezYxUVLJ@AnUUkK;R<xobbvos@q9`jVNei>?GpFu_IMmm;``etAuDGJFtjFTR&AVwk&<zw8sc;s)Gg3e'
    'dtNFo>;GRR_on5ps34b?B!yRthe{9|H$`#;k&sYe9cp`Baz1a80S}VEsKF2<64e73>MP!&lPRTUFbMAqs1n}EZ7Rp&Vyn6e?v)&KlVd42IS}@D'
    'guW*#`-W7-zmyTTXSZEH)H=Z^LZP)L$%02)M_R?h={${C^>8^ilHi+hAFR}(@x;@;Le{+*?J(Jk9>@-rQ4B06BpNhxWV|COG)h7dvI#KsSjzFi'
    'a;gb(Al>QwnI%@mZHi6UTs|QTomgi6HWew3$VNiOoRQda^5yAiLT84lHQ&IZjf2d>n~f;$vV}k;hP4%Zi0APn!L$r$k4l&i_p>F+MJv|c@OFz?'
    'HfHAE$N-L#9GOp*WG9IyGbMkn9o+k!(!S>7?D8A+m5cyiy{~OwS3!oDFSNCS_op3Emrt(M=S|iM4hJ^R``#5`+4xfl?XKloAke*C0tZI$bnG2j'
    'C~Mt$VJyt?A`!t@<6N`Q_?snr%rL!3wN*DQwK-@7J;6;q9GTy5pt1OW$~a#}qj~cAS{r~f%5wAiWR9;fvX=q2hdcL*I6H_|m=sWxm_2Bof!_8_'
    '@x%r<Kos@2e4OH}N0;ez2)(?qazhemAfN^Us+wMnJvj|;Zi@X>(n`?y^vkoRe2tVxjB}ka1)JzurV55p*Mi#dS||!K7huRVfx`r70<NMLzc9MW'
    'g4X0!FESd4s<PJQSt<lfJnW2bM@#6AZ=eftX$Z^&?NhbP8or)L0Hm%dZLJvh**xpV{d5d0h3pgzq1WrpR*N$!C_R1pG;sWBh9jDts;0}nsH@$x'
    '#MVM~HROxcIPtlWu#Tx_b`6(Xvl2MV?~Jc>YwSGwp}IMFdsNYzhEdXL^4UP#38iTaH9Nu%7X9)@812G$%32E)A+?h3*AhV+{Ovqhp({U1Pp8>D'
    'iDJBLihMGIgrikvcqy1N1Lq=CMMmwbEbOJXMHTL)F1wFHuP3I9YcH&DnRLk|5>~i3y7xO_rSripV4Js%0);WPE3Kp9S~F&s%#>k;O|gp>!_aTS'
    '>$Z~}y^qSiMs}&N(kvxXy88@7U19zS{f3;bb@QvcD>$&N&(`EQe6!ophs5DWT@yFVT}MF~cCYE8>c~`yU~WcuvAQ<d9b=;I-8;XG=ch7Y+_)2t'
    't^h0ipIwbwom|p>t97@G5Ul#T)3F<rUmNeb3cK!Fu<E#uzOK7EJauf}TsNq^3zh15B!;esV60kB4eJG_&~gFO5ys-;bYm{eZMo}`oOv-H?o^ob'
    '4sW(2sR(F?wyBjF+T`&G%JiS+X6P$fGJc+s_t&)E`Y8x~J6(-8IrL+@FuE=avg#O4gs#DmJhLw|U8~<Vw8Adk)p(Gvl+<Us20OA!y3$M6<skMb'
    'NDhL%m%pttK4yY3Sf1A=E&Ywdf@E;b91m_xVf#k;Zcg_DdyQT*X(7mUU_U%3t2T<x36feIY~x0~%?2C{=(<Qy#|M#YTUm#<Tkc^=V?c%D5b2hz'
    'TgO{uBi~YaPW93UCD$Fx?;r%ImO0kbS44cZb{Hp;im$FT7KME=rxlFv+37s*Li^%WgXxRFJWm*$87la@M8JJm9bzb@5inB-a6&Ae#CXu&6GM(B'
    'hEPRLhQZ+yC^roLIu&#)FKfgm%qNauYZbn937@bLC8tw=1YY3>R+9U(=ZBqF@4Vr9*l9D)?m4~+gBNU3e$ITvt$CVE6Cvt~ia)A1Z_}%ES{dFa'
    ';;5_u&>!MFA(8w&ef|Y>>klC03R02bpw7qUhLMCH-+y=+y?*!lFnaaZch3)Bzkm0s5`rDYXv9F*JQ)xVhn$Ql8fF9b#LFe%`-Fl&D+l7z*WxUI'
    'lBqzP@|_jXzMs;`-FcMFqvSKDpy@<yDb9%qNbxjJW@A_^5qot_9OdxV4&fHD;H9{#faHoXw<SkoOr5V{kTwqT<WThw?_a+IT6q64dhz}_DPl%^'
    'A+Gl5`TKXTUjOJ6QimuQ9<e7h_>}Z4Z;_>_=q5kD!M{Jg{e<_%vqg#C>^!9=j964rs-dnIZkFG?rOC2<2R-UAx|D6-6{EL+I~5QOu#z!{0S1Tr'
    '>T>&v>t^gd!@hLoQXW%-9w^g%gV`Qzig`oB+PSfXovJ{mGAg1QkSOHdk!+3tUHEOJMy7k4heGo<inWHO*V?|f-YmSx1ny+ExvwAFl*l@DX4z$<'
    '(`;dA6QvF~Pm<X%osdq|tl4mIPiz(Q{PTR#P;GthAE>7PlI2U81jgx<HbBOu*L>?ahBmUi;NEUMI^Ozifm>ZeYobMmZJS@14uA9F_~lks5s7&('
    'rS!3GHJPQ?tg2E=3QnG-sh1fcI(rXvg&RXW*ao13OD;e_u^|e9PS~rvZP#dklLbn*FcuoBw!Dd5N+U-a^Fp~%Ype2V_NFc&AHIL_K4|=!#aZ(J'
    'W}gINl)QWCs2B8;adsI7nX;6lqW8mdpR&OMp>;(M=Km-3O4K}N=q;!^C4mA6$#cf?c&q1|=pKbKPeGIIvvqb=C;`BC$hm=m%RajwyqIUpQ}(pz'
    'tWRr`OZHUMeJgl29&^BD5sA-MZi=VIp&OzKaPgq^;Gqq)S*8XQ1Lwm+Qw<IZ0ttOZnWnIuNoC%`K<!u({_%dMWIynok{<aAl$m9=4$4-I1?EfH'
    'cnEf0blfc`E@O{iQ)3-@SjSMLsv8bE0jCZCPcj+SFpX&nF5_IqXaG%0-OWifA)@2TG|RFJB=kY@fIva1VIUM%6yp-jdUx(P&1-C`?kovZZai7*'
    ';8-k5ta@3`$H0g=ju5s-wKy{SZX^Xtrx)ae3~hr-VSjO2X=qr5Vy_56t+v^MEj!JVQ&31j8|W|bYzNGsY#ryhzyvscjs>R+wk_LmiVQxYibPU@'
    ')4dQPFey@=&xqsngn*_dARXqiGgT6*0LO7*!o;WoQaP*#_aA{tPeyT!vN7yNIH1tnl9gkv6}-*hbj}6($ts%;*@DEBXGbqLVgx2Rr8Yd6Y=`PC'
    '=X7+&u9&LR87dBo6YDkNi!2?&mPQ-S40@8%VG<Hv@(Bjgso1n;)m4v@#91`N{CfHTH!-W2WuM|2Ps^4aU9E&eV<!{57Uh=k1)G)y3SIH9kGN}W'
    'kk=)5CE1{0gIm%n=z|`8&a;6umX2dLwyd0Qi5E^DH6HpuRq!e_XO+QyT3(MP4UZlzHXaxM3a?}e>pEY8q#Ta_W?G$Q(dj%MN&(YV<*rO-Vcl~B'
    '+X-_9+mebb*Jnm8XG}hp)V*p6-eek_pW_)5;zj*&HaK?~dz4(wVu~8HnB);j+`&Oe8}<pSk05Dtc>@X7osRZSFsU3SnA1*MYsK`Y^!G=i4QNa?'
    'qxazzj6ZNlR$l`c?Ie&#VNQYKmwi~D6a4_@uC9^o&OmzJ?L6r`-f!9(jNI$x;I&Md%q*lok@1Mnx`I@%eW5%iO<EUv`Ts(BVlo2}l08$$TZ7qB'
    '=ZqW~MT8bF7ug`r7dqIk(F8{bi;K9Ot5;y|P~vVlGc3n8_;3HT$nVV-jUSXrFoB?xq-xSv4156P8`NUR4L3v5HFxYB*n<plf~qiAxpSFcIu#$`'
    'SHq&<XTu8|<e>;sr%3Ha(mHJg7(Uq)(bzF|X|Xe#XEGYX40sDIQ<41YiNj@7@$Y*jTZgux5$5=Jd0EH)0HD!;L8c2_w`9Ty5sLfzM)n5uC@qrc'
    'EFO=Jr<75<c+t3$gu|Qk$BeBu_)R~V3bUyKz{up%(l>R=lzw&8YPDeT!ZKn;u+u)FG)lswLy6BtA-f1Hd1>F}(>euN?O+PCtF}&2Hy8^dlZbt6'
    'C{?3C&4c3m(tsW19CeGF=EdafZ1=!WTHwpcKavRXYn#LQjN<C@fYtiogLLW@b{}zm;4__kM1FxM5P^8|Y+gh^S?8@k2aO(&&><D<H4n&}#6Y;R'
    '=HELnDPFYHGRfXwe1KA4g7$}nqy*usR7y$9ToQgbn?TXW+Xh#8Xb6uq(+N2m)8&+ojB4>I`lPKtwC;;z+cscqD|3ko59D~(p*nFLQ*7<!SvokQ'
    '5Jy=Z2lhzsC>`;4Sff=W&*bSum_>2keTI_6e2j8n77uar;E+xO4_d~bnbi0IYH}4%$m0iUxn+vr)AT<4JSd&T!mh>%3y=DEafPtzD-jvv5^%sy'
    'C^B)1hlUv&v*GhfXcXMw14>7hQLI|fDPU`Df{c8g<j#&%t8;2d`BY9;pk@x}C(bA_UKKqPov}xD{W#^`o>{$;Clfvr=hWFmkFDGKqG|R>F_@>c'
    'H;VP=<vn7P7#7Dx%9&V8XL<Zv?Vc@#ZNtD@?EPxzTvP5dTRu>r`3##Z07fOTq3>j$@wNkiBx;-&HO@<FoSQYo-Xod{C4{>B%>?w0c|0z59Qooc'
    '|BI;>NWK%#K07~@spT;n7>QkK`LoRMF=1QUEQ(B?bzKuoc8c$T7yZi4vo~}yrC}oxmb4QF{WQ+~CY|N%>%bjmm(vo+9yo4`<&Z(@WP=-rHXNDb'
    '4qs2|IR!obD--jE)?CA!hK2#CDP>eKnAX>uNt($li5JoNg^)AlqtQlw5a=ukwjFfS9bk-NGuU}*d^~VDi=Hk+6XubZv7eC}rFk1Cz;_#0EINV?'
    '!{72QwJ2iag@7x(m8ThZxj{@>*z-M+b;PNSy{Czrf~e+;;`7vSJEv858#F3A&RW_`irjzLLpxEuARMC}f30-kqGDd?=A@nr?Y|0L!obGb<g{i7'
    '<_UG70w_>!8B{A1mO238a#MY{->MuTC@88U2t)zfMH-UVIl0CDCtC*Zl4Q8enK)Vj0Oq7sYfh{j1aCm};{_x_X(*yj1|mTfED(HB=Xz<M_(Yeq'
    '={acf<1K5iU9&?R=m9Sw8=^s;xzZf0uu!C9b))^H+TEa9!tfUP+r05vmAHxDbacFBW&vZDeH0NfL{Uoy(tM#Y#n?~jFYPG|deSGQpPt1jBFCrS'
    'bqZ6S%;zX5@hN0szMpe=a6BH9{D<a6;q&68H6MT_<p~L*FvN3`B!LU#Ni{qMuA$^_meW&VxD>YufT^b~k1zIS0sw`7i_l_GcztZQ#+{rtLC-e<'
    '*NSS^nywCRHcL=`9d%V%(H9+wYIx=0`gQ$#ZY;Czr1XU}>p{mwVi9lbu`i}sRPls18!NA2PSrhYjGAbsv}ezone%TrcrS4k_*bSr`x9{*6i;R5'
    'De6%DRZz@J;}n)(Ppi;75KCEm_r<p+@M;HKt33rwRaI}kG+Pz13RL!UZOO7g`kqPjLJqW|qo~OFM$p}DuWmaR->99eq@9YfStB*OLU3smQ<X63'
    'J0gVF8fY7yUfv2RvQ17-E!?KdYZ~X=w7L|V?5QAbA}!Jo9RzrC77E&jrPN7@(h?jXEs@_4ahHo}04N;%$Q0R)F;o!IPA_7GU?0U@ZaR#i=&Fbq'
    ';>ax&^L=3ba}-wLa&mH_@uq(Mk1!Axl*kp4@m>RCV<gi@A@*vVz*Jc@`o{@2E1XwX;1y<8x$svhMNEHeU8KonL+`1%^g?sxRk(4Ct`rXy4S2Mm'
    'dQDk_pg-PbbH+aB+r4iVqjzYWRMrz&suXcC%5;bpHBFu;)FwFOqp+kI*@F0Zo(!e(N}D2-ucvdTUeM^g5cawvrvm66$yK79HJla|@8imZDKEKF'
    'jfp^%VOe;fKkpAMLW0kx!cb4lg@!+2LB%A5V%^(_%ZNgMyE4v@wskKKvcBRltu?WT(uGrmdyt<8eLOlXTE6?rxHt`Dr5LEM-OyZfexFlG`N9w7'
    '5W2DPR0P=ZCZR=RZUHokrbOo!Uc^Arjnoc28Cwc{Fy4GYe*cNR9Rs8<B5Sq}S=M~56BZ{hac_y(hO!EwrCo+bmn@etTYG}@-3yTsgzO^c7eYe$'
    'Q{d<@#^e;%D;=9sU89`Y6@WZlDFJ23G1G{H({n9iYB5=kOG!mzjGZvJ-)siEbo{GDG597EH7LmxnhZ+95uIiTW+AdcakLmExx*`9(EKsLlZH0U'
    'q4vV?x>f_ivV{NBu@}DK&WUx~ku?;?<M(<)CLJti=(o`rH<0RRcE3|P#vs(nY)SFBm1m;`MyTn9q%#`+B|p7+v;zJ2#*QtW__f*-N}JW*tm{|%'
    '!a$}u;{x)@ZcXHdv<S#-Lnlw0$7n!=9EEBkZ`-!~p>N}g&aS0F_OxbuS|Ur>d`PSOZ36_pfX1=JjJRc5<Zo^tzf_u4%z@QwTY^c8t_eI{W=kXJ'
    'rM$kfh1yb)e(0zi6-`HLXkY9(#kK9~AeC#5&(gb>xEGCtfkn6mgLf>oukeJNGPQOZruo)qwB<vtbh1uc(|m13s726nIlDxQ9SHm&v+d_Br1i|k'
    'd%{-Bz6Lo4Laq`3V6m+7U^!i6OAI6w*$eS335M&XbIjIxMu+Gx741$9TVZh#$t9oQUIf!58RneVpN@v3q$YDp&z5~)v%#|uuMrb_6gc249<w+&'
    'i)D-n%7DT*fZ6TyqzL4Z!W)WhE77}2D|iM~nIWMo8_&d)S+{8e;P5NVDny+Z{&CJbBt62)YG!6$Fj4}WcI}%Su(AkC8L~?)=J)8dw>tM-X$-f^'
    'ni~#A<sJl|6t|M{Eb0{j;ics`$D#e0J*RVw{pDoF?u1TDPs8_JP#jSG6MPE9p=OLR$!0s_WCYvWI6cLbp2C2L+h8bCJY8ZG2kzxSS5QJG3gFq<'
    ';X~?N9RpZGj{3kO9+P;eZq6V?3pAc~5k+21c!7A@$B1_A3J{U2lvE>mOT!?+3n|3U17G;&i#XeP_WDiS-=(H@6qW5_)@2~ve0LF_?p|Qyd&UU;'
    'kX(}PMYtpNn?ok^UVe_)4Trnex%}M_IbUTqDq10`Bzn=BXr&)6(KizZmvi*ikVMmv<KJxA$sx2e#V{s#QhF9)Nhu16OOVzbgKpdz$Nsa7V|V7P'
    'J9E~ZIqS}xb!X1HGiUv~n6rwjgvn?Tq3jBv_F8*V4>gPzd^*N&@Gu7jR^*45uMSbVTM`erB<XD-?)$*2oG=j?u{C*gh@?E(5(e~A7B}cuoM%%b'
    'wHD>n8p&JmSY5K>#S%~n^AtnSNK2!T>!Phz@Dqx<q#~j8tpk^6PAILMs?>hS)-+DdOfe#O;mm45fh2hKM2;$HA})hymE{U?4N%4+GYU(qkih!D'
    'ImgYPL0Xlkk|J^VsNu^z4C^0L*+@*dGQLr@PiIU1Y@Y%8F8E364MGfOt++^T0PFx{V<3rt*tal&3OE9_BS&?_JbZk-1p-PEiF%wCXeiEwug6~l'
    ')hdHi=WMwTEK6rH`G|mZ%|NKb5*p2uWRgpIDkarWZ2P1QlF~Yk`TRhcy^O3Fm`o5aHlc8RwzMYN?K5~SPq7R&8AC0QUlQK_XgSbIDApQs^AcN~'
    'KA2ysqo39ZpqLs*gVro#_NG)>U8_n!bZhEt;UlmiIu{^pC~~2T(*nr6EE7UKF?(aN`C7tnwFukr9B>s2t11DgjF^NW^Q~>G+7^eZHg&<lg=Ix4'
    'J~fxQCl{z9E&G(NlbBS4Q`RF7hj8H7rdHF$Tb5d^QBL<^sT5OAyP;zZ*6aq>M^<;NJL~iVbH<5fvz$!hs|yHW)c|$PN2_d+|JRJGvx*_n8D^Po'
    '4bzQ%Q}ZoZ^K1DC{^*ANpMQDzt`j{wJbd{MbBaXI-#q*D>E)*)^Uv{C62oqW(zj`B59N1HD&_;Ps{m?zaoh<n_!2fs0Jp(On({blJe6>}$80>Q'
    'qwMU)bj7a_ID9!~=^VLlpsp!7lwBV{x%F7kBi|XBEF~%;RaWn}3QWKB{&$H|yqv0y4VDL;w4@9c3Ei{0L<<56w`-y{@h@r@6XpA9@2CR3lg7ZF'
    'aW`&yGM=e4{BJ&p8cpZ86fQP`LV`}$%!26B0VkM~0l`|nWE(ObXxC=>mYVo?wkZ381Z&xs6$CURpHy3#*OvWPvN`*b+}kmluTWrj0yRrZIIqk='
    'O*<u!b;8{}lOzv+YY_23IqL~RRAxm3V*^N3?#ZJ>>QF5WC)rwiRdS8DG4r^&?pMYdA|SzYfscj+9qx%ZY^K>0nyqxwCd`*|GPwdDPWkwHe--Rm'
    '7jik2>$`Esr`Ot7rDpo&s2VtV1Q;(>(l3OOuG-*Jl4w`x<-K4z9VVX3#~3^+kpkk#yzjW>iesgB!TLJ%dc+r{$-800pl4k+(xWrYY)BN8gyX9a'
    'qn(yd&r8Q;K1=Y}^NcSZ_FgbZ$!DCVnl1ajX2Xuz=pAb3u{F|a+O#E-63xVQwc3T$DEK9OQL=t~&(Cud85g{!r>_o&BG(Q#QYDPwFqz?{tit-Q'
    'n$y#R6gkD%VlNYl$=9tCHNm(ecLV0K7I(JtTQ|sxlorCkUvwoN>+TGjp_HXGfB|&dB}Yb27U^l1oEA2cB1Z%?WnD*)HLbxC2Jc31xbE&;=B)h`'
    'v~uA2q8=N9__m&5SmZo?_36b2zDAgs!~}RnCNG{(r$}~rfWo!g1jxu)VWD%GyslN`>2OKso2*l)<>`cd@>PYa@jKv7<B`ugV$@*`EV5J@Ohz`<'
    '7j4&)GaD6qRM%{vYKQ4oD$;e8W|P5m&Bt)frqOlYgeBa#tA2%5?51lzhHF=Nw>pDrMTTibM&Fl{(RaZs%9jf|-IVl<I#gImgJqI<sjJ)z1=NY!'
    '73%a;-cZmDOt7JC6j)~vkbu7L@(Dq6ZP?HXQMeq6vz$yQLii9hbOXYt!)G54F@OE1!;i1u{fHysqyP}!03(3!sAt`J&^u`b&+&E&aeXZ)?`$j>'
    'W`kv+Ps_L;3ad&78t)U_Zk+asg(~%tMH!TiuyMwvkuL{yk0mB$fW1lDzw`J4FEF6l{D<WJyi00?56ovt8<Xb5OBREc8Zy1tAjORHEQkYn^M}|a'
    'mBr(vA%m-CE0ME)e!7d-CG2tvn_Vb+wwR2+vnn?6QgCskcq;BX#+bwy1E1!}>2e&;RjXWuHjQ~Q;;S36bMm`8$~a;oguHMth;X(s5LGO%7DAZ)'
    '<P3(6(o^wEm4{!aU2zb-an#dnIK2~lFMR7>iN{B$i%Y;sno&(?k)Fx8W!~w>Rb<m646++$3E7)%6HGib!bhROY>-mNRgA-g?<(ZT$usLlKuN%O'
    '#MimB`Kb5*EQ)X8LWtz;*U7MVBB+_~AQjj!83~+P@){Xlg8%Ki=c6Ibb99{~iqITh%cIImm?Xk^d?|R<j$mW*+&zYu=F+{6L{5>g*{r1!z<U=z'
    'U+kfo_^b{$tY<mz;@%*x*flTykQwd8)g>@mqx6cBJ+lNO_zdt1yv8G^jGrlTvf`#2P92L{2bYbeS(MMl=_2CD;GFM1bc~Yff}<!uPiGNbo+z?9'
    '(#<B6LJTo+ZcUWcBvSgFB_=AW2=B;p2z(`!D^*>}<tv3VnGUQXnn{&xp@N!(U3Fp@UTHKJL(4l-ajTe_SyKjgtJB8z;uf*p(v2D~;uh}A@i=PW'
    '?G<!8AR^F(|2G})wy2Hgk=JA=*Gh6LfO=*afyoK&M_uL1`YPB9JA%xpvAz>pb4M<WlcH<IzWn;#hrb?vD&LqbD-H|odg{IsP1I7WOs&)&EvbcO'
    'qiR)?E@H&(YK3>J%Ud=$&`M}OXkC%v&Ejwg`m#kr4r`<rE!(LqF%brYtiK>OBm&cpde4@N>=1E63P(`{?Iya6qCU3J=siq&p45N-@==5*0-^|N'
    ';I_)<l-$Aa=oi%9*rJ3(_4-Br=aWXOwR<|w@*IZdJ9@ukMu(MH!#16`D$VpQ0)`!l6N-aahH?EyCRW@Xi~|g86if0JS7x?|{Di#-FzdY|SSRh1'
    '-hn*~J3*UeJSLEZ9F%+aaG78yli|I4p<QXOq>`*qG2Etk&>nR4bqBvT|MlqV^~hss6T$x=$js!)kwT^!qcN*D<ee~5oS`T3JhDxKXcUcWW@}zc'
    'n}ZOp+{>8r`ie6*l76Hh8B!sYhaW_cqh4{<+iXGMJs<%lkwny73cV9IkYIvIpy<R4z@GEj9iEY48iU|40onuFJYwMzD7_HP<M~y3L4tYQ&v)BT'
    '+Iy|`{Ra=*O=TywbJ^o6iRJ2`w4RcCEzo0rx@!4IUs{v-AQOpRFjs4dzpR=@{*`%Jufy!Cc!tLKWqY}@m{o2Cp35|GiVDs_X=sULkcZfTv7CV@'
    'DjMBEE>p4kReLVKm7gq@Yf(6iA)Slpq5dfIc$Qd?s|O7h`HqT<HU;yBVO!QrwBu26?uHY_a}cyBFqlJMc04JwNvx*gO_4da#InxQgtDJ9vWufc'
    'cr)cd1anU@lfOdCVFK29QfbleuQUw@=L4;Ju=tlbn#@lr(pxr)2c%$Q>6w>g^Pp>SN!ll8(ho1`5_?#c>{*~YH<ICjy_{my)wEkITRguffs_<+'
    'F#vK)%Z+k$e!+IO;B*NxNx?j{ME4+1vubZ7yicaP=NF87^MBjusG#>=8ClVq6&Hf^)EZz95KtT`ue9<A0J3r9v>nZPWQFf|mcMbx!{7wcSBK>y'
    'V4746WWkoo|40Uq^J#WDl_xaE$!R=Tc}$dIg50?)G+~qrMU)M#fq0S_b%hx`4@T%T%0&)z+;*m!XDcKo09qIBhK|zCc$(mgE4G-7>+QJM%u+Uj'
    'zvi^$wEdUaqO}WayKe(@^IV*o8)nmO%X}e|FsJjv8)Z1tPScBI%4Q#Foobdb`*yQO)K02DyB0N6EHh%w{lX4^&e@ji_ZQcMYFtQJ&0XKmiZilW'
    '*Hnr@ZKP51z{we$Wi;X$$OqzZFBS&|D|RP}h!Lh!VF5y#w*%a8x<|_~YJf`eA=fJUxaJv<^97K@S$4?>iKp;nPQ4JP*?Xgzr7jY@DS-susY|>x'
    '#oNUhAI*(5&a!i0?m*7-^s42FnyBN}{WdP$x<clST<%}l-L86UB^F=nY8Rcc<122~A2Z}!bj9d^(VpE09c`Q7e0`Aj|E7b*jZmj;nKxwZLT9lp'
    'OGXvU#xY7ExY4!BB7APu+kTs#cJzRu132Y#`R38J(9TMqN_vz=q<A!9F`2HK<b3bJ>xF*Qn@>3vSCJ_}k<%9xc;nK9@}S4q(IQ+Ny(^2vzK`r*'
    '%s$OlX`iXum(4=U?i&cDQ;NPKMgKi51*%v>ysqx4_4N|vFphpg$tV!65Z&(FJ1MzL#1jB}{S?EMp|bwSN!@O#0%qZ5MC1&w?G%@2*0bVki+4`+'
    'CX10`X><)UY{RJ?2EHgt8_lLK*RcT39<ettdsd{tksF@u#-0+>))FI6St)IlQ9$#=>n`UYc3<Fdw<@gIPN;jkajeA=RRz?#)R=*E6`9!A3Y0lv'
    'DDu)g6e#@)h1(4{59cTu(S3%+4N<;uTO?RB?*g9IVcDrLA(UM#?a-YLC@%vfPKFvQ$V0#?*q-jB^RYrGEBsnAwAXLEZ*C9+ti(nc|66QwZuIK!'
    '9SW>h#+7*FiA8l`80v%5yk6O@TNM|4jj&^X6)O&VT`f&QT3`6O!O9dR{MRXz<w)Q#_#w^{O_KTrHrEd@J9_M|D@=Di?|cy}x?@W_BP_U&Q+$oc'
    '+qKW5c}o>|Snq3~QW9O%^yx-j?YACZQ{9j%<_6ZVaNISX(vkVN4zyO-KFU`(_e2tkX7GQ-Tcw0+@Ym;m+rs-5C`~v^VJp0tiu7YdPkQ^PMU+nU'
    'hK{$I&aI{8JBB*`Cl7(U=bg)<o9dsc5h*8Z%OYK^i-IofxR?Z;%xDtNz`zsLCYD(>D74r0TCJAvR87-;rPG(#C-t53TxLD`;d#)Qf`A~+&K~a$'
    '4S3U`GPY*fjrhir2A;DXDfhV5Oc+r64zYNfDk2q1lSh<gNb#B6yTJ>-YL!rE)VnXd8VCpD=xEy=Zr+T(ej(ME^)s^Jp(#ijABl8an;|L)3zhQ('
    'f8>Q#b!c9k=c6)l5rr~Xa&i-KpyZLd;zF97$ScNE26Tqkq{LGc#pc;;js_=1a;!p?w^-luEzREF@yPGc!wy?on<Wd!O^%>*F%JouWxB4gR&Yog'
    'yk$7>!>5VM6WBhK>}Sm-rBo16=%6MynkDpaviQBqBLgy9L(Gat3BhY<g)XAsn`0x=xwcQ#CB^1vep!0XPtSammd=Jzq-*s`yIXG0lh?E)<1H-!'
    '%1|=3McVoqZmU!77!~akxVErj?s$445lS29b8^b04`(q3yTs(U6K-R+G}wpR{TH6tDTb^-dMn3}j#^ne7-QfFW6e@oiwM^|$Bz(Hl(JJ@6B#{q'
    'r^(eqdHCgSVh=xuKX4ggTAAwlzAH3%<#cxixqbh>I(GXOL>k&x$gWU?i?fx$4=}=f#3j3qF(@ivB=i@qIuhYo1|=mb&OnrG#3iNHoq2G?rc#tE'
    'jyhB<rf{v$Rw8HElV?mq<N9N79ZT74D!LobOZS@7)ZKXEx|>a5_qsFMtv$<Cr(3(`1Rw34co;yHQsQN4Cd^I+e{B2OTX$f@!-kQQnXSUM-&}jT'
    'YweM)csR5?Oq>x|y0ndb#&_zaiYaf*kZG{xAbUpMP<lN3PoahDs)2=-+Xy|h7MZC~_;P>fwjZA+#bSTsf;PDe3Qx24^rFGg9iPye_7iOP!qD~|'
    'qiBg;$YNbMnMPZcXX-3w($$czI;Q9Lr{t&}7cKf}XMQmgnP>Co#691<^*VpTKwkHlP2Z+hsmg60EmF`KgrS-a1n|*;F~`P;IBhyNaFhFh{oWT2'
    'f9dL;D}TXtZWq%d!l)bfgZW$s9tVu)Qm{XcQJ;?j-I{6N1!o)hC6<DV)kISazm1z82(I-@?WKdIHkn328@%9w0;qWM_qh;Y?2hU{Drey~T_FEG'
    'Pw4E8GIthkWie)S$%kCtCI<_RlSbX>`TKXTUjG=qdjApIxpT7Inv-Q5_hB#H4Cnt?$BewTdYql63tA87iMV7N^R9+^-WLo_j@;xghFix^`?^<W'
    'UL&OoPvKV!5XV1nxCs<&%p}zuwGJpuPnQ@5EaUzI?0!o{+=#;gNlNiTbSQz(m9$zdk6iEUU?&tkywl#>6InvEh&#nsFpQP?J=y<dB%Ta>!`d9&'
    'Db6W*GLUe0njHCwT|n`R;yhKfw>;In(3%o6pm)mG2Pgd0r6rg}Af~nR@jM<6mUz%N;OYu14n=k0_5{8ZT-)30VNPTZ2$0M?$#(R}R`Mdnlvp%@'
    '0e_EULyhujA^{6WS=@_(U{95d7K%VZZd*{%gol{>5HCDAUI{jaALbZ3J4q(ld`0(g49-LrrW^y(7mHPp!|Y<P5k_uO;0N9+k*hnvDHLRWsvl&d'
    '97D$3d?aR)8z-J#Ud0&KVZ7`0db8EyOb_yS`t+%LuL_=U&*qE3IARn!67I1D#}Fg6a`tKD#)oHzKShV{ahLqt%a4aIUqrvWeEs83ho71zz=doZ'
    'fEu{efOz=+FE8J{{y#52Hf=OPK%C6@c<oOKObbpv#pMFKAK(#zKLfo^>D3}0(*d<f@@Ji?Ar#)k$W$(nYB1nX@^%r-*gf8wEiB(80{Bj<0*U!v'
    '-s&QZ*eo8(m%_E$QY0P*n5C5YV)y=D*#Nefow8un(3=46ibs&eIuQ~m`6*$|jgxp9EoYH{!?}+08AwS|JSQVX;hx!KK#!-dw<n}5s)Bqu*K1lj'
    'vM5wD{G33<Zc3TEXo7F(_%AH_(iEwV>A(iwfRE8AiJ>YQ3D^)s=#oAlI>5hCp>F!Jm=q#@!{cxxvtzLYzIBeBe2oFqkZ(;t*T&g(HaCL$=%nj='
    '8G7C8IN4q8_cgQqRGRyA17|^0?=`svc4JiHakt|wnm6KMMieXVa-^*ouW>|;0f`&0Aj6%USc)$8p=hM(5WMMk4b9dx_2HeAJ08(F3=U3=u948y'
    'bxnIx#40_5FDfz$D0|Dx9~-j63?j5GFQF-{Z)W*ZIfP{^&ER`A9qoe?ubr@g&1og6e?r(a83=#(r1$Y_4h?rmy404F!4^>M3XH2)rszwndWHHH'
    'N8YMw4wS{cD-fn*D-5=zUa|WbWuaYT|D;!~*=p+`*rL1bR($zdXRq^sPNSjx6dWe;glHb${KUZyuVCW^9c9o87Vs5B!0&=jA71=@=M5+h(>&RE'
    'Jw${?X)-?up3NxUq0`zUBPH#ahZ-6jX_2rTO%hwH*j9!7_iLUN4_U5$E~hE-e@q&2JSFLoh!>PsB|47hOBs^J;^k_Fi&1=)3T((n5Va=f$!T(h'
    ';VE-^>t6Y13?-e}P|_-%nxZa-f#+1a513IOLvF}wi2rf?4Z2VGErnU|%SIGU<4F=lwDkV={`s>XqF1lqy!;fsd-m2$bIq$3GXr7%YVq8nVck->'
    't^t$NB-cvui`L?5L8U}~>$lJT9=(14;^iAQU`3xmxaG#SkfMmvc9g(}knVRLJ$y{HKYV;C<>Bb#%O79<y~x?o_;Y%&o&R%t@z3pf^3UxH`2Xae'
    '+vD*+w};Jt9{u0r<2eXw$H%kZ-h9Eo-+ejxGe3R*Pk#Pq@MkQR8)d#f;)*Btpj3WOK4G2w`^KOD>-hLnlYSd#Iuy~Mg?9b5kEgRm$D{{((_bwp'
    'TYDtP^mvMfH~VeFcMd}+Qo$I68I)y~i_y+ws>)_&5VmJC&P5>?K_HBTbccmJIYOr{DXT@UR{;eOkU;TySDeoiS&Gi$c|4$WQ*>eGETJ5Wc!Xty'
    '`h2U%{)Mqt7f_Jm24%sVZiK;;Dh^VdFAT)^oaAp*a-kcnQhrBW6g4y)(kaJl5w(g7L(R+zoUr4s@BV__#>3O_ahnoe1bK#;rpTm{p2M4<z4wDq'
    'h>>5Qbps(KQ#y)=_)zG=JX^w>{l64Pzd$~uXh=N47@sEEnO{R^#$;}Ic7e}ybM^&Fn*$qX49dv;&R=Mpc<BX)5_#0z=|&#v6Q5e-hAA^z4to0H'
    'G6Ad$8em_8J;W{*^({RBsj0!CR2iCnxBd&%!7$+8{||n{&f0F`-%iQn<=$T0){4`M!TLIhRcAC(b%b?KkRIaN2YE+q{TTcug2QlQ=oiONI`{59'
    ')@?gL691pQcVTbaND@W=3ReDh4LT4cQnqYshq>eUk<qU0_}H4s%qQuRAPGu{NP;0q$(pS9x1W0S8x8Ow$?@c9&e>Q58r{{^-PQH39y@YQpaX}`'
    '=)v_kox&oAJ-29?dV?JTL=&gM1P!lE1G?k>`7E4pSj>b<*A@a;!CWAp6aF)shCQq&Ry2fJZes$1q<)%Y=&B?w67(~KQ#Pn38>G2I4d(T@myZJa'
    'gxtpPbm~~{pWz2?V-?=ZvE<4blRqzlQ2xQmVlR_fh8uHRMB|I?*YNB=9vOMdxpZK~><1k@;_w)uQLHzqNbxxUN-uy3JTTS$&{doa24SeWi0Ffc'
    'nYY5|9sjXbM_|%1l&Whv^jy_G9KQrUYd%}_F_!~J^=ah>*_A0$505Ee=rSDzKj5DqVDRnN6Omm)F2N@Z>?cj=j~|X&hK7qHAC=bWJVESZXgdQ)'
    'A@1mB7qAY5N!&D<&7IPO0i|U`d{RumAPZd80F{ig6fXL;Ux}#Hj0Ez8QxEh~U^*ru_KyQOVWVI)WGEvQ3513SW>soFN5xmG-m@3t#Q4NiyxFZv'
    'nm5jFxAT7-?xW%9EJXd#Xp|#jMgwSbe6ppac$E_G`1#Y_r%}*+_VoFw7bfT?Sf!s<^La~mBvnT`i)LokPvaN=(3ycq&;*&r+9EO{I6^gUg0IE;'
    's1+r>uE}T5Fb`mC<?oyu-Bo2SHBvhX#jz^UJ<gdsVS@%7T#>0X#x$L?LZ>$3WSrtw!-0+5rz$es%qIlgcQWH-Ta*z(c3crPs@dBh?850RWoi46'
    'U~<dc$;3vfGK~;wg8^QlA8zgTkz)8rAwD<F5s}P}M1HVcEW}A>-jglQDCh=x#ySP6AkUd$RK5Ua0^sdmkKSxQqlk7VzLQ$StRB-@TF~SZ^Rc8v'
    'PkTEemF!M2r|Mq?XJIwJZg^M7OsFuqV7=@rlm65Hz>S*^=jrG9|9tP6Ix?2fAth3wPUGN(;M!!)6LHl$O9?y}zUV1ef2NWNW<BC`LCT`l<2v9@'
    '$hS!9MyJY{T`w;xXXgXnBA9kCd>p*T2=?i81RNu>&1|?a&!M@IY6jj2t;p$?VKT4S!R-#novDoOB4#<FA1PafcwduEAhXR)LxYkrr%{;64r&Yn'
    'z6=Zw{2aHmEG>h|$0O(%4GOXXU4uLn*LTOl840CIq3KZK)w!Ko84(o>7BlhGv-c#K7H&hO%_6hf%r%!#WNIDd*D~)f`6x1wW+`||32U2JZWCS7'
    '$SIi*#vA(l<G{xJE2ey8<dBnD@`Cb7<r1ybr9Gl0W-`4P23>2fQqUV^@KOh?OJ?gy(ACFK{uP#XdLw8%r1qiR|7jfjgm}5iiZr6}eOPacbTVrU'
    'tP3+^nxmi#13z?>gAf1bVH7;X&+y;joPK?#f9TEk4^O)SM<94pGHhk27wL6n9jyXxovz+&+EPBw@w3>Ml|f&_1!<;`_b;iXxGDQXF|#C_+T0V?'
    'QgotMDx3~0U6?dHk@u31+-tjS+{%&XhHIBrCGy-9ZO8LPZM53Y75S}gwcHXK)gSj#Be&AhfoTdM4KSfmQx?JgsmxB??|%vkD2_hqcSO7okMFxu'
    'OIAv)It^8;C>WuPITerxGwYY=T^Nd4<ANEamDi@@Ms7rV8Y4iLp2!Ccsn@=lt~8JI&gYRXokv<b56!^U%|o@ivw0XJRLn!Q@XkXsFm>}#t?q0d'
    '#t0SjP%SF#l%x{KrvR-%!dPgB3ls<h=>qXGN(w>c&Z$uKfv2P`%larmqrN}Jtoxz)jq6)2ZR+jdx66azf(ZXb#E$u9hjBJf$3@uq;-&~;38*ba'
    'As{0n?;DRNpJx+KO)KgF#gC2#yR0$ywO<Oh*{cWk-Zbq9kq)k+Q{Q5kS4Y)qh&JBkK#(%nc39jAph_oj1Cuk2^^wnSjXr79KQV37&=aanc@VIC'
    '`Ys7PqA@Cka0<XtJRpUAJ~;qbrNV}q9aID*VGY>H!c${OBTFh#IfjGB#A_#PG)l_^AI*}<1x1W0RjKMJ68hZ{5`XoyVfFJMc>mMU>jNXX7kcQN'
    '%}_!X>Zl$m@DTyTE}f-l%Og7o8yl2w+ibydn&3WDD~dHti)qX}4_v||k0@VFk)Bd1X6W`+YI;<*!S0C9uv7-11Bk_cLMQ$637ccgQHW6&Zh0W('
    'Bp2j%gcwa`SeyfMO~mF0whriAq#C9CpfC*(6UGs${6^{gVIgE;^oP+5n_3JR%k$fpQfi}V!u~oj`Yym;4@*`%*VfM*1nR3{m!S(h*RWu|1X(#I'
    'nwu7c{f&b|*{8kJiP`S?MYAM=cMZDk*j@BmU|B39*&oR{;cb!!Ddm!@uhw1)Tydy5vyJ?Z!;EWwP%|3N*Q!NY(LytY`ciQ`cNA}`uTRI&QO7dY'
    'y>u&H`?yKpL+3k(!L(5PXH2}1&gR#et=A4F9Cm@u-`Qkdj=n%TVctio(~a#d*6AYai(z?`78rU|6%j&Y%>)XaWau|>#{xl}JAxRfriK-Mh~O9c'
    '%(0kBb(cX5{IPlZ{V{!GUW{l~Xa_Q7GY;YyY~Zb9wDLF!(c6l^K<1H*A<t1ai-*{?^JSFWz8#D+WbvTK2otlmN6EN1NN9`$=N2Mzg)bs@+zL_r'
    'MC)Xu%-xCZh3e7n$8&l1;KsRd?tTDij?@T9GGdK;bb~CfIs&Q3jt5kx&7n3fvfxj25=@hzr7f9VSKRKu2oqeKO$O=bPym>D7>o3CA2ajl^hgyw'
    'O88^AGLa`BJ6=YPYQE+kEY-xE*9`DW)d+~SU)<mUv84R%De_A+bygCa*raL&#7Iz6G+I-0Uc+S|3ciT@{!V;U3Fmk1ptTt)#5iI)bptdjcN-sJ'
    '`M~Kzw2^O1XG7x)aMN7vG*sp<svsH;EGJv=|KnfsY~nU3kt?yuOO%=)-YWT(cc?*x)gwuYVdFMRg{Nf~=utw}^w^DFLSmz0GELf`O2UwTL>dp_'
    'FawtvN=B|cq&9Z6X?R9sc`ZsZ({b-qyJE2MfZv2#0&+MhMD4p|T2T6B^oUW0E5~^yVrLlS(`>bpG%x<};Wft4B%C5V;4DrhIb5SjQV8{gL07*F'
    'DlV=rA-2%gYLp|!b=SAOYlOW@WS9jpS0<UI4di)Sb*=NJdC4D8UM;rfpb~=oKVaIjY>%>@mA0CdHdIK<cvDJ^hWnfzDBiQtxGVHz$UT`_6AL}Y'
    'TY(<B)#}hQ;U<_RaGru7!Z@;!whpd@{(*49W8X3+tersm5I#H_8&xgL4O7(a7-byHhw=c1B1n+2g9sU=Rb+~+Tqao&LzY?~9Y|)g<T^YqX>gAF'
    'OlSH;XQ2IQLm*cZnn+4shQ_IUX7OXwVjz2|vKPo4&!(v22}(K$&|oIB3$Cb~_XO9L)%Do0BNPd%4(jUfQC@=OvDEn8+OE5nwT~oZ#J^Y%B9$Xj'
    ';7MDMl?#RI8)(}*)e42pj#vEEc{)oumMT*<?WBPZ#-h1s$8W<0$UR{~3@@!<)v{GAUEA~Q&cx*!l(%c&t|36*5^cfo(2n+|Fj6VlF9ao_r(7^3'
    '2l)c`TSSg9<d4n@7-Pf3YZ(wu+mn3l>FGg98q6_dVtyWH1@s@<163%H==g^qx&2MMgJ2&)_?!Lxmxf`IP6GA3#JKRT^ZFOH6@(tX$sR5WLI)=x'
    '2b^qe1B(HZbzhNT04XPW$Cj+7$F>i@KhrvCOn+}Mttf)iwG135TUSKd{hIZmt4)uuX4yOq$*M6}jHiX8dBIzFGH<s~B|4)Z92rjvXAEdmCn@?_'
    'CJZ_>XN)j#lJ^peV9DN&Jzyo+2~t;V=7(YyhIT&Kjaw2D*L<Ma&`8off)I%`S?@zf$sr6KmNw)94J4Px%~PMkfD(>km;cr_N2m}(Z5?h!>Nlar'
    '{lrDN45ux)0)Ep-{Bl5SK0Q!b2y`6X1Vre?JiAQes;sQc(x{ZLqSCtXWHo#pr5$@VTTB6E|5nvsl`t*#gn^xMLuYZ7ys8(_-g5pySZXg_hvuQt'
    '_%m4@SveB+jCEHTa%iM37IPa22F-}9ik0yb=JEc`OWfupRF=RBxmy(vxKaRGXP6Z8Z#xQYwZO5>Lc~`4j*uij_W0($f!C|Yl~KF6ycE^`*xytf'
    '8n;sC*=VT!s8pA?5VYox7eJJU6-%)!Zs24K*Au9M6F?Azy`+D^)s2iz{;&uhY3}<`Ae?p^-gpBZ?LO#Wy`Td=K&u2EV9P&v@WJ}A0f>}M;{#-n'
    '4hIQ*o%n#-8nqk73ar|vKW@M#XKQqMtSN(b^gn}Rvgn}!-^@?HqnkDY9)>fppm)SMrQ#>T8IbZ6dC`U^v5;C`jQ_mm5KED>2*5%2BAeg(gNF(%'
    '_0zr&CisSf3BJAqGYR0M{Hmi*JQKQhMI79^zRs@uu|cWaS2odc-+n`G&$Uj<Kc~vwIKlNrR+);}8e|`jwMnw^Y?F|Ps5d)9$7XzyI7&|%LY6Sd'
    'RQA`we+D@*FkIu+DukeQq4V=uzBoH4np;>7T^!zp+tp&m-cTw}Dk{wzJusYO4Zmfamf_rc?_+)MV}0*qy_%2p63;$^o_fPbT*~SXEAbFn`{&x;'
    'd|$__@GeXatmr9`xo^StKy+CZA<+yJ=RCv_Lsw^xqqUu3Jr+oEtfB5*eOQs5jk4*E+<tW$up8^UJPNF%e|&h|dHI*uFaP`S{f~b2m8Coq$3x=v'
    'Rf=9mz+PPg8~Wlh$wtV8itW;QoBZ>dYtM=&S<#U)JZznQ+DV*GGTf=N{<-hPWrm`N^+sgY=rC{{R;<8PYwDWh{VQyAgl<&q9Gmv<PxM$68a6o&'
    '`RWLmk($?kBiNh<r%FksbHG~wh>t(;+k6I$p;T3Ae?4Ikt43XDIiOMn!}{gHpaJ2_SKshJ=I=n>$ne!F;rm$W+-xkLGB+$xs1+ioyl%cpx~u$c'
    '%q|-zXL!6z%V^E;5L=n6Sv5&#kAE9;lqAWSTWPOTK?d(BaEeP*9;E8Ge2%K$zsYBW<d?{=Q_3Z(5v@)f?<+vGMc2fnS>NrW2M%>3^{7D+BKNVT'
    '2BlXfC5WulMm&6J56Q7zS|F}j+i`b2VzAD3G8}wY+ERL3c^o-CmIdErOEPZNWNV4S?>x&2(qPZjp$to=TSp##=h-?$!gFx+QScwu$k&vM(fHoD'
    '1#5h56ixBIHE2q?R*o@AF_q$4xpu;zIAKhfb1aQpNE8jmMj5@L;B1ulKwt?`1K%O8zSF@Qy-aMQfO{o>pLU1-A+wo}GxX7`JLM8t7sj{~nJ+9D'
    'N@+PU6Q_r@q*4ujmeQV+#bAC=))c`_-A$TIY0Pu!dFYNW&z@CDLYhn{{?MF@s24lWK^{S4MY%yf43glu>ug+17ja0sYEo^&_blShsTge{EOxvk'
    'WSW;@u}Qz53kyiDb8ie$E|aMp3R1LE0Y)R4NdhD$!y>(~AF<Gc9F4Lcifnfl;AASC>&zSl2UcEZ>1aS6p<|#WGv^}$DpA%S3v}^}XC=&3XL)`$'
    'O2yXSM%CQ*6cxG6{q0uk@o81H-TKcO_GIPherc~u?-zJqNGq3N<g&Iy_f)z%q5@DZmU32C=#@!%gO4d2LWDe<_i0Ud%<3^JaxK+ZwWtVIx%!2z'
    'WZ!G;?(Vv`$=~cs4)(h6lCc0Byuxji^0uEW#yws{ty33vUs`d2Av3FB@7mFboXbii1y{$wyGLfv!5~@Oi}I+*V^YrIy~@O&?mu&<{VE%?aVr^R'
    '^J_V4xjDI<Dy=m>p62;TK8;Pcuz3n3X@IA6#TD3n+FA_~G$QEUPqu<Ei_<s?hG{z37G;|nfhq!vpB!l-DJK%!m3IWr#O#&<+bn_lUah()d?<Qa'
    '831hw!6fH513~o_a9?Btf}6~Eu?BjuZiTPX@G=PZAN%Z7&|5ZMjJ0J<q{$S1XqcUd=){;VZ#1DQdV+*D4vye1<-EOQgJg@InaviX*zIwDSrl_+'
    'xCk-yJPMvPTj(9pMD76oZS6jO)+nE$q(2u1I&9RHso(3(b!X{l(V1Q|vkJGzd4<Vtl25irRBjJ$t9f=QXrCA(2-ROOv=L-ro<=-|d<3Zg54VaW'
    '5HPO8_)%Itet~}%{5N$Y0iUE-_y~Ufy2$z$oio`6h_1?k6q|*<ux<n@r#tBscLfED*6rMMdw1V!Go2RMC>KV|-nFPZg!@#gz7xH7O*4@`JA+w1'
    'MfaEGQ0&ILJ{V*Nbor^APWs$n7J;Of0X@nrOO;X-1HoQ?c6}pf14-b7?N-OF+4gy#y?m8XA3uNW$GaX`+0(c4;kIqNu7x9A7mla(aDXJXA_Uxv'
    'TSI9bKc|x7!j+v&_adXPCGmr$ubCUHXS?8-UXL91R}W$HiS05Y-dGVJ0-im>FmuQN<}XTdHgrASn?NedNI64>R&xj17k!}(Fj&GB5y8AxUkE?m'
    '3PbF4fQdRO%8F;zK~9YeOzM{FI1M2G(k2cu@-kU%V4|WL-lvVcO;p2gw43g0ha0dPR9wjI{l}5-0&9D3q^OF5w}C$IM>WK?4Wu(NZS*AcF|#Xt'
    ';JPp09)8e?aUT{cs0nWxYJvd*NhUU)+1sZy4VJlS8OL~Mx)_aiT92PT-FuJ~#R7}Ap&+r{+}~@po;`iC_smbF+s(*NNZUjxBR_ScmrqyoyeL^v'
    't>Midkqv<PBeI82tyjuHy_BQIDgazr`PY;b+UAya+LhV)c6pmS-;}xPc6sM}zHu4hw(mf=Woj!})OOWbaf{~KJWSg;5`L^DRJKe1a9Wj_l6Ltl'
    'sXQ;F9$_y{qXw<xMcc<JL;ANUnoG)?eAdBerUOw2l#IT45u+32+w3AeHTDWxc~7=V>ZGj$Gf=aLJ6I)HhH!`)MxkfBiF&$Jf=YxoV0QTt<qzo?'
    'Z^K&1D>m~3N>HU#B-~vfi=>G<Qs`5OR|*jp%VKHjkPT2oCMJDdP~K8~Wmw2?CaDT)(=RmeQ5|!E#8W4Mi#<rdX(3ftNDa@E0=C7B7uSV}R4UeJ'
    '=KE8OK|)OHhqWoB{Mn;0l{#XPgSiu_rx^xxkN##3|4<L~@bpVb8|QrL3md|0RW`(5bioiKRh4jK#JmU=A3)BE>_(yVO{{6-Am{pQRl@540id%*'
    '+tS*=)+V%dp{4=r>uN&{Mg2PZP&N+zi)ut2;O}*!x1$ri*A3q525(U}h|u-VlgWe&g)dpgYN%8nVpL8FPZnfT%fcq*mG%rCWsi=5zn!H;K3b4?'
    'ilfu~sc%c8BkP(k$M^nvLs8fF%Eo(T<C@CG+fsH}t~h09fEDyhW)F9z^0N4?#>zl_Wwn<v=r^bM65c@@slI+k{g!#DwD%$(`ESE2MLt^HZtnjc'
    'O0adbK-Ef{KZ-iW;tJZ<dX`fu%S*OxtfD(#eeJi6O|Y55<t`LI_ez`<l{iAG`xezWf>_=wa{lm&oOM(<QU#-g?ORai_!T+A*V0-B%hzxN{f+if'
    'A{>j;XEGm>vvssOcc9R@SKZuDbz|t6LFB+)UK=0(DkFIgz1Q-gyB()|1ho!P#_L)hm=a7w`mw0wR{+Bl0xUqpB^9nZ@<n=08CTR%xkY)(mZVmd'
    'q3%|CT0?I7$`aF8l$Fd*f96^!6f<sYG-owvz8<#@t}5?|Y1`bv#O`J#mkt8M{JI`39g(ce=R)GN6j#G?*;|}^dz<kkOa#gz5JhMJ64_%l))8eZ'
    'q*tA+O$^967Hlddp81lwttOjcIS7OH!j+9Zx+%kr7F;#P!@ePv2Zpt4G^TosUF@(M+Y3)#z$IeL^g_B*Tw=)q<ds~2Z944-*>1BGXsfvec2)tl'
    'MZ8x5oTMxplcl};=#kOD&1~;Z$+MduJ+c~=$JfMR)&a%?I6CO4$sr(1#60f_KyknT`M^UEtWOTSS~}nMow@m?4=wJo#Jh&bhy%2<>@poN(<&%@'
    ';rg^Ygca#dVOkXpZ=`bb*u?2aV%6O$ATDeN!HY>CvllSC-1qQpZfb77Oc0tB!T39#=PW5SToPgZGMgC$p&|x}GKx!2v%GUY{0cEhA|p9&l(&Z;'
    'IFsiz&n7Yl8J@u$hR%R5!bk&jfXRiw<X=C&J@7eL@ddw)K?%r4@AMR<-kHzRbO#fZos?so06EiSR;21dKOc=!;=E+^6KV!VSx*Hr!B3-u$znXc'
    '#vHJdsd_d;$IoJ3s0XC96WtjyK?i`B-JTEeS7(r4O;C{_8guU|B1%2~aQId}rBFO?I945(X9q4?ykF#otVZRq&UItAFk}mLIk4<KT+Qnak>$o('
    'kzIzu26n36H99N61CjMdZzvh5U%~isUdbKGSGO|(d9aw_&^xb$095So9^JW0d!wwAPCC8jeg{76?C(8oJ%0ME6;G!BG}2aj$9b~9;W$tBJNvs&'
    'c6ayocUKr^@7ad)fe$<T&z?Me_Uy^y6~=keY;86Ue2C*b+kN(Y&l=}}DBf4SaP<CW7#(ShK50FD()7>N9%v(!jwDL=_V*q?dG^Ga>KGV^45{z}'
    'sGFYgk{z_2HJ?q^gatp*8)?7Ue7d*y*ny_VAMfstYcS6Z1SpR;!wC05NO|(?>Hd?)#y<N}KH8ST{I?a3ur{aEo<R6#l>-F-tg`=~D63EXF7=G+'
    '-DJy#H8+8$Pyae`ry|d-tZ^r<duA6xjeE(Qd4(h`M_JCfEg(<3yR2en=uo+=`{Zm*<bE#0v5m#8TK8l9;#8)(Fib7YeI8Jt#y@8Bu}*1Z20)rx'
    'Hgmo<_g!zb@qSTj4U)W>!Om4WIjgiHCtJ7!-04+B^;3HeAFKYr1BPw3!-%rl=05A)<K5@`wIBJ+cT2|cTko3XaqLA{VK&Y>uR0(5vlVAl{-}CR'
    'VhQ;m-)Ie+Pg?7*;ok1!_1Ey}S6D;eY*tu9-+WeB!>2b~Lo2xkXY|mO6~mJ)R##dkCPR%Nr9h=tW2)W9#N26T#_7ng)P+z@DXWiuW0a745Ddrj'
    '10Z}uO$u$#R6VZxUX~mTk-pB7aj}z)&vsPs!OlfGn@}2`Kv^AGZK>lE1SwPxVphoN%4=&3*UOS(y}|g4m4^cHQh&T;q*5q`H8YV92+M#|y*{dU'
    'z1q&Ri_MlH9$Fw9UX43qA$y~|f6@6Z`?B*EJA98W0J}RyKN(Sl=Ak<&;R`AJ!{9r^nV?x(uUiI7uIZXk*$aECFf}(sRqJk1)!YnKt=mV{yV3Lq'
    '`mww73Y9$YuO<X505={@<6-|!c9sXUHLxZDyL*f^H^Erzt6;3TKF021kD+Az7r0Sh<?|1-e6Z-xU#OcF_uKV)7({(I-qn-%0uy=tawn)<4)YqQ'
    'EkXRXaZHAdO|g9WdtsTO-U!c^zYm^&U8J*(ck{2m7oHjFjqv>I?}O)jG2ax+@agY`XM}nqEGw8bo>%08J>U{6q4^>Q4&a67%)DZ=^*Q{?0V^1O'
    'XS}s1`0m9AL^xm<;}U!`LRl^rFIS$1?$*n9`-Zpy()^=UV9Xh8O|eK1Rm=MM#;oG?ZZ>A!Ed5N}O~z}kG{@Gh&at_|99uV-<2t0$2Z2(1RMW=W'
    '1*Z-j9$;LW@W4t;xDut9f1dBWM&TYHaFhZ%4~wZ1VDIz|H61vbNB!U*=?2P&S~cG4ss}=jAl198D;~`0+_u_b4Umirb#;i=<yTgQr@0Y|T6c+}'
    '<|Zg=-8PDrDH`el+g#1CK3FSbYf}}&x*#>z$5iVpV5+$;rfz>vt*Kb32l5uw3O9#$Wt84nrEoJ?Jqm?Y;l2EQuv=0ltO@Ve#ji(^uqwR2{yz9E'
    'DG}C$_v_-fq(E2|+8d}3ZVvGV*xgKZ;03j$I51!sqElx_^AGDY%oW3FS)p#UHH0S|3ndLpC^GscDP(QvsYfrY)L&VI8KudL$}^|qX*x@&3=;Y?'
    'Q_B^`<Z(v%SvK#8C0PmBp7~)JEH*6LR%@d_ZMxPE!vRJ%=A%IrOwubEPa(w*AiF_3fF>a5OhahZFdSh3EV)0~F#<ZkR39K1`W{~R9{y@S<e*k5'
    'S*pio0(OE3{u|HI;yjtA;dV1x>ao!H8bIT=SsUaEhmnTma8zZvuN#XTfk8Y?!@VeIMZs<qG#iFb3w#H(XHZ}W1t3M888Ml%U<<>E+P;VgF=boO'
    'wwidMI<n{G8^jvm?l!_Iqu<qvI=B}HdC35(12~Sy&;u4nAs=>*lp?S65I$*`$2=Kekp8coi_Rsv#e~HieO5T0**@zJs5wqHn+&qcY=Ew?oHPly'
    'JNw@UW>N<rA9NBrym{>MlzJed9h>Q!Od-IysW(}2`tHao5~bkB1;A2-w*|6TP<!#jx-QZ?#+l9B-#*riIou}ROb*|JIbu-n7;`q;e*2g+=5U*s'
    'GkN!hk6=6if`hS=$$-Mi$12z#N9T+gREQqKV@Fs{VLr<~2j2ykRCrUAXHt+J*DHR3+!(9Gh3!gY%1F#J#mL$|bUuOq3y(|d9JypBDxacLnA_*9'
    'Xzw;Ge6I~N<8X6`@%zXoE0KF%Jh0;bqhMmnfN+`4vf*`*FJ?fH3(AZ@(Q@qXOcC#kIR+kMQ~*->=`4k9lXhU!sOn{l*@!}2F(~U`XGdhc>*v6m'
    'C-bC8=W#we!!y^~Z9d)E<^SzHf4cuX{$+Y5<NtWXHit9Cm|i=SVN{eS1xBG-z-cn3g2J%-lB<Trer7TfieVj&2<x2*^pOrk`WM>sQGk<}%#kT='
    'V?Y@6oKE4d9ep)sz;8=eO08PI0374B#r?$8$SUIomnHT}4&ZVK*xt}R{PzMcWaBxxsb4suuSON*%g78zi<4eIez%xoVYhd{u49fnCoGZRiQxbu'
    'EAN~CWh5cwDZ~FY<Sfjx+_o31?pjI-s*}rxy#_Q2wH2G~&%C`SdFN~fJV8VFv)X}&I{L(jMu1fk*{%j~EILCF2~wG-hc|AAgp)!Pn4T_*a|&Tx'
    '^hAZ}0~sQr!%Q)#ac7zUMt0^`X>!3H)(;DrG63ln9tRPcs7kxVc2|<vQMwY9S40IgmZRELNJVY@h3*(L4!N;GR>WIXFq|DZVRlCm22=~#U`%}>'
    '8>Jdl_zfNL6=HtO=q9f2yknKLj8!@}&qiB+xD>9D4A#ANBqd7cz;vp>Qhg-}UKgkp#Ua8r)%YyYcWB}}S@^ZFezSC(U$UUi0x*^M30O2PJ8W0`'
    'cWc^H%=(xghid^DExqPDi}_YxDn48T`s>9d-@^U0uoJ9eu1t)G#|sRoMmmrM-2<e;b6f^Fl~q`OXBo^;VqPenKEa!7qY<ZYhsfLye79mx9X*yY'
    'tT7zpHkj%vo&20*Rg=m501$AI>O$@$SPFWkJ^jZ?ae-`0--u1l@$BbmF?q;AzZs^&nS0`#mP8ndJthO*w$24Zb71s>%9yKcFh92+e=$c>b5t0#'
    '<9tKPF=%j(dd_V($T!r7v914%1f3vZji^P@#yZOMGaDdFBf2ibqQP*WZ9Bgs4Od(JWvr-VTj07`jK|j^(Jm)$hRd%|d`YLNI1>yP?++$17w({t'
    'aUxYPon*8<Nf$G?Vt24m9S|3|=0++1cRQQrgA$`c9oMibou-FbiN&o4y!Xc+e^6XMHgDY=&fX}QT$BNAEdjK(I-rDGu8WPCXmC{45hlSj<NEAK'
    'K-4RB|8Olp<caJZ09!|3Gm_Ara<U0ZdjNbIpTZ8DLHMWu4sA;MOi^MAeq(_QF2QC5(Wo7wwq=l<CCFA;N4HS-pm|o?7)@%{$Aw)_1l-6>I?X=A'
    '84yW85F|s{<@EJre~rE{P`AFny=Uvdh7q^*{XJ>6*69zLTm3BycA}lfgsm##3CBdH4s%l4IS@ij{eV?tPPUi@*qOuD-DWl)jT-{mw)`WaWD%5n'
    '2fh)~huXe!>8%+l`HKdWurzKBG=)aB6=UAQD?zIFZxxtjR!(R|z3d#4joDnB$5?|jhnv2bhm?J?Jh*x-hnvojQ$N{ymnM`OP#r45tVoV?fV1@C'
    'JRZY$O5SigLzQT$9|5ty7o~Z~=jj13@((eGAuI<nFO;>%3m@<`n5CO8^eeo^@e)5zfN4uVV~SL>Eswxy1Z{D~K(5H8CwcQw;jiuDI||H+w7p0&'
    '=2bK9gPV=)L94UKGC?3qTAe9KV6@4P5B8$qVDHpup7-=Ng-t)svq9zO4RX#qPYX_+O63;JY~dydn5vU{Ki)ls|9=oS|NRT6W@MegF6MH2_#53A'
    '@czp~b&fpWH<x_hl;5j^Wr>=0S$8Ai+(_H%%TQp;(&=Ylq)~DxM!<F}r1351NsfAayxoeQ@3-KZgg=`5Pw|>O-rkK{Pn)m|@bB~H?&)b%$~da!'
    '`BkB%lm$12b}w!=AMXRS&G`AV=Z^)nPxqQ=9E$h%o;|%4XgSBhE#UUq?(=7i+h@<40@}x|#{});=5Fg&pyjj?w*c+aXV0G#w5|R9XA0WqPvF^Z'
    'y!&+T$)@1?(pTIV-e&ydS?ejm_T<^F#P0Lv6WY1Wz5Tu00r%E6L-R>%kHBpKp5;M%{^U7=`}Enf$D4xdP0+CpyoAiw<Ht>cZh!x=gzo9?{w_hc'
    '_Y|Pp@Z8JXA~(h$Ebi|9V_IC;i~_{n)^kGOlf7Lp0!us!kCTi>{Sg|Lm)%2=18D@3dp^QE_>T>INM7S@o?jF}e(`TMsjt#E%AC+n5<<)b7FK=J'
    'h{qR$Y!;@UfpzNS7c5@6pupYqDY-;Og%uya>BXY7*1uJC|HU#Iu{NXt=EY3aUI|(?6k2FseQ@>7Isj;#Ypv~pJ0as8!YDXx6Gs(*toR|Nal?F^'
    'oMw!VN2i0S;yN956evF&YB6N4xcG%g!ux^VNR`xs;3KPm25cTk=gNA`UBn=@c4PwI(98w6y{$pB$~p+ea=V7NFj|v1fob=R6c*<>YWCVy>f)Gm'
    'jSaKtL^?uK$a|z0B4ZBk&Fg&ea1e|#Qk6*!eyjj!tUA@IU4eYQoG<4*Rr8TR;fa$LMWzVBhwCHkO~42zjtq&{D*yV*U|D>gh1P`ZFSrJn9=$U}'
    '+k(|VK=<=`3V_kk5&8%2Oer6%yoMQOZB|LPH^HOc<cK;4$~ll5eC)uMvvw0e#Ls1_cU(!>wv0o8-;>4Q*<K3&9W<W?m}SwxDE}eKEC@*HpAJ=M'
    'XegUUg^>qa#ipf)9#orq=(V*#@)M1NiWHZYY7w3bXud+0OV)j8ipp8`QIpTMi&z_zIGy4(n)GA}#aE;|f19Dw{wI+mJmSh4-eiw)X`)s}#vg!g'
    'xH%txKr-?sC6~fT7^eQo>htOW)Fah(Xs<S`#_X&YNz(wSuBEbB)=_~mMESc8=Z%LJjp{M2nTX*I-@W+p_50UHoli$E-oJYB@m1&D>lg2%s+XUR'
    'UWs=v83a{;Pkw?U9ilue9(*`_OV2*01twU>T47R~IfRW_GZSi5u(RcW*%a{<?F^j}F$6<?QE1$d@gEM*EV+)9`%5;k+tDL5a-b!im$IYFCsbh3'
    '!ru|GZbv{`NlgWpR0q8jCpo&5oK0vqh*B#NPD9u%lFK9;q3Jm)XZKt&e<vBYOZ_QbhRm;e__kd0!ZBl%ceA(p#mjuc2Hb+vkfUujvrlDJwSocc'
    ';#5@6>*e{Vu8JLUcBP_q#$_NnqQZbU`okg!&xL;pS#%nK+C@dVIz|`2?#97OF6NNKDVuR2#@@A4u}(qVmBsJ6dY^OyPWo(AArKQpTEpdffH<Fa'
    't7mGNrZDNEX_JEAUrK2BWF+2W)F(sOa*Pw)F5YR0Ij=T}86QxbNcqgxt($HY|8V&7uke2*@O2Y$R`A0`ks{|Rr=aInH5m!viWA~hMCx%nn~!?Z'
    '3m~cm>J7#Xx}v2>8s4p%jD`-U<7AOvWwDGwL}^N{7rCSeVYj){dh(>a!3cQKowYddV%60#=j~e~dr04c$*PODv1GchnngqHH|I5<!+(?60LHk?'
    'n3r6URH$)Q+?RYA)4;N3B&{4WDaZ8tv~JqRApMgHfc|q@!T=cn^a@JooHdj+nv#G<L;&zYMH)j7xV{ma1SR2_$U?17IW4zpYt1ep>}UjZwTQP4'
    '?uz-~PEhx$v<L14k@ldI>hia%Ari2$HXi9(zk`0^I7kK-xYt`a#G*Ij2*bmVKmOo#BrEw;u$A4k%NXYL)hb{rUf>|Wxj>KPCTCy%2Z4R}Ln{)!'
    '%eU)+hara6<%jYvjeHkc6)ZLlmK6j6qKG#m0@e(60=NMpKi#eBS)bVhB}W}|_tZ1XSmF$xdJv%Y)P~>NJ4=D-v~nwOVu3_Q-B?%c#bW5%{0|&`'
    '{#Pq6+2eF67w0AUZS#lnBj8`|2!7$rM-AJ?-`Y6=e=P44WV%xI6V11`^$Bp=7)Wy7C|0tq_Pw`%;yK#JzApD;aye~hf0z5zSAN^t)!s^?rRDI~'
    '8?~`p9@C_EZD*URVlhi3M5*eLzTr$?tBDfJvIk{p+fT8aLvkw=t9(3N+Enh(qD_2V`Z_8TBQGua9f^_cPgUxKu&NaVUUHVq6QSxbK6jNW_@VnO'
    'eslQo(~(t)8SguPhXy>QzBeMWL{VWw#Ow@5|2-(z<59L9wV91@h*{7>l^L=;BFe#}KaK`LZ+GA(^nNe#3`+S#N?A2*9K0rjmJuvd9-m!9EUy;K'
    '`=hGR?qZ*CXJWzo@~D_Ro-~e=>h5~!{3-<wYnNmV^lD7!rVPlQ_FacVTSt3hlWi}0YIG^iz<!0sve7FCUaGyYKQS~BDAq0;R;dMiL$9lOAa-8V'
    '`@}m_K%D}Y9{BVkaq!b*bX^|zJfR(*4u>gMH@1{Y1+Bm0f4$4TqT<52PFGlXTaQu3y{%YSs!#CT)Kx3t=oq{@6^$yP8(E60b#w+LMqwGgJW7Pv'
    'L!34j=N*QmhOUhb!G1^vC<aX^D{vqBx1Fh=sK@5b#A};ts+mbqDQDk!UgaUZ6SXgJm@QB_o~ypLYYW=q|Io-q6)KTPh@NO$W8YeCCo1Ckd>h0w'
    '0Qx}G#)At=Dh|4nX^aS(CD$(C{GkD5eDwPLr=LE)dHd7PJb{Pv;lr-csk-Y3)b4y3^T*43{_u~ZpFY0)i|BSbukDunOjhmr{a@dG_=o6sGOg=('
    '9ZdY!i@%FLe@Q;qbqcRzG80wxZ7@@?H(bt%2f-ava~S+=TeoT-3EhvjU<6`#JQ0SS-q8tF@2&31meeP8hS{tDDV5=_089kKAQuy2ZbSJon!YU?'
    '3&RyJ8*`M@4EgqjGom#{6qo?ijwnvhUkg!gy$Xwx`Iu-NI+V|4qE}w`jaSqJ1{lj=fpdxK=YhILw42V-L3vbF^wH}1eygYvLG(^52YUb0``6V2'
    'Rdl_!zSE;MSjK`ZQGxozF0c9quTA5Js~EBch&6R#ivKTtXZb^f9Dzql<&FyM_-AVPETQ8=@#`X)37M$ju~2HYl3?S--(IN78mJ28lg1DKszH#g'
    't6HL_(snfcqzY26cB@9QC15swS3&L3g>9&nbU4<VsUvNkvC@#>D%QljttVA8_vqscr9#{1O(Npz+;ZC3zv)aOn>pvV9KcmN-X~j!6Y5ExXyjjl'
    'cMIVEdZ}Vm96d`ESWFA*n|(c-tQcG;lHWtBM~n}D9lIYltXxLb-Z5d7u-evtMs-4E+x{6<3v_MsM^q(twe_D-g*ekze?=7<mbUsUDyu@;p68OR'
    'wB;YhdjpjZf)}Hab|WI0rVvinvgk1`M-h8==CSS06pEN1N=_}WDQ<k~>{LgQ=T)ac|F*N)8vQ$dS>hlZ@1AlpGD92eEhc}J%qHktmt%Sk@9Dte'
    '$LL&6ezMp3g5#TJ*QBn|s)J;NQS;aG_dqHXk)KkWkpGxSG*GPCCdxX5Be~48f$b$RW<OQm`La=1;y8hVIkqa^K|iPH%@3!K!>y9h$by|58qm$S'
    'ifdfT>b93S;~C3a-e<zrV2@$(PJ(+l*7mZ~NInq;v(u^_T3$IGk3r&d&7HL!hAr|O(em8er6&y!V_iNFcB4}-T0Hx|Zek`6;7pHpokKRi;#{;r'
    'X3g2C5AFH5R6DRq^w*hL1?)B6wV?pt07JrReByz%9yALqkv-)%*K0utCdC)m2xYk+Q~H!P{wER|DUOY{)m;@X-tza?bM2Nv2ke|GqE)m@hwx+_'
    'V^D+qA#RRB_~+tO$-y;Mj=AcQ*B|`F<l1xuKUxr3Q2yYemAIIJqnxT|#8%nA*dsVL6AbZ?G%MNH#C#f0Cd!xm<?Elb`T1LN#DAL?AV0sH!9TAj'
    '^BJI=pS=6@8vk&*h|zUH_R|5X&|i#@UmGS9%pe2T7X6vS-RE_3#~_DG!HY=5ygX{XdGBki_66oQru;3oCwWI^GkZMf*m+)Rdy~=DhG#wHsriYX'
    'NA3ee)dx<c6R{fkB`z}UE^bjfng#KPS$f2=Rqr@u&Jtg5!J?SpiO`J!<;?*OYLNGf`3!ev%`4=y*hdM5ft@_iUhYGf%sFGYWjKlZ-k(t-o>>PD'
    'cls-)hiY}2yP~D=9#>%l(-9oejyV1$7&%S^-Gj8pdu(Oh60|@f!YpJS$xJut=ehmlZ{(W((x|=59HW1Z%5lAB_0*SQbJ_dPeJhZ=repns;Ky{5'
    '&Jy6CB%V$totHnoe{=X_=gm(aJFk9vNeMknBp{*cy!^}Sm;e3Yr^EM0R=cx2KO3bxl@Vt;#U4f(P5R=R?Syk7x~aP{3V@#Xbh3h%i+S)L0i|DK'
    'tVu$e4~H40mKx)`S+D?v0JAuMNIr#0FvNVwvdt_Vr3n(w?cn3jvg>!`(H9`J1I!F0bLmXe94KX6^y6W%h||Gh=YN0cEoK*h=lKp0fI>!K6+1MU'
    '1Vc!9Lh*Dk)b1_#tkWM1!_TJJjJM1f&J{qTj6?i&&|i#^xMi1^WDM7(w;+qr5Ju67;fxD0Y{t96cXVyU=?n%6cKAo}>uesv1=MIX&<BohBAH~G'
    'bRB;J9Bl`!;1OcIff6%6|32tJkMg+zx4e!>qH!ddlP#$~8RwLo0e-Zw175RZT+>s^RLt}|wst{WlI~Bo@ana2kPoHIGES~hbC6B?`3z(CVk@#v'
    'q+aV^jMB?=)Na|bs$BcOaaSm5ozYRiV7A_Mupsw_Bv1<r`zpgTY6Anyrs_qVgZRXsu@nYUW|0OjU%gS<Z?U>&53&6GN07G`(<y9V6u?IL75oKT'
    '#>ygsekp1%g}7{pZCqGfB9RixYSSyVk>mZ9r!%@nAAynJoY;#)pqEz+7>31Z#Iqwpg_o`}Iz9-FTO&+2+gfZlPh*+v3Hyo#!a5=Dp%DOr&EU@k'
    '{%Jx|Xl&t(6*)94qc+w-2*s>~YHegs&M~r}fl~p4bV6fbRYs8AXL3ZoqCeKha2h)y?&3%8sj`~k{YQt#7zA$Y@e~JuCJV~(_E0%Hi!rI*meo-o'
    'KZ35ZNgGbgIQiV}8TqoUGczAbM_&H+Ah2~|>@g-C&vxTonw@^q#3zaVjod06QvHBv2Dw0)9b5S^-@pzk3@gl0c{2@W$<>6nRN^k2?^qDaeW?#8'
    'Tfu=K!iF{aQl90$ZE}EXb<jP5|KVT2GXD1dOL%gE=~{k6ynKlQ@e5G$<^OLrx<<AhF*vYnMK(xfXH3DmKzA7mzHI=EYXTF$l#wlnk6yl*3<{nN'
    'JC38te?U81&>KlGU>6KvMq=2je0CvDeGl_CUg7x@FeCv^X_WpNm~&ANdtK#Q_Wq}%*RUbbzXwg^(D^J&Ca7;kElLqkf&?<S@RGrrUje1~uNQx('
    '7-JljQ|t&xXf~3H<9sn)8-|n+c*^Ujg4Qu!Nf72;*lyz9BF=F#U?fl~VRCI)^a@NkH<`nV3q=>jtej^<L8B?YT*tBuIIg!MDo>scXg=9iQZ7Jf'
    'oAHf20(5ZH7&0EDJS{W5_uxRLtahC}R{1^=<H5~1UW_q;GB7T@BXpylXq0@eXk;43mgJqPV8XBn1Ca&fz+4n6V%Tu#9TO+sV!(h4^`P#RH9FKW'
    'GP&)OEf(F#5N6;56QeQxRvDGKVH5}f52#YD04RpkRTOYMPSGHTG~+eX37osO7{o|-XMcr)Wz}aSbXI(qT%LKq3wb=?u|yBVpvm1U(TI9P()#Z3'
    'hX(s*9W5Hp<wC7-AD~ecHzpsi`2hAN?n;;U)Ep0xg>btCm$tbF+`C7uE*xilH%jnvPP_55r_c9WPxkklyL*pYd(WRfGb?<Qr{Mv9$v!rAr`*dG'
    '@YIhiSQcw&B6?&plb!7W8=yMe1V+UfZByqFkI`5yGvW|arm`K)n!s=7FP)?W&*UiWFDA|@*R^x1&g-&6THbBOkkYaZa_VQUTtcBIw>G#l<ev=<'
    '7-{eDQmPQH3t+FZsTQ>9*UPtuA6SMrG#<R9_-gxmmVoWh^3n&P@wYF9IO=zdWDXq9uJVXW?U1l|6yydm$NV2ck*X`Y%IicXS{6Uit0dqe51e+o'
    '@;YSAGj)irfV^$rAj>B%LLv=Vq9N#p<Vq$o2{VwW^lg~Y-a-(StN^D2A<vy`o{>6yz}W2Ke>=KVOWPX_cclD2%sKbwq`<<v9~qBp$!oZUUt}OW'
    'VuvG80A#}=glLk5Y&n(pIv8NK1U4-PI7We4RihkVDOzK1+;E#$CP?O{lbgAMJP9*{jsd(0H!U5ibHft&4aXzLUZ1X5MyoSk3S2dG??+w8OtcnR'
    'r>j3ItseqKZk-;yhUl)~`1nbmbG6zLUU52ukaUKX?-@r?@SQ!bzvndRpYsQe%@)-P*1-S$#ac-#*HLL%_2|KLv|u?&E{r@M#>8a0GJ`FYTD}`E'
    'x9(}xwRXSQY@t_J0IDsWCYWc`T?<F2W54w*&I^q%S(<@kFA(!ONqvzI=fFirERm7Z8L8u_8Gnc}$0Z19Y{VkYJ{=9WRUfVa7{K=i>6+Bmh*T<}'
    'Ook^3b&pG?q-<||VSQj7Ok*5cr1EG{5XbSUJ*Hmo^}oNhix}1GYh1%l^EEGFs`I|W6`X8-*;1{wZ~Cs~V)yq~?fq4If7RY!wg1XjtwC$=Z`!3d'
    '?P?e7YWJ#h>5cMY`EA#XmYgPG#>dX+2>Fwzyo5`725X20QyNz}A8pgkR%GLBl+3dEwRtr;`GA)K%Uqm!l4!H(2MzDtDwNlQoGO%JSwwi0&kU!_'
    'uE`&ENuAv__HkE)=%QmX8J!Cq7eVuLy?5ng#W?;s6JnN4*_oiMwQ>tnHs{APwvgbTGJYf~Wvo2qn=@Z`$T1C!%j>x7u(RFM_*NA*cd2FZO~g=-'
    'SyR!4%LTs(^QrcH@<MI1GWD1<OKRTofW8^p*AY^qHSQsYBrUs!oa4N#xpg<zH?<|!6}mOidr;%ZW_{^dnw7rQt~K+#!)zLI3uL9{EA++W`t+dW'
    'Cc{l#CTBD}zBol5_(E2|Ip(6;7GPYj$^$W$Vx{1lvlQ5Y`HVn`f{RA1fO9-|sT?SdLk`27M%(dD@hO^Lt3w)2CNZFBF;aypt;Ajo0CSfI9tX8X'
    'U23R?lkP{ApX>|gHtQ{`_JO$QN_7T1@6Rq|)wOdg*{Q};MZ<Js{j8L|$tgaMUNDmwrhC~Mgl;T=3Z86>eyxa3N6gWl&PCgvveS|)GIuz5#-2gR'
    'EoqhwmNn9a%E;Jb`8G95)5jT5SLELvy6E27ago{x9z8;HW4RYmZ=MQ2J6*dtFO{Zew5yF8mV1qyebXrAzdFm%kFV2P42LNe0#l<<jlf8jcOb7B'
    '&j5`${X9*u(kvFcXf_&A06V<EPA78}U#U+`Hi6NmIhm57lMV<xxq>}1ZbsL$hC?q70!lSU@fY-Wm8*;|W|epx;TMvOM)?&5%o-4hix=2D_T3HS'
    'Gy@4{fe1l2IP@!(DJ8UsjP8s8CMTZBM~=gRj2trIz`TyA_(}%JR9JyU?KLs0Uov#PO8Vp=H$l6x(57ir*~yJWM<|;ZxY$;HmT-q*l6b6FI86F!'
    'FV8QK=}Ta+9q2CK>5cN<&LDlBrh9|oet)?49NIm9(n@+yo;-W}_-Qh14wL5I{*%4^R<a{Vv>^q>)9VLsn@=BazimE$9BHS$^}x7nF(7HP^%RC|'
    '^^)h0pYIQQ$zb?2N&EY0FL|<`J{vrH)*3X^y(jV`mcXjPdcOU()ohkGiBTEaI@|`DiQhYJo;pq46I?#0F$>+IgIy~e;{mMyRdQWe`|wKm|Mf*M'
    'n($^LMbc+3%qMmnc`nfvH@Kc8<E$^7!f2?X6kDqHRGbwZF+j)>b=I`o&eX60pFn)a$IqAVqzVI}{TyhEMO2wk5<S2#mSb`;DSlm~=|9tO*XoF`'
    '#?qQ?H!YlFQEWLIIju6yw%f}%Gyd1t+90jJ&gLC=Zzxyz(0cRe(Qf>tQHKiW2?lWCq>$j22l79&DgW{zfP{}1b3kqs>_y~Bx@5WF8j{}9$tX6q'
    'a_C;O)|b9W*if%dq%q%cnhKAFUe;g0|CL;k;SN8;^1Z+#%nH7SsxI`ku+d_KG~&xFz2Ym+#jZFAPABFaD53|%m1jm_d3gV*^2!vqYt04NiSYls'
    'k&;$zM%qd)GKAdQ_VRk6RfLcnfk_T-2gi?8&Wtl)x%j^ZwKT-uwlz{L>U4)K;Np>DPqC6zY1?Jh?1Nhj%N)u0@txp|tC$c|OK-<|nx(pmbiLtL'
    'B!|uh(*3Z6awGem7G_p4@7+lB(a}1q3O>uE8i!}4Leteuu&m%sm9SYUQ_21=XQ@SRSiQL1xUjo6FLiTm1Ki4^bYifR<6Z+<KQwn{XN!D6PU_=y'
    'oX@W5u2r`@yZWXxVQE~(zB}O@wak3|)K*~ZJp|PxZ`HYFWs%d8s^)DzIkPo2#%Jba*87sHr><eAFjJ9M(AFaTRTye0xLsD@Rs4nyz{ed>N*!W>'
    'B)@98k<r#J@(1r*O<cYSo*f@V!GQ+$)R!Yb!cj4V-KOn5g}RijeYuu5RyA`MRuDw%TN{|~d?TDpmJ(Bw;wh_Inb@&#@D9tmZ=(?m&1TBXMs)b0'
    '`Z_o)C=<s&@<s4IO$QI*ih~nl%NH9HaW#yhpi9W{@1=E8=1$!hAL-ahdg*+G&%^-Eh?3D(-CDcX{`ip#!N0lPt)E6`c(SE|8H?;WgABz>)ByCZ'
    'wS6}T&)|r6RrBsa@S8pS7dE}9FS-Z5#33-G_)s<c(x^+O!>O@EwkZoGj~-z$S2v^hET6JlzgQo=HsV$3%Do`B7QP$bW)qOoLq|EcM9*}2(++;U'
    'X$PoTT)&GoZZ*e9ZE3GOE)NvGUb>4audtHWg7GP4gu_ovF~?BO+j=zrfTtz3*`Zb=9B>AR;B#=4jZ^qXWVS1-PE1P}-8dkCpondvhLX62)-3^|'
    '{Onpf{<O^n53Ns)HN8mP4=(CIAbG|4<kJ61C1B=HmH<5ywoh(kA4rdU<DHTIaqHtah%kTN+E~V{I+q>6ec017SRUhSOghrBMHVm~C-d=Q6q=}y'
    '%zpIPD4RDL@)i&o^sMW`96)SCydxZcDSN5{1I9cW8{HzBPmnV#&hy1+AhNNM-Qc%%A^8#oJ<P*dRNf<=DSug(H_E0_6*ZN;ZL5&lTm_FS;<AKK'
    '0&*ELoAvl3M3pD<#fMkNyy)y&>*sfYo36x^5f{@wIz?)Za-Ijn>0YbyMA%w3;pP>ZjYt{QCZ!am<>50#!Nq&1T*<A26jP(disK39z!`7aHkpbj'
    'AkztIr&vl5zuYl|f!{ggcF!hdYWPmFaDQFzAlsyCqodq4M(=KVjBPkpJjm9J>(8-`X7g)@TE|)m5jIx&Uh1E#?g_t2#O}{x$wK2@!y~*IuOn-V'
    'mSwp&n5adOhF0iEcs@rur%6D$UW`LICW%m5?dS&xq37ug3r3R|*LAw+VqPchYiYXC;>WO~uZ0-1n9TD<|2!RV%zBQA=x4GvIoiR`pcA#d*iln3'
    'qU7ExLdLUsFiBAyCSOm=&H)|uF^wNtG#9;5)(>8MI8>GO+(}YO#WWd-!Xj)84@kx*)l@$bc22e~X*d1JwV2)5ABs_Q`pKe5Mv^^UT@W6Pl5wIX'
    'VgC6R|LZLBa*6m6*T_|6xN2*4Ic;r{ee~0-pALdg>d4R1aehg8Z2D<Ha0SLr%4fmH{N)($m)>GLW%F(;ZpLPpI%m<J;FZdYf{Qub9;L%M8#8c2'
    'FedRLiD2ck5gH0Pt0OLdiYB@lCFt$t^Lakr=A3H$Ic8RkIMU)O!E7#rRECWbpc;(t+Xy1>cl5BqFi`4Mx)Z~`98ggWcs9sT@rZ8Aaquw}e;)X('
    'yswgc`^Dkgq_;yuX<y?VnI>pwo}BGm;^4a$hJHw|l9@<NN&C}AStRnn{EF8)@|kcGT*$w}^m9L*>XhiTkyXofJ7_k9(<BBi1Xr^hdLoV@l=yGN'
    'jworSnF5?((yeI+TQa}YX1}K88MVypD5lx|hQ~oAr~100Ypn*>Bfg5tPd5-+BflrN6lqPFwluy=kN$|#>jOcNG?^E7i<v!!&a=$%Y8wL|7#IPm'
    '-_Uq}4wTaEv&0YlFh9lI^63!MQeCCNFQ_;}8im7Mr0JAnMSwnK7%Sq8doOk0RO*%&6)+WeZ_9?1L<V*MlJiMQS=v&}PXpV)y}PWPGD@ydVR&2{'
    'S7ZPxVLXj7voJF-a`j<Ti1JL(@qhmf2y*}2+H<yl6{kNQYd*uA&yHY4cMyZ|Y8z<nov_i8)~t<?lsb6H%cCq+vQnij#U{2D?dm*ojH1Iy@Zt5F'
    'BfR+FE&*2WiY~UfT&$E6c@gI(WIw(DWw!ujFB_o)Kio?NkVF5;)l|X?yhPXA0z{n=rwhW7F)y|nD-QmGJ0n7{aM`V_C<4qe(3bv%3_cP}D+{em'
    'EqEJ{+f#Z8>xY@1&V;*DBJ-Q41B+2$7M+-$Vw7JE@~erS^M7y(Y6@8_a^#sQ;sIC_V2uX6Iq~9Fh<uBPe2;iE$G~ZVjBfediF~{SytCctp)V^p'
    'H<kjhz+73s!mojuNEY)PLF=RwJR9X~C*&IrENiME;g9i&`uc3T;McB~^n>6plC2Y5l1fLVV_t!E0rK!qw=@jpcR5cA<&P~4O6bCkxm!EpJ5Kn?'
    'DA`&8exs|nsvifCl5a*;w|qlOe=^b6CaL6yoHfyRPKq=g7jh?89+!wYY2LRaQ;bAHpekhmiGe^xDQ!L7-mFT3kBo53nL!Hp&%!#SNSZMjA#Hd%'
    'jYegkNu?sfMCO<J9P~?RaUPnzHCzwx3W8kwRV`I6eOaZSsLJ>nR!=Uff<CGSFe<4M4G!CR$xMM$T!Qb7wt1(x6)L0z&I&p8(`g2Bj4j5rR}gio'
    'T<UH^ceIjPYBe>I@O>@IAhn8IGxlXOWz?s^G||olE32YPx+j_ERHo)Ob--LAV~`UHF!WE)uSR9X!)>Cs@~W&@Q^lzxeOjuy@NZX%Oh3K)@B;Nw'
    '>*|awDcONu%DtT3a4iYiTBzSke*P9rzi?8Pof0Wp9FAebT?#FbxFT6Qlg4Lp(B<q7+QVSd!5o*QhT;^Bm;ny>`UU0{>#_ok`pPKnHn+6+yLGC&'
    '@ZNYQ`4&Dvo>R$c_wKq!s~ZgRl+9u?TB9lxL5yod;dFe(<|3aYPTV-@Hgf!hmmVUl1@OJuP=$$#jvU<5y-uuz&?VD>IWi7KfEj?X>E)9c<O{$*'
    'ZgCl$BG*%tDVADJoo`vzY6`lZrEQ;VrO6q(K6TlD5jc*40+{y2!%AlsGnPQ;X8m_>>EN}kUs1WWeE{3Fx2^Z!wM$Q;8t&)b8RA|o<Emxo;uDE^'
    '(b;#~(82(D06~=8f)^kTld!<~P}-!CZ#95e0r?Z@h9>!B+wvc#R7&)!4+1o%ewNiVA`^sqE&cSGMpXZ1MX^YWoo2K3<muDcx+!cqH@(dIX)LAM'
    'ldb+@kbt;=m+{K}1a&*i&kok@mFh3*R^9%}MBLAm8lU18kc{I}J}gR#GZ4F8t2N~yRUo!m!k1fDF*jXO_TZRAN3#itZ^-~zF3jjZfg1zWB9lQT'
    '^66kDlCuf6b+i$*rHrt_Uak_OV%`@JI{``=9L8Z`v?{{|+!1H9WO^==_5ksJ;T;+z?@LMw;q*e63Xg%IS;potI5@lnm0nA~KnU0?==-3_YOaz#'
    '-sdT>`W=1eqS`9gh!o6hFNtbBU#0NoliR+My}y<Y-mS4;=im8s_~YBd51kiBN3Y)>9scyb^YZPBPoG|YqDodmh!9N2fh!qAvVkkqw{&C2h^9U%'
    'fLqk9Y~Jx?^5ur1Iq%+nSQd{1P1Xp)xkayUUBRg{L~fnSHyzs!;LzyQ`EJ;6T0XXAkMT%lBfvPUaXQE_5R1a(OMSI`eU-k8Jbvko>X5q{d95zw'
    '!%k4wR@1Fyv%DWzcwO&PKu6^V;`msv%PNW280nVINVUB|Y}W<}E!GMi!2gi42^v?Oc)0^R9?9cW>=2P@!i9yDpp-vbZc(Y?FL{EeQ<bhlf7OAf'
    'ZlB}P6oFM|b=k<Rn~W^=ActoYw9oYO>GhqfDJq)XSSw+MUDYIQaPpja6GfJ?Gx=-_htk}wjbv^Pr%2kk&A7g9T&JX6d3eWPwLG-nL$xc+Fw!L}'
    '<=xJ@whVZ*zO12yMlzV!><o==XI#!o1o@V7tT?_9%Zti$J=(f5@>D!%&K6$I{$u7OEJQMX(kY@}n$~=<Wg-qfr739>#0TAAF#$^DtIjG?*2x~x'
    'vPt!?2I)M>Mm0t2v9Rl4L2PM>PSz4$gK(@yu_ITTY?HUJW<%GiG1gIGtf#)HxqB4azN<!CHELHxY-_!%5XkPzAFKkuOhw!d>~161lCVraDm(fP'
    'db0ePeax<CT)wBB`jMgY+nW#)c;{Y$IX&hgLyUISllxHREKZcyXM*Ycreq)hrV3NBa5j<rY9iHkTq2GC23b%f$Kg>0f9uoZ^Ci|{SlSi`AqpHN'
    'MJjT8LAS%~GwFo6nxUMPkZKF6@PRUQN7q8-aBUPY<&dzGI>mI9Q7$oHY@F)5QZH=BxLe61WD}E7@-}djgl*Ofg_ff3NL>#i`PyILnfgMo*d2uo'
    'v(BYLf!zcJ7Y94U-VvK7_JiQ@#ugK3hJeWo*QQm-C8|_;GoxBJ{WRphqKZsv?gb)KQiK)Non<UPC-bReAN|l+z6cfzTmF(naC6kkah#83aYb42'
    'ML3LCCa)32xPqNvGUXCCoMOVtG2zFmnVP`GWmOFr0S=G7!Wb2HN_V!OlgkW9w<@k9hOyi#8f$%evH7Q$n|~_x67?VdGo9syJIP*_6f3pAShM}*'
    'n(fsboG<E^7^Iao7Z+sRjc^QdYyEf5(ZO+WD&nN1+xx42*Q)+*2{P&k@tSCYrRb$GCX8MWP1<swO=vwk(mScTD|;i9dq8x!-!H+0$zWKSjHzx^'
    'N>oMZW8sC!+0N`Zmd0%~O1YX4IHM1OD8^E#ozU&N?qf9@nzx-qrgl<q@|ji9yPwq2Ho2|JmxO?(uL^@pn(n#4U<<BviE)s=j9XPIE_wIPI_xkT'
    'RbE0>QxBEmEYj&Y8>OYS%9T~!7P_<_zQ%Q^vEx*4&=m=ksUUITHF~Pp3PvmA0Ez8}`_ob>e>!^c@d!(5e>(bj`2I(f7fcpM`p}>wffe<2+;y1p'
    '?r9vnMCO!z=}|Gpyze0IFSKo5Mxs!lPu3TSAPZghbCmUF3CQwFwZ$SXO3gSp!x9#Y{yF;Vn2b8KDuA?L{d$yn{;8`Yqw2#<TQr?5$nIdI)JW(3'
    'Sji~ntYwBuRRjE3nPOOdcp9o2-fT6D^V!)B);QlmN8?et15MB8<Iw}>!v@wJB^(I>f{GK*aasw?;A}BUW@=RagoKe9nlPm*N4_s(88J3Sk~lMb'
    'Dn)?RC|DFKu{rihMOw3brW|`eB(nt8CY_lQo=};GG9@*0likzCLBP%?R&9GBhmnR~1>>o1tLmVzvPv9(<yDi&v4<G<rQA<&=`Eiv7%=Z=G+8c@'
    'C`X9cS8IOmaI+S~atEW4ic@d@ud~Zz-25Fa26j(nDpSC%f<`2#6mjP#6D=8vQ+M?G??;`(_aFXx^r`ynxmEO3Iwh%e(=kV@7PTp7&_6^a8rM1z'
    'sWW9^8y)_LQ=)`oxFfG)*c+^klYTZC>0?O^Cu8bFSTK6%qN7wcaL}I3OXWQzB*N=PrG{s>o8#__#XLVknDACWYJ(~>lMJbsF=ez%VQsAJtN1^E'
    '`wmuk9d$bJ1xf#;Nh?Ya9{)_0nr4!u9KI_4ztb>|cg`^LI-uqOJ>RzTH&tj%*IhVMJ?SpuJM376ivZ<C#)@<o+Au${<62pJiuriyV^Ikj?qG^5'
    'z*X-aV`<*g?tzPj?V!nIoHxi^BIVH|Jd!xcbnxg=<bJeU@sa$X2HlCJJ$8|<-frXDhCgN6jSfv6?u`Sa^PS4_K($B8%!elvw!)^?D<CR$pQW@G'
    'KDt$vz7E&uv|EEwHFkLq%U8}eUP*86BIl0P<4POut9)Ch8uwMay77GM=m#+sYXa7I?i@_8U2e`d2T`pCcHjI)peK0EE#^q@j7kVV3Qeb+U^+@N'
    'QPaS8q5bXf6B;Skywc2Es%zC5v2y4;xk^h{+y@**8iO{wyzsuV`tjdV?p=dd771P-W;xIZoRPY>(C5K^SA7lLpTTeR3|6Jm-PoomC%1NH{%@5q'
    '+%viNOzu6Cd(Y(lYcRR@49R*7iBU1}##eXWsjGO;_uTV6|AKtaD*n*PXl}bi%Lgw;b3BH^ae`Ht;+fw+TC}Xn-7VDF8U#ih{0pZIv)PLh<6R?H'
    'wb&@UdAekl@f?cRy81Dy$ax^C7d9O~s8*qu*Yi2o`aEgw!Y{Yp`CFt5AXimJ>s!E8c<k{a77nj;-aB}oS=9wf_-vfIkW_m1IKjW0IlWr>;=fWf'
    'uK$y&D#jIG8Hc&PF?rVhbpr1>4OZmVbhJt{uHav5r`W%{3i9|rIkC}oWKtIl8^=x1RpL;WE`Pk^`{z04G6Hfn$V93`iZWMK)YE~kWXo2K^0-GL'
    ')Ob;(!^H?qo7nwTGMO{Mr@J^G1^FeM@ALeMQ!7s3*W3b1@YhJQbJO;6nWD|M@Z=%^9!Z->9*VkO6RT4CD9<kzQ?y;pvd^ozAKxag`}JjwjpdCE'
    'WsX~sJ2vLazIC_hoA78~i%;_;r>ULKnyk^Svqx1f>Au8-<!3&><8n<I(|@GYd<pwq^1<}B(r`G>R{*8LExh4pU)$C%7}xq)HymcNF&djkU$B10'
    'oEw!>qJDMlWG*T<vKPww+<0SrTkVx@-LkqFZet0j;4UX7PgYJ5=4CHD{;lrhod@4}Jl~y>>ew~(p|E0FJs_~Rzjn3f6hr<FI~e9$k^bGnkCM?N'
    'f8%9YA(kdysywRE(%kSB`N&*6Sn~<L_B7fnPOL5nt8LH8sd2hu=!Ex-IS*i$+9Nejt3ovD4#Xyhp_EX{<j}ndOt>8RC7tO@s&wuI3E87Y&Sa^E'
    '*riGB%JH5KtM6={5vKez$8>^=oXDdRIC_uRkkyt?$LI<};QlHl1nWiGwPH}MYGdA-i2e{ysaI$_iV1(~Att`<=<p{uDs-)YP^*rX6)<gfQ&kxW'
    '6T6l2-L*8TI^OI}ZGSwy1nJKp8Q?ye%{%A_2!ae?t~1X?ypI&y>%74PoPP#cZj|)(rLMXQ#X+Y^7tTTWIh&sg5AMi{L~56|*sWA9tkf55s%NOJ'
    'cKAoqP8cgytEl)$BNmqIc+XA$j@-0c;L_s=UaqII^$(e{6jRo4E;HTD$yTGBS67x7Hi~SDq*-N~x!H>HmS8CyqZ_Qu*;pFxtFHiE*0QbsimbEQ'
    't}JT)0MVM#odz`SA<FWf=fX2q`R^_;E4N+QVKEtH7pa{g9rfQON8twnmuM{&8Ct296Bm8Q40hF*Kq>d1n*tp5NvC+ikDX#m^RszJ1=AO^zWi}o'
    'tIcW$=vh2XW@+c~pku!b9q!giq*l)8D!Ca)-Vl$&8lC6w6oOqvDB8Y<QKf#PZTwx8EvUp88Z5BM!`HR<@-bnlrE?};6vcuRi6>jdJnLUvpKJxM'
    'fBMAQI}w;1(Rfm;i17zkn0geN#pVT>W9}!FWvDbTR=&mKe{fp;k>Z*{>&-^oetGJVJzfLE3Kz-EgHumjc=Z*zl)*0p;noWVh~)Tf3%;l>D}e)e'
    'ywVXVe^!-y0RNwtXADh)7-pO`sHjW>i^LHvg&4Jfl>&$j2AK;Z;L^TSR?qWk%-x8g{Q*GmvQZXjI1Jw#oG)b*Z@jzaGgmg|ttkEY<qE_O^Vk*$'
    'raV^q@IhcidaF1RyjP~v(RDya9C3+05td7Ztm8cANMkGG5@y-vE7Q15@dIKAl}vb0+WXEljLX=qIc*MCR1GKU$XTddc?6jp@LsC9mukMARP*|C'
    'g2J}_u-olUuYtrFkG}u@`?7}xh7DpGNXw5haw*w~(@mLTc!Rxwe?QCfvr)RkSvs~cg~v9f#BA<ww_1;hfj2vzOAs)FoC(rqUE7^M5R*+Nuw3Kh'
    'LevXE$s(Og<76AtOn(-JPs~WlCPLuH$*ik`)$oOx+C+{bGOM=rT*Bj%tt6P9=P(l4diu#6Zk51!I>^sXwoao8Kq9%@ghdNpPv@RVYW9`j3?DsW'
    'fE+~oI5c_bmfqQ)x=fM$%wiSrn&5^(sQ9%;;0mCyP0LxA?pP4@UmQLL-x1_CXY4B_z;9guusIvf&vG+mE%)&?)wRnNwr<{;TT$5(qeGI;4HULk'
    'kGQ9W%2uYy#RDKK>F6RlB6omn4^)B=h@DzMy`H!>JNd>DTuhzu3TZGjUN+%|DgvbLlpnJ8<{BK4=>`2w^C5MvsH-GoyjRTr0Tr{$beh)guG~KD'
    'suy2F$@a%roY}X@61ADliJ5Av@^>Yo?{j6ju}*9Djq9{#--u4jO66U9xmGx^tB;Hfvf{INO)v3vI5n5|!IW0a@~FHhBJUG0kAn%OmT!8qaJxK@'
    '_Fyzp`j;_Nmo^-I$~AW`s&KIVVVKQ|IThaVW?=R)ZoYwTp8a{1)Z^wF7?pPI45~5PtxTFC7elX14BS(i%tOZr)2uXgsbu5|-_(x{p98Jq)Kii1'
    'wSGKJ@ElrK@sW|b)tmO!msHiY0WjFBs+XCU{<;cgKA4-Tn9XA?lCr{=LK`}Pvr1`I%}f6-Xoj=%)d>-@iLGEN%blLjzd^OMIT=%JOlqpKV)|Y{'
    '`o;yMsv7j)jAr-4;ajQDrN~rAGs$nT%3E5@i#0UED#A4?lc0Ltu+i}5+Tp1LLuiL};O#AFhZnOEQ9evgcd)aQA^xzw6D)o*htKDfrVjRSYq$Ay'
    'XP5uC_x$Pp^Z1wPnYomVYxo@cqh*NmX*vld-m$`R2%}-z(OH^|ng0OB=&A&h(j}{zhYVMn{`uF{2@iq&Sj@g9O|VI;9SOsvUhN7}_N_^wb%d-^'
    'RA1G`?q^gSE@e~xWKHw;X1s<FXpLN=Vz-b%z)k_~H?V7ewIjGlg}a)kv}n0s+^(`7y{Pt$sU5Kto;x6w8_f#&zPiXZR%kDe)L2q*yAlgl^4Vw!'
    'Z?`5UtiG4x!W?uHY*?c$!-u&@%9da}GzvjllV}IN(QM}Q4T_qPE~&sCd*q#hOsXkIh!mhe^I;=b7JYTVt}@AX=xSt$LUV7FCR2TJTYcCVb?Q~7'
    'SO_pew%I$qV$qY87D9}2z#^7Zgc-{M!mN7PD4So`I;j|r82Zn8A;MzA%^xM{!$>Y&tGy_xDRjQ+rQhA5#BzK7CQ|kx6wAx{-X-L=T|!Es@~cS*'
    'cK#G&h1QC`7ZV(0<8I?1P`+}8eN^Sf>0LY8P28X|U9_Rq;0Q#KX<4v(g7*U;WVQ=MoQzZ#cjd@}Ki}CT!VrZBSycjz`t;td<@a!FF$d?=)U5u^'
    'DFQS!IgmJxJ!e5kJ8_wgdE_6D`0;@9iwkw0OurzrYGdy7e4GN2PDhya0<9<%o{*)3h8funTel#lMqGja!KVx_JRk;>d41ZZd$Z@A34gRD96RK_'
    'JJ4!oO;_AEae3VIne(hK?n2?Iw6&-hS%GCS<@gdGVHTVGS70VKQ(N8w_hDoEhU^FH%<i85FhrZ1av)zp;reZGF!!q0%~Y=|axTs}GFcagpsmTf'
    'NJZQ|>+%O;U4#z9VP5Vv*kuiN6~kg<@XlD4n;B>M__7(%YmU9m*d3h#*{5{boLRaPJx?vLMvmQ;8M8l@3EJE^e+Z-VJrDG|@IZLob4_GZrPD%1'
    'Pf^nJv{2Ink56l~Q&%bH-t0<ckX8kCgCe7B=ss1HtWA~`OPMr;xlMpafz7bpEUs!rMy|%x?ikzAVo-FZ>8v9cOWE3;SEXIpE0i}RY_M^j4MO*u'
    '1|S|#iRmI{GA(r8OUS^IrG?ui5S(y$vh^Eve!P1M|NkIv{`(hBkI4Y;BERzRH^MEv|MJi}AbZXD`M;N&m-fxWd!5@o<9g4yuFAOHYbiI^QnKVM'
    '@=|+wehbU0VU0Eulk{Na+w7lSCnje^>*tR3lZpn1({?Kg2HCiciAOHdbc%mF?&u!aRFB55<sF){=)-ct*)6hhHcDpM{2C2He1{mpDEdwz`dK1T'
    'd<Bh@`FJr34Ok+3s`IcB&vUauJV-CIe%ffrGR9`H=k)W`EzkR%Yi0M6FVB_gOd+1z!3tD|qg@VmF+~VBr_qZ+GM&p}1+tPYjB>u{0R<fIFz2_8'
    '`NR`I&rT17Hr{$Wy;0uV*=?qS-m_-%_-S*0_<S#U{H&Kg-s{01$*}o2Z9VS~;K$RU`607GvtkEzoy;53d<1_PPp==mZ9RMHB(1zo6`KHmhb}HC'
    'E!J)W%T3^7wB7e&jhylpO*nGNSrdBR7R7rfY0<$HRPF#6>(hb-hOtZ4!J)kw(1FQYSt7zUmOPCb>yf*TZgum~NW)Vs#vzX&PeQ{To)#TGipOa('
    '3D?Wsqf`hbmMe0hjDpHD5m`hga%fY-A6yk8<PoA(22%mFG@H_WY%48{PE64-EaRn3LwJh08}n?JCSKW|m~j^?9{&Xq;%B1EXJ-)2EJS*GV^xd@'
    '@3Qc0H#YeW+1y@;O_C0YSPo`k1p}Iz^+}S6;E?-Pb$JmbLUAEMm;@tA$tg@4;;|?JEuCn|LCDRJ&f|0KXb?C>p8H0S#%D3{&?FPSPv@}-^_n3l'
    'Z}J&+^U|6^y|?ErhM5S(7b-rSG=eJ!nqXAeDCwcMUE7Ql<7f75`4Z%0QC!E0_wl;vnIEDv*Bm`EsFt?nmZn_cR`i5h&q>eg{dfyra)1l5;#lXx'
    'N*=%>fPP!z7Id0fA9R(Wd%Pe->q<|9??KEYf(M-+4sZeH*CzAuE&1EsOc`t)w{+&!3U^-q^#0A^kDWI^eZ*lljnG+LNz1G7uk40*6K(69wXNQj'
    'sa3m*&AWQ>!TJI$Z@D`pBS<|Kb7>j(T{Ang^lXM39L?oeHBQwk6h-r}qMoR|634N>z@Tvgxu#%nXxGG&l(ii+yFn5-V<Nx8kYakEcff>gfBJQS'
    'iYj~y9O>cE9=7W{`P8#>hn9PPIfF;rudzJO^txeL`|mWFmR%Vzs=YVswAwdSx4VvrEU`@%$*5DL=^)%|AYlZe1Lq3P%6Er9zW?dt>(0m5pZ@yx'
    '=-XFO%dELqtmqu&2_@o7g05lw?uw<tdNPY1+*>pA_;WV}JB<(%CM!##*LFT0T^nBe11wf4_0-c@iWG*Ff%Z3?RRPNw6(VWwJV<)^rCHlWA)(_M'
    '7M!Y*H#S_@OoM+9-i8fBspFk#t>`(iNyr+>7A4)eySR)1v0T6;Pg2QB&#!W(A!VI7HBJY7w|26@XQfYwliArg`5c018E5l!95#IV^pmX*vJW2O'
    '0IdzoUkQ4a4O&Fbjy<E9h40*sg10Yzc>T81tktF8Yqr0lW_vT0_WvtdY*kOr4G?vbM1aI}na=QDKoPej#MX5vZZ=ftI4@ZZ)U-0#GRLo_6<3KU'
    '*HO1{dtFJ7D0mfPQ<c<-;~dNPayu#-IKv`^LMMk$(n-=ANt@_2nI)r9I?9T%n*~RRvit|?TLrE1s_|sfAF;8~>^DlsBxzLS<~P4`IJ^UY<)F#|'
    'kcZG5p)ZUNCRiQ{I6jirLNUbwcP5v9lw)buS>S&QJGQY2oG1v6QU;-8Gn;<Ey%n664l~A#HkG5SzIJUFx4v>xuvt4pB82Zgy?zV7Ty^{UYb_3d'
    '>{{-<0**|I#TO#EEb2*PE#&$8m%WzCXe~ui?XuS6j06fHPgEhV0v)Qvgo9a)9?_z){*3B&Q2m5{c=;NWRJ<h^*PD^C05?#f8!GMMxOH%PciL~S'
    'sJ@mOTxhdv^MtLLA8c7(Fkl+DWZ`SUjPGbIRY#eCwhn?VDy4lrL)BK$+HE~1zxux{&d#9an*=VD!%065<{(qWSa<u=hgW~!ehb&b1bBPapCMij'
    'f)~@If1Yl);$2dE3(k_zw}at(hA$MWg;DvyI&Z7EH_KhuQ<TYsXo{7gBc+pxSeK&Pn$^FBfwcnz&JvQdI#fqfcJ%}D25-2L?|T+QtWRhPhZEW!'
    '<qr(BT3;zy>|ORb^MY#Dqu_5lC@`rG*!SoYd^1nat|R2=Wf0(iz=;CHtNe4enDF71U--tfKx73HM&Oq-$tmzddm{2iv=me?VFS)*3zU-*@x8Iy'
    'SLYe9V^_0eYP@DqUN<70s(h(qAGzppz_eF4S2`IPpP_*4o`RS*M8yhLeMN3i8w$yd$&Bi-g=Y8^p->^G5{T|8myLza<e>lSjH)U`sy<3B1}WNe'
    ';JvCMn@w<4Wcg@s`F2;<533+|85JEHqd&$nC#2=cW>hR0wvC}$&^Ql081VIPtWssqcK$%jIh^$tEa4rWY&p|o|3KBP@(XmDspKCgTRvpSO{w}R'
    'wLj7$)R>Q)Q6t@9ZRU`g&+Nf?!&Ix+(F%DWopkj%_fP>OkGjdJcu;D1>eFMBmgT^paUQp7CPWezRB)4)EpkbeoBx+)p~#Zw`!db)fT%q$50D&G'
    'D(;1D$k5&Mcs!CKb%X`|Gyd?8qn|#${EMloH}8mp>Wsthu=?^iouB8FheVW>;zHrdBbD?<Ggw;`Qb$S%EyAKfl!6uje#BfaUUVmHF9mhf#huVK'
    '+~eW1E?WN1if#b^U0{9>k_4UY{jku4NQaLbqzbqp?p*RiT|{k{MMxx9#yW~ve}Gd@RT&II<VWX~6X0E@frSBd&%QHkLhfib9q?t>)&%0<!zjh8'
    'be0Zva33nHi<xDm6B<2Ivyxtq!7u8Gw1-g30~>^0*l*n^=+YU4zfaW<_2jhclPi&=%Iy)5c^9=9Xxx!|Mr6C;r5VJ*YhbwaC;eoCZ<Bycf!GCl'
    '1QZm<D9nT9QA1SUN-`muHOMwIH{dBqQw~J!GJ}!H@yqVm?guu*b-oD3i-OeH<x#s?(J7{IA$RbC?_A}uRY+ss4MLbCh%THACK>m#vjzH^u^WU3'
    'DUI0U_JM}74~K7KHWbi*aTzAlcrr+4v*a3onTAKTNX}P!_4K#;nPs}HTvWb?%@q|e@^Xs9B-c|I;N)Vm>4tRR?-%0_*8zcmy?wYovdukuA0j+)'
    '@IwwL>z^l+2}n5Gl(&{>hk8UM(0F_E0>fp7j3L0{=pUKR>$3K=$q@)urrx;<x~Hb~5&gx62}qkX=Pun5UUYp*N2tl7Jxu)&VNC|K8JN!FE>*iS'
    'O!|G;J7m1fHPr?3Awe!+!_ut91CQ2XGDtrIF&^jMisIWrmp3NNw)+q@MrkpB*ge3@PYx=c>az$J13u>k58=y6t{_6KHM(|4uGXi$&>aocsdG2F'
    '0Pdl_9}TDyyf(A(VjS=*?e#9>B2rC_yJkp5uIQ!c!3Q6pJ)O=;$w=Iu9L2u-vLQS7tAPZOmNW8Ch<690dyseFgASnXAy5Xfi5dsM0q8xM#c3ST'
    'fe}gqNhN5adauB_8)PUN03^OS@f1q*MB@gwKTxY$vpTZ{7UIPfqc$%QPTY{P6W9uJE^?eA6?c*gjOM2FS*(ZV1;{>^WEC;KMWr$3MWdzD7@}>A'
    '3kbNh_5>bU<9F;e3<&;9+<+Ew(Cn5?7kmI8Ql+(sjia*4x#*gb!vL|IXZ^(}nFT}4m<5-|MGW&)JCf}rNYD;kJ*Yf0^C6xUHs%Xv@XaW{f;)lK'
    'WtiCxn4-LgNdk(Bb2ptEeF$RgTq;X?m`sfxn7W%+)RM1%Nj@8&|JRGZ2i;%%UoXqZaNIRH-tOr^@S^lglZ|0M0#xC4<)E4plD7k4c}2(nM~{N-'
    'Z4yZ6ByhdWfxP>Msian9;uP>9kvfNVKHfAgk7Y+B*aK$LQHAROBh;*pVC#_Kz(TzYL`uPHgNlU+i@O9}vFV-Q2z65c>3ejt<y6hxdW3U9PZ`cM'
    'P99&{yAc^=ln5NAyfif7q2WP<X(nVee84C2K&*svKR;a-f-K!FalXO^q@|i0mTXR8!yz@oz7eDBIwH6l;9vkcn9c<HkKLlq8lnmtnlwJD@EDb#'
    '%EwNjTejnXVE@#Cg5uxEptf!OXJp(aMjT^(x=oGCipi7`LC}=^{uE0lvJ8#R;x57NvFogRs{A}8w6WE|r^J_?83kcR%NMxj<D#OiN$*1RPHLka'
    '0=-Bqh*8ibWU-HB1^!UQ!x}%KO?rglI{fp91@}gcUcsvfKy7A5Z2B^JU%OIknO`a&OM<a2;442;=2|Q;eByA`46Jf_4iKS>2@_@PzxWLy&ICxK'
    '&zz~PV7No{S-7b};-A^Id@hn7k!>fFkw=y#fYZjteKG`T8wUNxTJT!qQo5&@7r;i_-!mqo;OL3`Td!v6LoS1G;88kT-Md7&3_+!a+}DrOq^>uC'
    'dF0VUJpJV1c9C8ep&!!k>c($W7~(9u&sQ_W9T}sR3~+)O(3iWg5aosX6U}AfPu}F}C+xrpE0<`O73awGOB+)kCN#`Osn@P|^p6j(J1_t8`sIIP'
    '#7OXBavcS)(5I+m@uMsi24DqD7KIkr+<=w@8N9LIAPDUIB^-!O0ZePhFj;Uv{12|*Jnt=rtC{JviLXR3Hn8)(7z{)QY=wi%&BBW~E^GIz3<68-'
    'fTNstSR_q2*iRq&9uy1_zNhF6(U5XX@x!n~d%e>k3iP+1UcUID^XBmF>rb8cFW$ZW1mFC2vUQpFlU|2*0qpXVt%6pK&*90|7aru(hr`z|Uw8ic'
    '{?+S`Zx7$UCVS1vqV?oileeT(F<o(No>j2xTRApLkC`#13LF$kb^J-}tPcp|X;>MpxnpfX?Ili^kZHgKCKm4>T7%=R2U--aNG5@;iplHX+Q@~t'
    'L|PbSWdQE?uLYVmh16nS;&kDVIZB^y_tHVH`b6l<DB*$ZxWfLZS5AoauEf46x&rR=ES;cPWOR)cNnv3g#c+-v1+a60rUASSQXrM$2^dQhTZI6k'
    'F5o(tq)3ACFSw35JN-Pv7_7utXoC^V3nm9jWnZbUJlXowPVxQ8R@Y|*O(z3WAI)PK<18i@Ym5V}s|JEsll*ET1^^MAd*$>Q%m@p$qk$AHC<!_!'
    'q`ygU_==hNf*j3RHng*)s0&)=$%rkR;H_Yf`&i)N!~LqnC;X9eTm=iJ#Ql&$*Q?Nz+B*Y8?1~}kkVpyt3ZXx=j5F2hs28e-K=dHh1_ZFs68JV('
    'F%|J#%$~uB$W7D^_Ol2Ez+^Y6(kyqa^JQAWWFft-IKwgF^Y{Y=oshQ$h8O}K?O`@MuRHTun(mAf*aov5j%#U18wKJbV)c2Dv&fDMRSL7E+|nLp'
    'QJ1|{I|Z&beAqn*UbFvdvFM>?2#FT%W^vV(6d&D5FD>R2^#tF365WvBE9KK?Qn;98XYc@Hmq6Ogc)$Jd;q~?_;FxoASa4;3t82W<cBAH(E@_`7'
    'uy}p|&vdPp_1Sg@7=CoXU#5i)FELuc)r-ewNk4xIcA+2GCW|o>av%W4Yf>={aeJ`1UK$W3qVwx1u2lhz!%;Gym)(7db`_-NV^L}wNH$;-o@_(6'
    '+cNreTcIKuo#k-VosVPBA}6Eid4kI`&U4xydVS30QHYHQ_5rVrk%;ITk3Fz3Xi)>iCmeiG+U5Z*?LaN9(&p2ttkvn;<ZJ)2k8C-{l@JK;WHIhx'
    ')b&tBPxA##h7gKFF~At)z&~A~YYH-K9#y_WLrJe#q8R5X9XB#B5ecV0sY|lVlX{=1r=U?((bMCXqp$fwiB2V6PDEsvghBS?k@{Yd<I7Qeo_hUk'
    'DqUYRJdx+ZZp#}b^4zj*b%{ov+sgN8<vr1M{7f98ZaA8t%))*qAj0_EY}{?v-hUib>;}>y8!x4ZZ#t+r3a=8|E>tHcTS~666{Pz1E(O|XX(Uvw'
    'Krld&Z_1zrBlCyOZPRAgA9+=}IccOeQx^_0`iCA%?#o50;q(URXwIr`7OK@9&BExvViu}}cNUrtt(%2vbw{%>`mdOUYEc=lGaAM&-vf8%7YYk-'
    'o=aN+hfCQnOP$)pvTYw1H$j0GSlR{|sRUH&h)|*h(ZkS}H#X?m^#yd$s0f(Gw%tNgF}<rwo*1HqK0S_?cqKWjeB<=stVi3ovd$+Z`sO}x+S?S*'
    '`sK8A$e*?CWd448;VN~ExYpBg?B}S%M~{5-G<E?gJt^kc*$+r)XFVXXsD~OJ_>l>N591kD;coa)H|@2^HezX`yiH|zD}EAFsPYGF9=`$h?h`5I'
    'LOG_J@<@`08Lhw7(R=mo6TAxRzl<@5#b7ZWUt{p^F#9b1K0V;Z?ziudcSB%9=S`y$aoo`-;617NQ7-1$@W`qph`r5|0pUY_G9VoJ5F<Sdv;uEK'
    'Mf&`St&oG?M^CgNfwv1X=z8z!XQfSRngAr!M5IU@<8RbCxG`|pF#%(rD|%D{jh-T9m%&A(UIo?<vzRbJ3A4sQkP!o%p;7`MI!n!CLv`D~IDN*8'
    '=^&Ylb_w2nqPvdBg;ZWP0rV}R!KRGNzB<IF)WBtAc{dSCb`RJY8-$Eg^Lf+Y9#sRyID#mJq~f}Bo)pf2c7hRuCse)E7|1@RaB$K~!_O35F$i5k'
    'NyhF;BvPVAZj!s24_X)7Ss8O@V1%h~A#nZdN~#l?YD5#}uJD1x^f$cMl~!Y!&#lI^RO=uPyv~KvC*<&zD`6Y!V)e4jS4l1*;1i1&DyLW&fm6>)'
    'G*6Rk<_=oD#w!Sf)h?8-RH&3}1RO?c@VLV9b1ppson^8-`Bv~DGO&j3LnGyXGf&Sb%?Ya}i4p{uD5iOO5Hx9+#2z~ecID$LovOR9fy1mhF<xuG'
    '<kglRY0p+`8-H7;+IZ2dJH^D1*SLMioS^}7L<gM0*j^ypO~cr~6+Sm2BJz0m6jLJt6Jt!Fff15ca?_QX!+*UEn!3MRBFr6RIFM=1hU^cDnKGpp'
    'C#e+t4MK+;8^m}C&3u*9>49#MZ4Z(&w88+U7X{{R6DNphVUW#Z&wjK}(x9Ycl-W%_I^G5ZwuSVe<N<i&ESxm1#`s6*tzJvCSe+FK9Fp(f?B2hi'
    '@-rAN(2pg6Te~3L8QG0`>3MRQp~5P;2H4q&c(9m`Fd#S`_~O=2wg@$XUi4B@0n+W@DVXcoiwYI1IOnLwxU)()yS<&(r?h>dV|4JL9Zh*|l;J^d'
    'yH#4>HgmA%l`rxTqCa?Xp>@C*>!K|w+U6ki`ADuc{oTgv%T1d#hyH*McmlFq%GM{WBWY-7IGwsMQc;o!%3+F`l!j=Q)WIo$_?&bC(n<OccI&To'
    'cfK`|Qgi3w*=yIGg>~-+hswi2gKja~G9_GDr7`^19L?_FmJVodk&Onde=a)6xHvbf;Y+rp<?nnje?ago`+#J`%^<LZHj!muwzin(P!?#IgpG-W'
    'dy-IMKe%~peK*=sEp`I!$*R-yq;NgGiC46YciS>3TYh#BbpIZpcD-vha>0i?xch>d1wY8vcX8H{m<dnsbC-8giPyw+^)Eyfj@hUKot}nqymN+0'
    'Cz9Fq1A4qotcH&7nW7G3Mm`YEYFBSd6WxM4sba27mT)tYWAsqV1ehJJ3{<nRY)qS8-5$5v0il92cyv0Z)k{Au=~U>sni?fQ>2zZBowM}N*y<Sl'
    'r$m<Bt<zElDJkAy(_5TT;hj-o&j@5xYf|Q9P;YNbIMpePOL<{{AYtkNW!7>n23zidtTaZ{8Uk*h9JUp8sDE^|#Fo~2mSPUC&J<HqEI1FgxPvI@'
    'A5AcLkHEiWogfz_z(pL9wroC&k@KP_A?4)`_`mF?85@H%43%M#FJ?%zHz-H2*L#*eNqYOq;PLSJe!AD{?X{k@_J_~YX7AZ>(0cmxc@y6B%@0#_'
    '@lWQtryv3R9#7|;baIwW(v)Goe(?7BV}sF7Ciw&|In?FSM+UlpFmv4s6P=y(a{}9rQ9y>=vg5cQyH@7|8G93AXA9{xhpRPctPue24Q^%MaOSeD'
    'L7gk8b}}A^tBNo>8D>dL_k`s<>Pbe3!|#i9I*)@lhPM@eg|8p}_T?d8l?V_p5!XT9?=NOE<$i@NJR#*hbUVQBia7`IQNp^1zda-o_#yPlVWHD0'
    'W~(ny^#?n7bp3zZ`x@@XZ6v|JqV3#!lF~9GQr~*5?8{^_lk9a*a+99fw`+UVltkHVQ<Nf7wv~?m`_%^k0wAalcPE>@OI}AN37`NJstQ1%s?a>F'
    ';yH*G*HBCj7xCV&W7n3N7}vp}=y=!^W@=kDwxc^S(2u=wu%>3^e%LkHanq|>VclgRa&IFq+szsn(c~(7$ugjZf85!*;dpdxUS*+)7+qXRO|2oM'
    'u3i)t#!wN(H8+)9;t*OnDX66}THYPnv+|ijR>U$OhqXj^B0Rc=A@hHs<Wuc085zH1RET)$rF4PDro+?1*7J~($1jvT7^e~yJf$kN9R4*@5ubM?'
    '$zi>v>Xa-yzILBtWxv)DmO7I*+}4mj?3AkGtVAB3s+p2cc5o8o1O=(tG{faMNboeLASvYwD_|Ms!Vu^G9zi`9q41XGfCKOBbIy0dX&WW#E)x+l'
    'bVmdVatb0@Y%#**A>3zF9m!8E=!r!YMq4y-JQlv9Hp1}YJsZZ;x6vLsaV_HiBLl!f;vKi;CI%V&?y~-Ey;ZA6__MEHztgY4gG9Es|5ha%&3cP{'
    '#-Q2UD{e{c9X3S`+Y^b77wpN({U%JR4DY<!x6ziY6vMPI@x?vMJKdnCu!ZJ$H#UhumaguX`p$A0vJocW)!aC`?2fPB9g--TOfIiQm>5r3g1AJK'
    'Yjhc)Y~zG2a`d&c@72Cx^a}y`vL;UAqsMX0=<BI<n(QY8W2q5E69RM{ritAu$&L>7SM^pMKEm&pNKKNmfT}9Q)};wYj`R4sBxWZycUH0{GG*Y$'
    'Z++wew@~t~VY8@-B}b}$=uj3++T14W=V`Hz`c`y|*N#W|O#F~<pc+Cp3gFtg-cR5Ec%jbs9sSo*W%x7-t8$?KDQ7|BSvn(RBp6sZh72hzOl*9&'
    'c)O<kTZ7x+rZ;_dF-DQ>7&2i=Qw(Kxa4ZX*om`(#7jW$-bm>A_CT1S1BfP2xVX>P{6`NjO8?nOW`hQY0%L3!yPh)k$ShmHXqg>&~QFE>>J|o>d'
    'ESD&z7c9-~L;r0=`*3zXl|*VzeJk^>WnC3?`XWc$^IP4j2=c4p<>iDIYvqh)CX-l%KW%vcno>JdhF8XT`8fBaQ9TgBxLnr5{Zd*m`0-X*bnrYh'
    'l(?q=WA7k7TG(VEY<3ForGi1N1i_c|brECm^YwN==9q%0Yay3Jo)9&SJ?=V$zKpga<Wp+zMFjP&Gj9{-w_cn9{DxVPz1?O{NH3yT7O)!nXW@Q@'
    'FlvntX8EV49(JUrz8e=>6bLs5&@e-lt#oW;%_>{9hJ_cP?yDCUz%bBZJBj8|6f(#IB|m^m)1mXa>L%5tYF;?Y@NY>YR{NJpDUaDlj*(al27W7n'
    '8QcDW>~UN+)3jh)5hcuFKc%L^BGiP8yT%7)5E{F<dyS$KEl+6XHcXKfPiD$`QD_~>F4Pridl`EV+c=LJF1W4@zTIHqsV*D|C`^KvKj!F@j-kF1'
    'bhL>k6W{aO=u?hINb`I$3A`@7#%uB&fe&5J7hi5cT<xQMGy|cJDPG_6IU&zClS#+(J6?xAWbqBKCqidv=&^`<unSywLeaK2;P3l*J(+Z68?t>m'
    'zsTcN1bvxIdS090=lK?E>oKfZoM%8mvwhBhy3~jnps_>sL76_k&)3^X`}8M3brMsL5zZW@`FIKMK^s61vElREaGOQ>MPbE?o$69j63_1@lc5)2'
    'Z8#?D-FCA_XdN$rFZ+3d*`X@RA+AEouyfQ^l*ZcyX6Vx^?BA&M_!7nGdPAKVV%g{s<6eP)yV<UH8?FL!EyFAiH!;=WQ>T~96*i&G<ACFptg)3e'
    '5cnktv)c>6Uy6eTorF)&@_fBcsA0Vk!QHNZj#AXbU{yo<l;Mm+sOfqXa^NP;7c(q_9lT2a8t3elNR@fr9^}}D3mnWoVxTSGG57?K9@CYNU)Sq<'
    'YO{}MyY+o~CY;<ZFC>sQ=Y}tjQO;z9xq-~&ptijce|W^)0U_>|!zwKC8ok@y3==wiEEq4+kXqUn(aXR49$VBI;L!eCoW2hO?g+fTUqvLgxDtT+'
    'emzU#8NT)~{VslpvvqQhqmAtXq$c6d`wP5tiYWk)u?n|$i}gc#5$6{;L@<EZ!#D7GxA`$#Mp(ViIrB||ggL`pL@KMrZks3+tyZi>rw_n4@MW_S'
    '?IXINkpIlQeE{H#FT#4BcnwNIGt(%?z-o_J5fN#MJtu}hp6J4d-HK@%h8<r~YaCw8cQYU)_~QeXd>lw3C6&6sA0SvghaU{g9O?ZwpC8Pz-KEI-'
    '2lAw;NE{&xL$|cV<uaEgee4|$l}g0hxMV8O1hn7p>X3Oni}FCjNM1$y1_eXWG9(pWtA3TxC__qb(DQn}N=WDPL=+3WfCgNq+`o@u)YH4F{-*c&'
    'qbn_6Z<rQ=HgKYv-(PG5E0U=32AHsYp9$vlailll{4PY^AUWE`4<MpQhLfb&#Ap^k*dgZKBw>#0JXmdzr0C#)V>sA+PXYM|GBGVGB<C2KL@}*&'
    'bwG0LaP1k!rkG3y%mtf$2<s}SE$A?ZZT1Uv?4J`;_(<aN%aqw2T1iqCJ~LZGC#YP6B0Hykw6Tdl>~CnMBS<4IlHRWp!l9@f$C5CCT+qQUh`kH6'
    'Q|D#|9>_(y=(DX-nWdmEtR3p-l%K`y5n#>#@ztBx*ylFBZejcTC{jVnbny${YDc}Hins9`xwr$>VLFhom_MK+S%=r!Vosq!%0O-?iAQL{FEZ*E'
    'p&#fgo5yz`b@+~EMpdA`CQ-Oek$eW?BeT0tk}Zo?g!BtxM@UE{q#4^nu6;D(&)fY)wz#8C(dcKlQG(9Cx8aJ(6uOqhs1cMD8PJ$#@gvnt5-(e='
    '!VORO*sL^OE^!v{nK<!k)dWb7NsL?m5P~44!3d}Zcu&F^&xMG-y%1!Kxjfpg=Idm=WhoCe=C^24iPuZb^8~*VF~fpjiPNYUxE@UyN&!hY4oFbu'
    '>lH2~0+yrrCR)QQAc`2tbFMGm!W~U)_^JI8*}5wd_%M|~6Jqk87As`AK8bRD$Aw?JG-bmcEthjIzQUqvTg;;zkmn*!aT3o3a2gy=nW|VE1k&fE'
    'pD8|?qUT-ubG+d>5J19&1T$1ev>)E@W8h2z+@TK-xFi$!0ca2Nhj@W=6poJ%7<q_h8LUlcMdXWx9R@VW1rb1&$0U2CCGVD56KMJI@fKDvYIyeH'
    'hFGNA;kPZVFavtSwJt{{<+w`+qaEQqQ@n7<W${K+ATILLjR>++zG;&_;~D{QQe5qUrzezxyA}C)5k(t(mw}WDhV`i4B3@6pm})_la4<alF~bvq'
    'oSw$3n_zo06B%VBDlgpSYk9%A4m6DHoIgE8X(03bAAGT69Z~Iy^vqjR-pbE5*cl`y41Ah~Fj2{3e+%RL=xdxK4%`I8F`}&dV95FUWQ@2DEbT_d'
    '>9WAjUv|g@U&9pg6>m`1<nB$6knnd59=cUXDRxaN(i8%lBI2ifz1kGSSc}mPy`A4}U=yEDfi+;;p3xO{ExJT>8m%_@ekv}Xb|O;NA#?zL!f$-v'
    'd5d3KJaqCKz5Hdp{W{!bVe<M-%eWg{<|^G6PFK4B^YxqQ+i$=7+y9>a_#Z#M{pQW|hrfUS=7+bid#%^&7{^oSHaOlmhI+WeSjM|tgh38LvQzj6'
    'UUEvukX4Kqz@!VdRS>2NltL!=g;5)bZw%OUmA-X@ql2kroe94d0RM^_@n!AR7vdogF?JrtjaF%8Kxn626ya*=BXkpY)PMcvZy$jNSTm)^(ZJ!6'
    'a_yo8+3|Au=k>b;$Lux-qt|*ZgK&L^(<Ceq?xXKgiogRq<Z<OyDnzZV+I=Wg656FG{8fn$>vejA{y-oe^c-pQFfCjwY=bkyY1H=N4hvn2dbaZD'
    '?!uZ^XbFyLkbspJqRz~#Hy-oq?}y-G-)-3V%_?JH1M(a1R^BDau<4d$6)K3!GOgj(vpaR~uu+Kf{lxFLTc?*oKZGf-uNr<E9l?6=6ZCq`bAiyM'
    '=aYIy*MQegEN$EDp|+_&NLijYWG{o@oL-{pguGlQ-Qc%I4j$Y}AW$0r{WiE&mCx46!j5Bd>_04n=L#G@z(2Q2C<#PA<&|XnnWZhHoeo6HS_nZB'
    'fxxg+(#i^(#pbwH<kb=>Qle2`{q7zQwzK*e9Vo0G)zP%4B54z5`SdQ@i>S(wsaRAs3K=Qyg-nvwR6OGKlHvx!B;3O;LZ_Rjk``HOzOp&u`1in5'
    'sGXC#YO)a`=Wdz5Y{aPnYmUueHAZ&;QCil^Qg5~M2!TMmgv6_Eh=AC%^D&CAjo}gzr`WBeC2MFY02+FwPcxghW|m;ZleX#IJ(iwGD2^NmcS68h'
    'H7gPbgPmMrW3O8kp%5nKoMr<tCW}dVK502a)|5v?>$Zp%HYGjJRE#=}wp9$xw``0yO)j!{L>oG^?d0c*oMQ0MG!maGa+Qb&@>}0*&i+L7FRslh'
    'ZkjVVfR;fbst+s}IDU`873i@CD}Km^ZbFdfzlULtaEx&r2yM0P4n7lT!89}JdAx>;B@R#Da3853X1NpCuDD`qa!G%$9WaXB@T9!1b3r_V4;>D*'
    '=kp^ps9~;%TeVZzvub4Hyn19|Z~sY#HqL*SLu>9$eg{JvpC)NtL;3%*ti_esAM*D9LwU<Ogg<2NA2Rn3nfoisT+GaiTHCMQzN~|E3oSPa^w0C<'
    '&6IcN^Dv3foTOzY#;Y5V=6k)CF0EBjpkx7=qq)|~5RPR*O(&pg(rA0#R?YVeWeVolMrZ>xUBxLL_qr3wg#PNqO|qVai4lIbc*5N`9yVopw8<u@'
    '_|Yt##dknnyv!$bibbS5{-3Z>uAV&5g*DAT_1;*}8SHqaSYe{O{+Ru5%{6A9v1<s7w2Syk1LB~|QwC-V<*G_o4bq`9M+a;wiya#|%y@kCFxTM`'
    'q#Vx~MawY;7`i+d#O2PhF!}$?p|A^|FcNl=(*^=N^*?GHc4R{oEkxtY#>+#k(rn?7VH0~+wHjZ+X_Kl1M|lTvOe(%xrA>Klyj8W6XDjoyqD@6s'
    '_2$YcXD9u|kx!eW%-8C?G-X-uwwlqLmCV;tAm7p&v~n<sb~fuQ=FA|PD+SbL9iJRU>eol$|Fny?`zg5yMW<F^l&P+vAQ@u2W5BKKM_aD)pV|6S'
    'IcH5w_$LklWIk^MH0UP`fDQgQ{t^C3=vSam!M@h&6OnJX{4nlK&j~to7c!F_bA+t4<SL&1=xge-uTIQ1NcNqShW43ZnSiua<+rWczpHCIDYZzu'
    'ZPgwi?c3THU+5BEOi5_d%$JWKh$C5a1gKeqAG~cu^)~z#)!1?XB6}H{qs~vA--c;w1cg=&;N4CXr`c}RxO$P^K5TxPUwoO)Z#NG=p&v2lY@XEi'
    '%9^2Dwa<?<`{76qgMx5fun_LesXS;a^bTHeHBO`1^eCrQm@Je|D@|Fe5s<GNoMln~Zq=yHiE3xFg8GX}g@c;Hcew!Ha0_T^d%fCA{B3&k1c(dI'
    'bNwbTi?@lkt6ffm5#RwYWDa`0GAOcLqq`sJF6{<`#yAoUridL(wL<{E*)^ITSphbhE<YQC{|@h+wF6OOKU~DnA7gj{Xk8Bt0~Tw(Fey?2p;>#G'
    'M=QD(XSaovWwH*j1a4Z#RmaZK>ad}o!2X_gmFMm&(F9of@-|F0SIc2D(>!<GN+b0#%U0(l(B&Hvczh@7$zFUFBR$ccTlQVd3)NM}2KTbA$Brz`'
    'eFqn9O~W%$8;;VIzxKQ}ZndvnrTIZb)TzJvC0<RT{l9!!C+^ih(->2NBWy8)(aqKVk;dPC_xHk7t>Lm?V}-wVeJ$QpFkR2LG~)77o%J@piSZ5%'
    'X3zP0yJf@asWe@v3L<f}>-PrTt6i_#9$xLbQ2@{V*{t{d*MG;P;Q)RI!Tf5sT(;r2KkSU)x6_Xx^D>GsbH8Ke?Dv-lq&I5=*v@DPzoYrg$m392'
    'I6_w$y&Bglp<&yg@q-yYccXTjyFlMZwjZW}83bC@4HM{a?gn}+V4{g%_=UO#{h>mT6Digh!Mqld-S7%$RuNc<qM$Xl^{d@*xqvb0Ed#=L1fU1='
    '?gF0sOQ@{Zo+HluWe7R_2!1>5cG$T5FM60=zije@9N}Kjf#*R)r6O#4H;Ov=+fk(gSt_WPiOo0=Fq>}SMkeAlBzUdrb2{?DtidCX!-n5{I=xqJ'
    '5P<*@81Ze(e(_o-EY8yL0K?P|d>?-2A^r~9*wfAeo(CPIgV6xVb~qaPiXwZ+YD_D=L5FB%Hmajhu1&?xn3AetFHJ_$hlRbWjbv8`X6r{>)nOPR'
    'q`vyAtE*q|8wosk{<3j(wRqWVx&K`L*DIhGSq5Y$me4}9sp=s)k`WQziPQC#Z6Yt7=Zkc?9q|a>?*WS?*w&ysMnmg>B`j_C=|@<G?cx7NSc~o9'
    'e{|16!`*^1GTYgCzJ^xe#7gVEG;cimO|+8F8+-Zi7|1U{vsoeJsDqaZ7@mq?<%e=JI%Et6Rn&Q;!y!d5$08Q@YtgIU;?F@D6H~&)l(0e10fjo{'
    'Vx`OhWezAaA#WPU%z=e4bjI5HQ@o<5fz<$sVhzpO7qyD4gxfu5-5<&tD5wEL4Gv_LPy>eQ$|`-+8%3ipimG6-4y$#%bD(Hwk-X<*`X#4Y_o!re'
    'y%LRQgGUuzVeQcnu>gt;4hMeGRy&x7`G%~(4YXN}ZS-;=E;3k!)V7(3>_y{=>{YbHqr2!`@g7ZPlGNZhkEQm=k)v^Q4270c3>*zxMNX}s%v8uw'
    '4Ovr5L29}#)hLkg5}Opjpg!86d##I8?r(4m73y>tUr?kdZESHgi&qxHowh@&WK)VSIH53ujS|XBf+*RoIG&5_vN2>D$ZJ>0Df$#<x9em9Aj}R~'
    '1YGVvtxFfqs51|@7#M^zIEX}*bllb@kHfT{UAu0CrSe&nyW&tdB2(ZMP$>2iAJG!QUYRyg4N*<ABCZeKPKX7tzHzl3Hf|Q<bQfKv?mURuM~WG}'
    'TQv+ay~gm2G-Nzn3CTGm9(s0zWBehCB%K_{jKJSh*8^wQFXNZ~_!`6XqNgytG!JwX@bUfxe3&??%2Ay{_zF(KSD@i5XdY~NILjJhBrZf1&DyIH'
    'CmiWlfqajAQWwW+mh_5ui8bCgPk~+;%BdP{8qdhpLDABRN>!`klbEPrNv$%Q<XQY+k{BS&G_C^X;Y`>7(KboK)oc;g=Hr?*-u|^)=9P=ClX30h'
    'IK^lL(>y5Pu>!&$%20<OI<>0?9F1ro`@#BUj6J=$VO^s31opgs%JyS0?BJHP)5HDItT&*IZy&ZLL)xk?y0Cj1^?WEboJGqztwqWk5e}tHd#fJp'
    '#Ak8?b1wX&n;b1Q<G-bS@yE8p`}|+8uF?klX*T|K43F^d<uCuMS>E%uu;xF1|Msibuge$OmRmY@Pf5pjE@e<^zgs1z?fncf>+wC!r`g*$Xw%%_'
    '$kymfW6kymQeAt(a|yb;L@mN#iApPdpDk2W!x8Sbhu!|ik@e}d93^h-`17dx)JA+F+SVHSY@*g+oK4NYIVCHSbZyl2ZCw(Pt+$vgtflF>3ZY(r'
    'DzQ<&-&M-PXOSzGs)Bq1S&Ck{DATbSJ-Y>$Lf>|kU4LE>=R^`G{9Lv#MyS#A{R{dtx@L7O6RvAnVUe<H&n$@hp!IEu@J&=Ws8wW8$60!IMdn4U'
    '%F{M2$C*j1_Ea94{-z9Ma0V?m538+|?Z&b7jZSM!WlV9pL=;@s?t-KojFvMO&7&s;fEx^=nLHKHwf2K(f#(E+5h|5O!+@(ifq7I%PGujI%g^f7'
    'FXy(tc}d(9j`<IAU-o&`^zbWk84RS2B?k?wEP}V}A`b3)E;kufRq+!}3CL6UcQ5`vij~%xa}@8SP+KEdjA09=;tI4E#co_xvstBam_=Ex4v);M'
    'nT+jboGRrop2^@reWM<y%X;&i%_rg({pP7BEq+8C0H#g#wFekqxiyD+z*bp)!t*RgUzPANT&qIck<Gf+H%k+)HCF*UvNy*`K*D=vpl*EJMc=}-'
    'd^}#z7)jOYb4CYMj|C9KrO}vIHNV|H&;yq|zAON$MZMXq(l!;gu&$N8d6ZKOa*1^K<ip+*GN)N5DhkT=kvZL{&ehLxm2#3f)`^;`+8wyKRKVP4'
    'KUG1wEht5eB`79(b%GcCskOX<Ipd|SPc~{kp>3*Nu8m(sA&>o;8fxj?JU}JJ@?!kE8)9#>-ZT`zR?WJA<MdG`K$_ID0BPP%mfA`?Y_dS=Xe|zl'
    'C+e)&K|B;gU$5ZcVXFK5M=-6MQY;l)(X5wbGVrQ6h5qHs50Cy+>%$&@9s_*BAB6t&N*+A_@}>KLUg*}A+9CKc>$q6i>P%y4tYbTwX-iZx!A0e~'
    'yzZN#!(CEvcsA^#-)r60iNalp`>J$vFc5qwtnoUuDO(k}GELWMvX`eSEz{2KKoUm=E2p7N7&S=PWu+6CCv?BE!Zyq<DnhFihGA{5)2j5S>m2o{'
    'bGjN_N=G;YgCz&G)&bA9Pw7CXSte%~H7UTx!LRP7w~dyGSU1ntyEIqvg^D}Ls;I>90%sbKCxw^SRW_Am23Lyz({zl)$#$4>c#ak*wU!P(?aMDY'
    'pdPUt#w47FSuX5!C8{ybGMYF?DF*Qs@tlO=E*G59veoZ9CN831_vIcguBuoCWqY8Qm587NU8ssZST0>5+Fd{^0{N8-JBHX9)JTVK#x-)xHKkEK'
    '6|PnCpcaKrVM#7xa9f;8bSdc^r8vWz!Dz)^l%zXFtM(A*lk_u?tGDGi&@-^B11_W1MPSuP(&R@Kved$vefzY)q#C4(UTGsaal{Qomjk+*Onzuo'
    'KtBkc+g0R#?x?Y+NyV^n*bP*zx`aaI9$T0AMk77!_~m$f;a`6&jh-b^wqcw_wSS_m_c!tun17_%ZnL2k8@CW6*+%2q2mGTC!&B+r7u)?(d4IZm'
    '>|IE4>4hqs><@X&+t{iqXL-?E?9)0>nbRNgKrJq($g1p4&qzn+!%mE@=n^OUeV*%mdA<jxZQ)p~PoyydAj+Kymj##g#Wq~#6c++A+GPK6k_IYn'
    'mZFe+yttq>Z;3;t3ysbyB-oAeF0Ts9k-}U+w`^w(V`1L3uTgX0UpAy^IkNojR*Rc0>NiQfA1v^2o?HuN{T{2T>$R5($-)%_FB?uldR<=lWp@ri'
    'Un0C_wDF0^B!ok`$$X|Fw`}B9TCZB_0+iJTJ5M02OY};a5amZQ${qxiuU>a;Yk$q<^smd4p$d7cBHb8af<3y<1YVX?Evrt}4rtLr4_aWtn{OlL'
    '_!U5ov$dSk^d(VCZ`L94rG{Fvo6&&gM+>yW;AmY<#~NV@J+OC~W&|;{|6untm@?3I75;em<2tAKB;t}nnYr5Wd@lp<fQLb$#K`Bjv^*0>K4FqH'
    ')M<}XD4LsfDb4HkBS%W2re~NjR4nGS>QVPqKVS7#cj8-aIEByAZa7;G&^T&Io;W=`KMe4waWowCTm~oo;k+$Pr}Wds;c@|AhQsb0y?AEIU5rln'
    'Q-Hcy22=v;nT<k9M?;8qf621_D)qfub!F@nL>YIzK`>O_Vf4KlwL@|uLaUb`Y%50^txNqdK>LHiaAsP7=`{|!i-kOS<)#OAqh#yJQ)$q{1`gYZ'
    '!!G)a8GyM7n&BBYzdu8_B$(l_$!#=08@2IBw>a>v8N&_a1Jnb=1GM9+!ENTIYeD5Oq}cGj!cI3@&In(=9D)n<y7PyKgBfD08!hM<dEQ5284TKT'
    'Bwk2QD<%Tr?xtb;@Hj$y{7;Ei^7$enK7~Mget(EoDgD4ftb!#*{v3PJo)N*4J7D0q%@g|Gf}WSXxgufVFSr|y<^dL5ERUgr0rooc@HB)-2e)>g'
    'x`yBT0p{-_Dh308DJ=nXXk2O-U|pj@FRC_<l<<izb(6AS7@F-bOealqAKiROFwt*@7M2W}10gRYp@T!2+V7)*M8DsOO15-;lZJcIpl?}h`S|7Z'
    '_ULwa%p)VBvEC>^+Mc&9)QMhDy~80cdwYJjXVO0)9KVm+-D7sHjQq~|$Y-X5aYT>AUN9u@;9+}-eGEpJZ8;i{mt|YZUSYFB?yVp}u!XbPNJ$C5'
    ')0d_xAf=aTeEsgw>K!?SvTz&(-GOw&)g3{*(U2O6&%md1S1r1muQ{=~Y=2L~WcRlsV*zJ?Wh>2qG_LRYb0W)rq)3RYeOUCJYDgr@>Rhi|7FEGf'
    'lS}%EXG-B0UAc^TInftN2UI*~YlBbJ1b=e<`e};w-;1unZFR!-dF*>VT+Rn3IST^+B+k*oCmk2yvisFnv{!EM50VfW;lO6`&P|4>2w2uRY`};3'
    'rT?TBPKEekH9=L`53=P?6&b7b0JJeUTKAa{%j%Akiq+X@u<qVVWxJ#ZWMW0M(9mz}$tt1n18kEwNjQ%h^%}eW*IP(-^mGyybpk+um1T5A8fR@5'
    'F*=?hG-Mx-i4Ej`R4wooOh<L^r|92L(7}I<9{$<7_-E<kpP`dKTQC1C-Te9b`RC~9&)3sCboGZdlA8KnQeyM@a*$S%D7cqme*d5;o-PWF)Kf{Z'
    '>X`GYc=nl^M8($?@IJLJ@w8DmRipSTXceDvK=ARau4fz=l$RiwQ>nS?O2gH`75lK=?(oVykUK+OQ?TA=7z_fuc}y<YdABEbkfb7uMukrf?{#R='
    'THf)9{C(1LiK_Iqai>Fyti{q?{dQ@qqAM$ygJ8S2kO~U@3v&}Mn0Wo)T`m?Tw9#-*ZeSe_mlRpelD%B?rKY1d8dxA0Xus3;WyGbSzwD~Mc6qP6'
    'BskbCu|Hf6P~+B}pID1PgoaDrDnQc-8t#^9#dq}dK)X>-a6F1$o9inV1ueri!V|F#Z8OGjju9m^xuDnju_p}AV7})u-ZPo*h`hmemSv8CwdbA7'
    '(iJL44;!=_RGe|7&oJ9~9#+p-3m)G@p0R<{0Uk3U(xy}8p4gj{UQrACCiNI42}a*pMRSstLn!bI;aCODp^DL@rJ=~JmU2o&Nqt2jGsPB^Wo9%V'
    'BFs@os+bPLF+`VYbN$(a@7f4F%k<`t<^Lbc|C5&g6&40dF)4*&Q*2q{7*Q>$2&d`SEd$z<>GasuD`RzM`lW8;EIieW`$i~YqspD-TUV~nKfepN'
    'H<@NijW@Hi5%pK;{(4+#i(Kd4js-|s0f;xlOc{+VDJXB8Q6k@sO`-MpgYEjs_8~tcQJ6*Q8u2i%;fK725NO^R*P=MTjnKnQRIuI>T+h~ExUlc?'
    'Hn2=c^oFB>qob`w*<!PYCZh%r+1Ha{SaVp_2#8gU!?U6Q*^&d?JVuUJDUu=GWlzC!8}y6bowuyjT%qMMtUYcvyUEpyVf(M~pRQgM_7P$id-dY&'
    'Hbi%!5ThM`D1fkyB<KCC_WJdk8sAu3<Cwp3DsKEJ?y`k0YFFM<Ea&NaTnaAZ1XhBXhhV+;>o_&qdi6rP4j1!7#Z4Qw_tV`p<=ecAXhB+X9!Jv$'
    'bb5?dvuJ@9t(h5fOUq05>>|>3Nkp~*)ap0uw|}P4x!c#6UTES!tNk;6ThV#E7E5*h$M`0RH~)<3g70&2P(ZS9pe0~DWwTdWV(B7baq)R&@<q+p'
    'y7<*ML`y}_iW}4LQFW3CE{qz<8ss0fesZ8!VdQ1uGRmVgTW_<XQ$l87t$aB=f{hk+jpF_Ue3?|BUA2ZXF|IX1d>>~!XdKM__KnygWtW!j$}Z?*'
    'QatK{x4`A3cx;(5+-16YY?<YeEpZ??hCDK72C{N8FdT5!X&TM*^>zwdhe;j!x0`PEWUe;N!}M0a8N3vdeiB$tO(F-S`N2@S;25KzA(1Z7j>819'
    'YlA_ScC;OfV5PmEK1A`&ZEku`CSiKB!}S=Crw|2yS#Q4%cUhRce$yz>+AJ9tz80kgHCU}MP$<Cb%=U)lUok3W_{hNrK})cNi-y^mR#9=QR#gL&'
    '>+w1~1q+@?uf{Kh+`>!3@1res=03ns;xIF$QMlE5*do70ea9!7L8w5Nk=-26w*=#uob^`53a|ZOvXu%VE{9L$WI+bNa-1xpg~sMht8B}>;fo`1'
    'gp0*A+eGsiJrFrcyegozjMhdV;Kc^oZlAH`mUjlI+`<p7=LfBt-*444<dpX6<-$R`JFF<IUwv3EI2yKFwSYX}x=m|xt-lE$VWd{;?Y;)wUi<40'
    '|G+ucoSdJEu@UC0t8hb`byyxu(X@41d~aCfWU)KfHTe*a?InP1ILY>}Xdx@ElVJPFMzi^(dl9auWOpoP%PEC+$2+O6UJ${fb#2xBdF>K5%le=B'
    'o(t#;prYuQhJ`F#MXJkGFGZv)T2(js)6+OeP?s(w>$n#Y>w;)&q)7R>&N$sj1fmH*W|;+&RRG_OZ%SaBrDGi~C#Z`RaFJPse+&fSk9BY5ZI&0e'
    'gA^Ep{c+xK;EJxOVdLj<+i-M261XF;=5ndBKj9sFoXhH*QGjIP<sI%-^roKV_H6?wTFrT_V$VUve_QuCznoWS*)+e6=64%dCFN7tQ(;sJnz;|~'
    'bW!5aDvRmgTxjXO%c`A|;!#g`7{s>>v_X;E_4+Q;7fns0i`Mlu$8N1rt4my@stafGbwdZHUH+o%^lK(s1?S&~Ny9Y{@~V<HuvwG?&($#Njx^9z'
    'R!-0SD?`ikOs~yo)?!$@2-k7#v2|6hF&oNlTg)@OuqD^M7UOKXiLq<Cqa$CQt@}7CIl}qJwr@6FqJyKc@%TQS(mt13K;c(|UJo_{TDj83)4rTX'
    '7H&G(R20&{%5G=4?d8Kq<2a(lDNN7m$bfHQ;(Negogr!v#pJy~)mj~)k8fD;xG%52`rFt4`TdV_@q((iFiC34vI7rgnbj^&@Um&0;K$NB!T&}~'
    '6HRAN4Vux@3SvNwg_IM~BYGM)6k_$oI^<d1E#j?~TCUQ=L^Rn^nV!D*AV)*FBYh#9k)EvW0&G>&lM5^Bma7-u`^^onD!lwLhx8}3ba|*ZYhhMf'
    '>XU)@u#NMmA;+%y;_2!C0(e?Q$p'
)


def unpack_unihir(run):
    raw = zlib.decompress(base64.b85decode(PAYLOAD_B85))
    if hashlib.sha256(raw).hexdigest() != PAYLOAD_SHA256:
        raise ValueError('Sai SHA-256 của mã UniHIR nhúng.')
    data = json.loads(raw)
    vendor = run / 'vendor'
    for relative, text in data['vendor'].items():
        path = bundle_file(vendor, relative)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf8')
    runtime = run / 'flash_runtime.py'
    runtime.write_text(data['runtime'], encoding='utf8')
    return vendor

In [ ]:
WORKER_SOURCE = r'''
from __future__ import annotations

import functools
import json
import os
import sys
import threading
import time
from pathlib import Path


def restoration_memory_limits(torch, options):
    cap = options.get('max_memory', '18GiB')
    if not isinstance(cap, str) or not cap.endswith('GiB'):
        raise ValueError('max_memory must be a per-GPU limit such as 18GiB')
    requested = int(float(cap[:-3]) * 1024**3)
    reserve = int(float(options.get('activation_reserve_gib', 4)) * 1024**3)
    if requested <= 0 or reserve < 1024**3:
        raise ValueError('Positive weight budget and >=1 GiB activation reserve required')
    limits = {}
    for index in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(index)
        limit = min(requested, free - reserve)
        # VAE, decoder and prompt live on the entry GPU outside the sharded model.
        if index == 0:
            limit -= int(options.get('entry_reserve_gib', 2) * 1024**3)
        if limit < 4 * 1024**3:
            raise MemoryError(f'GPU {index}: insufficient free VRAM after activation reserve')
        limits[index] = limit
    return limits


def verify_restoration_device_map(mapping, count, *, require_multiple=True):
    devices = set()
    for value in mapping.values():
        if isinstance(value, int):
            index = value
        else:
            label = str(value)
            if not label.startswith('cuda:'):
                raise MemoryError(f'Model would spill to {label}; reduce scope or change explicit placement budgets')
            index = int(label.split(':')[1])
        if not 0 <= index < count:
            raise ValueError(f'Device map refers to unavailable GPU {index}')
        devices.add(index)
    if require_multiple and len(devices) < 2:
        raise ValueError('Sharded placement did not distribute the model across multiple GPUs')
    return mapping


def unihir_weight_limits(torch, options):
    """Reserve a whole visible GPU for the full-resolution VAE encode/decode."""
    limits=restoration_memory_limits(torch,options)
    vae=options.get('vae_gpu_id')
    if vae is None:return limits,None
    if vae=='auto':vae=torch.cuda.device_count()-1
    if isinstance(vae,bool) or not isinstance(vae,int) or vae not in limits:
        raise ValueError('UniHIR vae_gpu_id must be auto or a visible logical GPU index')
    if len(limits)<3:
        raise MemoryError('Dedicated UniHIR VAE requires >=3 visible GPUs so BAGEL remains sharded across >=2 GPUs')
    if torch.cuda.mem_get_info(vae)[0]<16*1024**3:
        raise MemoryError(f'UniHIR VAE GPU {vae} needs >=16 GiB free before loading; no automatic resize')
    del limits[vae]
    return limits,vae


def restoration_gpu_probe(torch, model, options):
    """Validate every visible GPU; aggregate capacity is only a placement precheck."""
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA required but unavailable')
    devices = []
    for index in range(torch.cuda.device_count()):
        (torch.ones(8, device=f'cuda:{index}') * 2).sum().item()
        torch.cuda.synchronize(index)
        devices.append(dict(index=index, name=torch.cuda.get_device_name(index),
            total_bytes=torch.cuda.get_device_properties(index).total_memory,
            free_bytes=torch.cuda.mem_get_info(index)[0], capability=list(torch.cuda.get_device_capability(index))))
    sharded = options.get('placement') == 'sharded'
    if sharded and len(devices) < 2:
        raise RuntimeError(f'{model}: sharded placement requires >=2 visible GPUs')
    if model in ('unihir_restoration', 'tadisr_x4') and sharded:
        limits = unihir_weight_limits(torch,options)[0] if model=='unihir_restoration' else restoration_memory_limits(torch, options)
        minimum = 32 if model == 'unihir_restoration' else 28
        if sum(limits.values()) < minimum * 1024**3:
            raise MemoryError(f'{model}: insufficient aggregate weight budgets; inference workspace is reserved separately')
    elif model == 'unihir_restoration' and devices[0]['total_bytes'] < 75 * 1024**3:
        raise RuntimeError('Single-GPU UniHIR needs >=75 GiB; use placement=sharded for multiple L4 GPUs')
    elif model == 'tadisr_x4' and devices[0]['total_bytes'] < 24 * 1024**3:
        raise RuntimeError('Single-GPU float32 TADiSR needs >=24 GiB; use placement=sharded on L4')
    if model == 'unihir_restoration' and any(d['capability'][0] < 8 for d in devices):
        raise RuntimeError('UniHIR BF16/FlashAttention requires Ampere-or-newer GPUs')
    report=dict(devices=devices, placement=options.get('placement', 'single'),
        cuda_visible_devices=__import__('os').environ.get('CUDA_VISIBLE_DEVICES'),
        weight_capacity_check_only=True)
    if model=='unihir_restoration' and sharded:
        weight_limits,vae_gpu=unihir_weight_limits(torch,options)
        report.update(weight_gpu_ids=sorted(weight_limits),vae_gpu_id=vae_gpu)
    return report


def cpu_staged_checkpoint_dispatch(dispatch, model, **kwargs):
    """Stage checkpoint tensors on CPU before Accelerate casts/places them.

    The pinned Accelerate safetensors loader otherwise holds the checkpoint's
    original dtype on GPU alongside the converted model. Clone function globals
    for this call only; neither Accelerate nor the upstream module is patched.
    Final model placement still follows the validated GPU-only device map.
    """
    import types

    def clone(function, replacements):
        result = types.FunctionType(function.__code__,
            {**function.__globals__, **replacements}, function.__name__,
            function.__defaults__, function.__closure__)
        result.__kwdefaults__ = function.__kwdefaults__
        return result

    load = dispatch.__globals__['load_checkpoint_in_model']
    read = load.__globals__['load_state_dict']

    def cpu_read(checkpoint_file, device_map=None):
        return read(checkpoint_file, device_map=None)

    staged_load = clone(load, {'load_state_dict': cpu_read})
    return clone(dispatch, {'load_checkpoint_in_model': staged_load})(model, **kwargs)


def verify_unihir_vae(vae):
    """Check the real codec on a tiny tensor, restoring all touched RNG states."""
    import time
    import torch
    parameter=next(vae.parameters());device=parameter.device
    devices=[device.index] if device.type=='cuda' else []
    started=time.monotonic()
    with torch.random.fork_rng(devices=devices),torch.inference_mode():
        image=torch.zeros((1,3,64,64),device=device,dtype=parameter.dtype)
        latent=vae.encode(image);decoded=vae.decode(latent)
        if decoded.shape!=image.shape:raise ValueError('UniHIR VAE roundtrip dimensions differ')
        if not torch.isfinite(latent).all().item() or not torch.isfinite(decoded).all().item():
            raise ValueError('UniHIR VAE produced non-finite values in codec precheck')
    return dict(device=str(device),dtype=str(parameter.dtype),input_size=[64,64],
                seconds=round(time.monotonic()-started,3),status='passed',scope='codec_only; not full model inference')


def load_unihir_multi_gpu(assets, options):
    import torch
    import types
    import utils as upstream
    from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch
    limits,vae_gpu=unihir_weight_limits(torch,options)

    def validate(mapping):
        verify_restoration_device_map(mapping,torch.cuda.device_count())
        for device in mapping.values():
            index=device if isinstance(device,int) else int(str(device).split(':')[1])
            if index not in limits:raise MemoryError('UniHIR weight map uses the GPU reserved for VAE')
        return mapping

    def infer(model, **kwargs):
        # The upstream estimate otherwise assumes float32 although loading is BF16.
        kwargs.update(max_memory=limits, dtype=torch.bfloat16,
                      no_split_module_classes=['Qwen2MoTDecoderLayer'])
        return validate(infer_auto_device_map(model, **kwargs))

    def dispatch(model, **kwargs):
        # Upstream co-locates its embedding/connector modules after inference.
        validate(kwargs['device_map'])
        return cpu_staged_checkpoint_dispatch(load_checkpoint_and_dispatch, model, **kwargs)

    # Bind two placement functions without changing the pinned upstream files/globals.
    loader = types.FunctionType(upstream.load_model.__code__, {
        **upstream.load_model.__globals__, 'infer_auto_device_map': infer,
        'load_checkpoint_and_dispatch': dispatch}, argdefs=upstream.load_model.__defaults__)
    result = loader(assets['base_model'], max_mem_per_gpu=options.get('max_memory', '18GiB'))
    result[0]._vtm_device_map = {k: str(v) for k, v in result[0].hf_device_map.items()}
    if vae_gpu is not None:
        result[2].to(f'cuda:{vae_gpu}')
        result[0]._vtm_device_map['vae_model']=f'cuda:{vae_gpu}'
    if options.get('verify_vae',False):result[0]._vtm_vae_smoke=verify_unihir_vae(result[2])
    return result


def restoration_cuda_metrics(torch, *, reset=False):
    records = []
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            torch.cuda.synchronize(index)
            if reset:
                torch.cuda.reset_peak_memory_stats(index)
            records.append(dict(index=index, peak_allocated_bytes=torch.cuda.max_memory_allocated(index),
                                peak_reserved_bytes=torch.cuda.max_memory_reserved(index)))
    return records


def load_backend(assets, options):
    from data.transforms import ImageTransform
    from utils import UniHIRInferencer
    m, t, v, _, _, _, nt = load_unihir_multi_gpu(assets, options)
    draft_edge, final_edge = options['draft_edge'], options['final_edge']
    vt = ImageTransform(draft_edge, 128, 16, True, max_pixels=draft_edge ** 2)
    vg = ImageTransform(final_edge, 128, 16, True, max_pixels=final_edge ** 2)
    it = ImageTransform(448, 224, 14, max_pixels=448 ** 2)
    sizes = {}

    def capture(name, transform):

        class ObservedTransform:

            def __getattr__(self, key):
                return getattr(transform, key)

            def __call__(self, image, *a, **kw):
                result = transform(image, *a, **kw)
                sizes[name] = {'input_size': list(image.size), 'output_shape': list(result.shape) if hasattr(result, 'shape') else None}
                return result
            def resize_transform(self, image, *a, **kw):
                result = transform.resize_transform(image, *a, **kw)
                sizes[f'{name}_resize'] = {'input_size': list(image.size),
                                           'output_size': list(result.size)}
                return result
        return ObservedTransform()
    net = UniHIRInferencer(model=m, tokenizer=t, vae_model=v, vae_transform=capture('vae', vt), vae_transform_gen=capture('vae_gen', vg), vit_transform=capture('vit', it), new_token_ids=nt)
    net._observed_processor_sizes = sizes
    net._vtm_device_map = getattr(m, '_vtm_device_map', getattr(m, 'hf_device_map', {}))
    net._vtm_vae_smoke = getattr(m, '_vtm_vae_smoke', None)
    return net


def run_backend(net, image, options):
    from utils import invert_image
    extras = {}
    result = net(image=invert_image(image), text='请对图中破损位置进行定位，并输出破损定位图。', und_only=False, gen_only=False, reflection_count=options['ref_count'], think=False, max_think_token_n=options['max_text_tokens'], do_sample=False, cfg_text_scale=1.0, cfg_img_scale=1.0, cfg_interval=[0.0, 1.0], timestep_shift=3.0, num_timesteps=options['steps'], cfg_renorm_min=0.0, cfg_renorm_type='global')
    final = invert_image(result['final_image'])
    extras['predictions'] = {k: result.get(k) for k in ('text_step1', 'text_step2', 'text_merge')}
    extras['predictions'].update(refinement_count=options['ref_count'], confidence=None, confidence_status='upstream_does_not_supply_calibrated_confidence')
    import re
    extras['predictions']['damage_predictions'] = [{'box_normalized_1000': [int(x) for x in match[:4]], 'predicted_character': match[4], 'evidence': 'model_generated; not_source_observation'} for match in re.findall('<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),(.?)<\\|box_end\\|>', result.get('text_merge', ''))]
    return (final, extras, {'generative_reconstruction': True, 'alignment': 'upstream_native_output; generated_glyphs_unverified', 'observed_processor_sizes': net._observed_processor_sizes})


class UniHIRProgress:

    def __init__(self, net, directory, interval=30, emit=print, input_size=None):
        self.net = net
        self.directory = Path(directory)
        self.interval = interval
        self.emit = emit
        self.lock = threading.RLock()
        self.stop = threading.Event()
        self.stack = []
        self.replacements = []
        self.hooks = []
        self.sequence = 0
        self.forwards = 0
        self.tokens = 0
        self.artifacts = []
        self.started = time.monotonic()
        self.status = 'running'
        self.error_stage = None
        self.input_size = input_size
        self.generation_sizes = []

    def snapshot(self):
        with self.lock:
            return dict(format='vtm-unihir-progress-v1', status=self.status, stage=' > '.join(self.stack) or 'between_stages', error_stage=self.error_stage, activity=self.sequence, completed_forward_events=self.forwards, text_tokens=self.tokens, seconds=round(time.monotonic() - self.started, 3), artifacts=list(self.artifacts), input_size=self.input_size, generation_sizes=list(self.generation_sizes), observed_processor_sizes=dict(getattr(self.net, '_observed_processor_sizes', {})), intermediate_images_are_final=False)

    def publish(self, announce=False):
        with self.lock:
            data = self.snapshot()
            self.directory.mkdir(parents=True, exist_ok=True)
            target = self.directory / 'progress.json'
            temporary = target.with_suffix('.tmp')
            temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf8')
            for attempt in range(20):
                try:
                    temporary.replace(target)
                    break
                except PermissionError:
                    if attempt == 19:
                        raise
                    time.sleep(0.005)
        if announce:
            self.emit('[UniHIR progress] ' + json.dumps(data, ensure_ascii=False), flush=True)

    def pulse(self, kind):
        with self.lock:
            self.sequence += 1
            if kind == 'forward':
                self.forwards += 1
            else:
                self.tokens += 1

    def wrap(self, owner, name, label):
        original = getattr(owner, name, None)
        if not callable(original):
            return
        owned = name in getattr(owner, '__dict__', {})

        @functools.wraps(original)
        def observed(*args, **kwargs):
            with self.lock:
                self.stack.append(label)
                self.sequence += 1
                if label == 'gen_image' and args and isinstance(args[0], (list, tuple)) and (len(args[0]) == 2):
                    self.generation_sizes.append(list(args[0]))
            self.publish(True)
            try:
                result = original(*args, **kwargs)
                if label in ('gen_image', 'gen_text'):
                    self.capture(label, result)
                return result
            except BaseException:
                with self.lock:
                    if self.error_stage is None:
                        self.error_stage = ' > '.join(self.stack)
                raise
            finally:
                with self.lock:
                    self.stack.pop()
                    self.sequence += 1
                self.publish(True)
        setattr(owner, name, observed)
        self.replacements.append((owner, name, original, owned))

    def capture(self, label, result):
        folder = self.directory / 'intermediate_not_final'
        folder.mkdir(exist_ok=True)
        number = len(self.artifacts) + 1
        if label == 'gen_text' and isinstance(result, str):
            target = folder / f'{number:03d}_text.txt'
            target.write_text(result, encoding='utf8')
        elif label == 'gen_image' and hasattr(result, 'save'):
            target = folder / f'{number:03d}_image.png'
            result.save(target, format='PNG')
        else:
            return
        with self.lock:
            self.artifacts.append(str(target.relative_to(self.directory)))

    def heartbeat(self):
        while not self.stop.wait(self.interval):
            self.publish(True)

    def __enter__(self):
        try:
            for name in ('update_context_text', 'update_context_image', 'update_context_image_gen', 'gen_image', 'gen_text', 'decode_image'):
                self.wrap(self.net, name, name)
            vae = getattr(self.net, 'vae_model', None)
            if vae is not None:
                for name in ('encode', 'decode'):
                    self.wrap(vae, name, 'vae.' + name)
            model = getattr(self.net, 'model', None)
            language = getattr(model, 'language_model', None)
            decoder = getattr(language, 'model', None)
            layers = getattr(decoder, 'layers', None)
            if callable(getattr(model, '_forward_flow', None)):
                original = model._forward_flow

                @functools.wraps(original)
                def observed_flow(*args, **kwargs):
                    output = original(*args, **kwargs)
                    self.pulse('forward')
                    return output

                owned = '_forward_flow' in model.__dict__
                model._forward_flow = observed_flow
                self.replacements.append((model, '_forward_flow', original, owned))
            elif layers is not None and len(layers):
                self.hooks.append(layers[-1].register_forward_hook(lambda *a: self.pulse('forward')))
            head = getattr(language, 'lm_head', None)
            if head is not None:
                self.hooks.append(head.register_forward_hook(lambda *a: self.pulse('token')))
            self.publish(True)
            self.thread = threading.Thread(target=self.heartbeat, daemon=True)
            self.thread.start()
            return self
        except BaseException:
            self.__exit__(*__import__('sys').exc_info())
            raise

    def __exit__(self, kind, value, traceback):
        self.stop.set()
        if hasattr(self, 'thread'):
            self.thread.join(timeout=2)
        for hook in self.hooks:
            hook.remove()
        for owner, name, original, owned in reversed(self.replacements):
            if owned:
                setattr(owner, name, original)
            else:
                delattr(owner, name)
        if kind is None:
            for relative in self.artifacts:
                target = (self.directory / relative).resolve()
                if target.is_relative_to(self.directory.resolve()) and target.is_file():
                    target.unlink()
            folder = self.directory / 'intermediate_not_final'
            if folder.is_dir():
                folder.rmdir()
            self.artifacts.clear()
        with self.lock:
            self.status = 'failed' if kind else 'complete'
        self.publish(True)
        return False


def write_json(path, data):
    temporary = path.with_suffix('.part')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf8')
    temporary.replace(path)


def main():
    import torch

    job = json.loads(Path(sys.argv[1]).read_text(encoding='utf8'))
    output = Path(job['output'])
    options = job['options']
    if sys.argv[2] == 'setup':
        import flash_runtime

        visible = restoration_gpu_probe(torch, 'unihir_restoration', options)
        print(json.dumps(visible, ensure_ascii=False, indent=2), flush=True)
        write_json(output / 'gpu_preflight.json', visible)
        python = flash_runtime.prepare_runtime(job['packages'], job['runtime'], 'unihir_restoration')
        write_json(output / 'runtime.json', {'python': python})
        return

    from PIL import Image
    from flash_attn import flash_attn_func, flash_attn_varlen_func

    # Font paths in the pinned release are relative to the model directory.
    os.chdir(job['assets']['base_model'])
    q = torch.randn(16, 2, 64, device='cuda:0', dtype=torch.bfloat16)
    cu = torch.tensor([0, 16], device=q.device, dtype=torch.int32)
    fixed = flash_attn_func(q[None], q[None], q[None])
    variable = flash_attn_varlen_func(q, q, q, cu, cu, 16, 16)
    if fixed.shape != q[None].shape or variable.shape != q.shape or not (
            torch.isfinite(fixed).all().item() and torch.isfinite(variable).all().item()):
        raise RuntimeError('FlashAttention CUDA probe thất bại.')
    print('FlashAttention CUDA probe đã qua.', flush=True)

    print('Đang nạp UniHIR và phân bổ checkpoint lên GPU.', flush=True)
    torch.manual_seed(job['seed'])
    started = time.monotonic()
    restoration_cuda_metrics(torch, reset=True)
    net = load_backend(job['assets'], options)
    load_metrics = restoration_cuda_metrics(torch, reset=True)
    load_seconds = time.monotonic() - started
    print(f'Đã nạp UniHIR trong {load_seconds:.1f}s. Device map: {net._vtm_device_map}', flush=True)

    with Image.open(output / 'original.png') as png:
        image = png.convert('RGB')
    started = time.monotonic()
    torch.manual_seed(job['seed'])
    with torch.no_grad(), UniHIRProgress(net, output / 'progress', input_size=list(image.size)):
        final, extras, diagnostics = run_backend(net, image, options)
        inference_seconds = time.monotonic() - started
        final.save(output / 'unihir.part.png')
        (output / 'unihir.part.png').replace(output / 'unihir.png')
        image.resize(final.size, Image.Resampling.BICUBIC).save(output / 'bicubic.png')
        write_json(output / 'predictions.json', extras['predictions'])
    inference_metrics = restoration_cuda_metrics(torch)
    write_json(output / 'run.json', {
        'source': job['source'], 'pdf_page': job['pdf_page'],
        'crop_box': job['crop_box'], 'source_acquisition': job['source_acquisition'],
        'input_sha256': job['input_sha256'],
        'input_size': list(image.size), 'output_size': list(final.size),
        'upstream_revision': job['upstream_revision'], 'options': options, 'seed': job['seed'],
        'assets': job['asset_inventory'], 'generative_reconstruction': True,
        'device_map': net._vtm_device_map, 'vae_probe': net._vtm_vae_smoke,
        'setup_seconds': job['setup_seconds'], 'load_seconds': load_seconds,
        'inference_seconds': inference_seconds, 'cuda_after_load': load_metrics,
        'cuda_after_inference': inference_metrics, 'diagnostics': diagnostics,
    })
    print(f'UniHIR hoàn tất inference trong {inference_seconds:.1f}s.', flush=True)


if __name__ == '__main__':
    main()
'''

In [ ]:
# Chuẩn bị và xem crop trước khi chạy GPU.
from IPython.display import display
from PIL import Image

if not INPUT_ROOT.is_dir() or not OUTPUT_ROOT.parent.is_dir() or sys.platform != 'linux':
    raise RuntimeError('Mở notebook trên Kaggle với GPU và Internet Off.')
if (STEPS < 2 or REFLECTION_COUNT < 0 or MAX_TEXT_TOKENS < 1
        or min(SETUP_TIMEOUT_MINUTES, LOAD_TIMEOUT_MINUTES, INFERENCE_TIMEOUT_MINUTES) <= 0):
    raise ValueError('Kiểm tra số bước, số token, reflection và thời hạn.')
if any(edge < 128 or edge > 1024 or edge % 16 for edge in (DRAFT_EDGE, FINAL_EDGE)):
    raise ValueError('DRAFT_EDGE và FINAL_EDGE phải là bội số 16 trong khoảng 128–1024.')
if GPU_IDS is not None and (len(GPU_IDS) != 4 or len(set(GPU_IDS)) != 4 or min(GPU_IDS) < 0):
    raise ValueError('GPU_IDS cần bốn GPU khác nhau, hoặc None.')

packages = find_bundle(PACKAGES_DIR, PLAN_PATH, 'packages', 'PACKAGES_DIR')
source = find_input(INPUT_FILE)
env = prep_worker_environment()
env.update(USE_TF='0', VTM_FLASH_REQUIRE_CACHE='0' if ALLOW_FLASH_BUILD else '1')
if GPU_IDS is not None:
    env['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, GPU_IDS))
if FLASH_CACHE_DIR:
    cache = input_path(FLASH_CACHE_DIR)
    if not cache.is_dir():
        raise FileNotFoundError(cache)
    env['VTM_FLASH_CACHE_INPUT'] = str(cache)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix='run_', dir=OUTPUT_ROOT))
job = {
    'source': str(source), 'pdf_page': PDF_PAGE if source.suffix.lower() == '.pdf' else None,
    'crop_box': list(CROP_BOX), 'output': str(run_dir), 'seed': SEED,
    'packages': str(packages), 'runtime': str(RUNTIME), 'upstream_revision': UPSTREAM_REVISION,
    'options': {'placement': 'sharded', 'device': 'cuda', 'steps': STEPS,
                'ref_count': REFLECTION_COUNT, 'max_text_tokens': MAX_TEXT_TOKENS,
                'draft_edge': DRAFT_EDGE, 'final_edge': FINAL_EDGE,
                'max_memory': MAX_GPU_MEMORY, 'activation_reserve_gib': 4,
                'entry_reserve_gib': 2, 'vae_gpu_id': 'auto', 'verify_vae': True},
}
with (run_dir / 'run.log').open('w', encoding='utf8') as log:
    if job['pdf_page'] is not None:
        ensure_pdfium(packages, env, log)
    image, acquisition = prepare_input(job)
    job['source_acquisition'] = acquisition
    job['input_sha256'] = sha256(run_dir / 'original.png')
    (run_dir / 'input.json').write_text(json.dumps(job, indent=2), encoding='utf8')
    log_message(log, f"{source.name}, trang {job['pdf_page']}, nguồn {acquisition['size']}, crop {CROP_BOX} → {image.size}.\n")
with Image.open(run_dir / 'source_crop_location.png') as overview:
    display(overview.copy())
display(image.resize((image.width * 3, image.height * 3), Image.Resampling.NEAREST)
        if max(image.size) <= 300 else image)

In [ ]:
# Nạp trọng số và chạy đúng một crop.
if (run_dir / 'runtime.json').exists():
    raise RuntimeError('Chạy lại ô xem crop để tạo thư mục mới cho lần thử tiếp theo.')
models = find_bundle(MODELS_DIR, ASSET_PATH, 'models', 'MODELS_DIR')
model_dir = (models / ASSET_PATH).parent
job['assets'] = {'base_model': str(model_dir)}
with (run_dir / 'run.log').open('a', encoding='utf8') as log, tempfile.TemporaryDirectory(
        prefix='unihir_smoke_') as directory:
    scratch = Path(directory)
    log_message(log, f'Kiểm tra tài sản từ {model_dir}.\n')
    job['asset_inventory'] = verify_assets(model_dir, VERIFY_WEIGHT_SHA256)
    vendor = unpack_unihir(scratch)
    worker = scratch / 'worker.py'
    worker.write_text(WORKER_SOURCE, encoding='utf8')
    job_file = scratch / 'job.json'
    job_file.write_text(json.dumps(job, indent=2), encoding='utf8')
    worker_env = {**env, 'PYTHONPATH': str(vendor)}
    started = time.monotonic()
    run_logged([sys.executable, '-u', worker, job_file, 'setup'], worker_env, log,
               stage='Setup và CUDA probe', timeout_minutes=SETUP_TIMEOUT_MINUTES)
    job['setup_seconds'] = time.monotonic() - started
    job_file.write_text(json.dumps(job, indent=2), encoding='utf8')
    python = read_json(run_dir / 'runtime.json')['python']
    run_logged([python, '-u', worker, job_file, 'inference'], worker_env, log,
               stage='Nạp checkpoint', timeout_minutes=LOAD_TIMEOUT_MINUTES,
               ready_prefix='Đã nạp UniHIR', inference_minutes=INFERENCE_TIMEOUT_MINUTES)

In [ ]:
result = read_json(run_dir / 'run.json')
print(f"Crop {result['input_size']} → ảnh sinh {result['output_size']} · "
      f"setup {result['setup_seconds']:.1f}s · nạp {result['load_seconds']:.1f}s · "
      f"inference {result['inference_seconds']:.1f}s")
from IPython.display import HTML
import base64

panels = []
for filename, title in [('original.png', 'Crop gốc'), ('bicubic.png', 'Bicubic'),
                        ('unihir.png', 'UniHIR · ảnh sinh cuối')]:
    encoded = base64.b64encode((run_dir / filename).read_bytes()).decode('ascii')
    panels.append(f'<div><strong>{title}</strong><br><img width="240" '
                  f'src="data:image/png;base64,{encoded}" /></div>')
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + ''.join(panels) + '</div>'))
predictions = read_json(run_dir / 'predictions.json')
print('Số vùng hỏng model dự đoán:', len(predictions['damage_predictions']))

In [ ]:
from zipfile import ZipFile
from IPython.display import FileLink, display

archive = run_dir / 'unihir_images.zip'
with ZipFile(archive, 'w') as bundle:
    for artifact in sorted(run_dir.rglob('*')):
        if artifact.is_file() and artifact != archive and not artifact.name.endswith(('.tmp', '.part.png')):
            bundle.write(artifact, artifact.relative_to(run_dir))
display(FileLink(str(archive)))